# VRP Unified Forecast Denominator - Locked Model Notebook

**Locked working denominator:** `unified_fds_no_min_return`

This cleaned notebook preserves the model-research audit trail and removes superseded display-only work. It is organized as:

1. **Cells 1-4:** setup, research panel, baseline diagnostics, feature construction.
2. **Cells 5-6:** Front/Middle candidate model research and common-row review.
3. **Cells 6B-6C:** Back unfreeze diagnostics and pure Back refinement.
4. **Cell 7A:** unified all-tenor `fds_no_min_return` OOS denominator test.
5. **Cell 7B:** visual sanity checks with completed-forward realized-vol guard.
6. **Cell 7C:** daily-style term-structure dashboard charts with inline chart display.

## Locked model contract

- One exact model spec across all nine tenors: `9, 12, 15, 18, 21, 24, 27, 30, 33`.
- Feature set: downside realized variance levels, downside shares, and recent max absolute returns.
- Explicitly excluded: `candidate_min_return_*`, implied-vol features, VRP/z-score inputs, RSI inputs, tenor/bucket labels, option/trade-outcome fields, and forward/target leakage fields.
- No blending.
- No threshold grid in the denominator lock test.
- Back thresholds unchanged in diagnostics.

## Usage

Run sequentially when rebuilding the research trail. For day-to-day chart review after Cell 7A output already exists, run **Cell 7B** and **Cell 7C** only.


In [ ]:
# ======================================================================================
# Cell 1 — Setup new Front / Middle Corsi forecast repair notebook
# ======================================================================================
#
# Objective:
#   Create a clean research branch for repairing Front and Middle Corsi forecast
#   denominator performance while preserving the locked Back tenor denominator.
#
# Scope:
#   - Define exact tenor grid and bucket mapping.
#   - Define the frozen Back contract.
#   - Create processed/audit/notebook folders for this research branch.
#   - Locate the latest required upstream files.
#   - Save a notebook manifest.
#
# Explicitly not in scope:
#   - No model fitting.
#   - No feature engineering.
#   - No signal threshold testing.
#   - No Core Back refit or retest.
#   - No Secondary.
#   - No sizing.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

# ======================================================================================
# 0. Project configuration
# ======================================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

SOURCE_CORE_BUCKET_BRANCH = "vrp_core_bucket_parameters_v1"
SOURCE_CORSI_BRANCH = "forecast_model_corsi_v1"
SOURCE_30D_BRANCH = "vrp_30d_corsi_v1"
NEW_BRANCH = "vrp_front_middle_corsi_forecast_repair_v1"

SOURCE_CORE_BUCKET_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / SOURCE_CORE_BUCKET_BRANCH
SOURCE_CORE_BUCKET_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / SOURCE_CORE_BUCKET_BRANCH
SOURCE_CORSI_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / SOURCE_CORSI_BRANCH
SOURCE_30D_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / SOURCE_30D_BRANCH

OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
OUT_NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / NEW_BRANCH

for d in [OUT_PROCESSED_DIR, OUT_AUDIT_DIR, OUT_NOTEBOOK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# ======================================================================================
# 1. Tenor grid and frozen Back contract
# ======================================================================================

EXACT_TENOR_GRID = [9, 12, 15, 18, 21, 24, 27, 30, 33]
FRONT_TENORS = [9, 12, 15]
MIDDLE_TENORS = [18, 21, 24]
BACK_TENORS = [27, 30, 33]
RESEARCH_TENORS = FRONT_TENORS + MIDDLE_TENORS

TENOR_BUCKET_MAP = {
    9: "Front",
    12: "Front",
    15: "Front",
    18: "Middle",
    21: "Middle",
    24: "Middle",
    27: "Back",
    30: "Back",
    33: "Back",
}

FROZEN_BACK_CONTRACT = {
    "tenors": BACK_TENORS,
    "forecast_model": "har_downside_v1",
    "forecast_variance_column": "forecast_variance_har_downside_v1",
    "forecast_vol_column": "forecast_vol_har_downside_v1",
    "policy": "Back tenors are diagnostic comparators only. Do not refit, retune, or overwrite Back forecasts in this branch.",
    "locked_core_back_thresholds": {
        "model_vrp_log": 0.70,
        "model_vrp_z_3m": 0.70,
        "model_vrp_z_1y": 0.70,
        "RSI14": 70.0,
        "forecast_vol_pct": 8.5,
    },
}

LOCKED_HAR_DOWNSIDE_FEATURES = [
    "log_rv_1d",
    "log_rv_std_5d",
    "log_rv_std_21d",
    "log_rv_std_63d",
    "log_downside_rv_5d",
    "log_downside_rv_21d",
    "log_downside_rv_63d",
    "downside_share_21d",
    "negative_return_count_21d",
]

FORBIDDEN_MODEL_FEATURE_KEYWORDS = [
    "implied", "vix", "vrp", "z_", "rsi", "actual_dte", "win", "pnl",
    "credit", "strike", "expiry", "expiration", "option", "delta", "bucket", "tenor",
    "target", "forward", "last_forward", "trade_outcome",
]

# ======================================================================================
# 2. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    """Return latest modified file matching pattern."""
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None

# ======================================================================================
# 3. Locate upstream files
# ======================================================================================

UPSTREAM_FILES = {
    "latest_denominator_panel": latest_file(
        SOURCE_CORE_BUCKET_PROCESSED_DIR,
        "02B_cross_tenor_har_downside_denominator_panel_*.parquet",
        required=True,
    ),
    "latest_signal_base_panel": latest_file(
        SOURCE_CORE_BUCKET_PROCESSED_DIR,
        "02C_cross_tenor_core_signal_base_panel_*.parquet",
        required=True,
    ),
    "cross_tenor_target_source": latest_file(
        SOURCE_CORSI_PROCESSED_DIR,
        "corsi_forward_realized_variance_targets_v1_*.parquet",
        required=True,
    ),
    "cross_tenor_implied_source": latest_file(
        SOURCE_CORSI_PROCESSED_DIR,
        "corsi_model_feature_panel_v1_*.parquet",
        required=False,
    ),
    "locked_30d_feature_source": latest_file(
        SOURCE_30D_PROCESSED_DIR,
        "02B_30d_corsi_extended_feature_target_panel_*.parquet",
        required=True,
    ),
}

# ======================================================================================
# 4. Save manifest
# ======================================================================================

manifest = {
    "run_timestamp": TIMESTAMP,
    "project_root": str(PROJECT_ROOT),
    "new_branch": NEW_BRANCH,
    "objective": "Repair Front/Middle Corsi forecast denominator performance while keeping Back unchanged.",
    "approved_cells": ["Cell 1", "Cell 2", "Cell 3"],
    "exact_tenor_grid": EXACT_TENOR_GRID,
    "front_tenors": FRONT_TENORS,
    "middle_tenors": MIDDLE_TENORS,
    "back_tenors_frozen": BACK_TENORS,
    "research_tenors": RESEARCH_TENORS,
    "frozen_back_contract": FROZEN_BACK_CONTRACT,
    "locked_har_downside_features": LOCKED_HAR_DOWNSIDE_FEATURES,
    "forbidden_model_feature_keywords": FORBIDDEN_MODEL_FEATURE_KEYWORDS,
    "upstream_files": {k: (str(v) if v is not None else None) for k, v in UPSTREAM_FILES.items()},
}

manifest_path = OUT_AUDIT_DIR / f"01_front_middle_forecast_repair_manifest_{TIMESTAMP}.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

manifest_table = pd.DataFrame([
    {"item": k, "value": json.dumps(v) if isinstance(v, (dict, list)) else v}
    for k, v in manifest.items()
])
manifest_csv_path = OUT_AUDIT_DIR / f"01_front_middle_forecast_repair_manifest_{TIMESTAMP}.csv"
manifest_table.to_csv(manifest_csv_path, index=False)

print("=" * 100)
print("Cell 1 — Setup complete")
print("=" * 100)
print(f"Project root:             {PROJECT_ROOT}")
print(f"New branch:               {NEW_BRANCH}")
print(f"Output processed dir:     {OUT_PROCESSED_DIR}")
print(f"Output audit dir:         {OUT_AUDIT_DIR}")
print(f"Run timestamp:            {TIMESTAMP}")
print()
print("Exact tenor grid:", EXACT_TENOR_GRID)
print("Research tenors: ", RESEARCH_TENORS)
print("Frozen Back tenors:", BACK_TENORS)
print()
print("Located upstream files:")
for k, v in UPSTREAM_FILES.items():
    print(f"  {k:30s} {v}")
print()
print(f"Saved manifest:           {manifest_path}")
print(f"Saved manifest CSV:       {manifest_csv_path}")
print("\nCELL 1 PASSED")


In [ ]:
# ======================================================================================
# Cell 2 — Build Front / Middle forecast-model research panel
# ======================================================================================
#
# Objective:
#   Build a clean date × tenor panel for diagnosing the current har_downside_v1
#   forecast denominator and preparing later Front/Middle denominator research.
#
# Scope:
#   - Load cross-tenor forward realized-variance targets.
#   - Load locked 30D date-level realized/downside features by exact column name.
#   - Load existing cross-tenor har_downside_v1 denominator forecasts.
#   - Load existing Core signal base to bring in outcome/context fields for diagnostics.
#   - Enforce exact tenor grid.
#   - Enforce target-window leakage fields are present.
#   - Audit forbidden model-feature fields.
#   - Mark Back tenors as frozen diagnostics only.
#   - Save research panel, column audit, validation audit, and feature audit.
#
# Runtime repair in this version:
#   - Parse YYYYMMDD integer/string date columns explicitly as calendar dates.
#     This prevents pandas from interpreting 20260701 as nanoseconds after 1970.
#   - Fail fast if the research panel collapses to a tiny 1970-era panel.
#
# Explicitly not in scope:
#   - No new feature engineering.
#   - No candidate model fitting.
#   - No threshold optimization.
#   - No Core Back refit or retest.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

# ======================================================================================
# 0. Configuration
# ======================================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

SOURCE_CORE_BUCKET_BRANCH = "vrp_core_bucket_parameters_v1"
SOURCE_CORSI_BRANCH = "forecast_model_corsi_v1"
SOURCE_30D_BRANCH = "vrp_30d_corsi_v1"
NEW_BRANCH = "vrp_front_middle_corsi_forecast_repair_v1"

SOURCE_CORE_BUCKET_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / SOURCE_CORE_BUCKET_BRANCH
SOURCE_CORSI_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / SOURCE_CORSI_BRANCH
SOURCE_30D_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / SOURCE_30D_BRANCH

OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

EXACT_TENOR_GRID = [9, 12, 15, 18, 21, 24, 27, 30, 33]
FRONT_TENORS = [9, 12, 15]
MIDDLE_TENORS = [18, 21, 24]
BACK_TENORS = [27, 30, 33]
RESEARCH_TENORS = FRONT_TENORS + MIDDLE_TENORS
TENOR_BUCKET_MAP = {
    9: "Front", 12: "Front", 15: "Front",
    18: "Middle", 21: "Middle", 24: "Middle",
    27: "Back", 30: "Back", 33: "Back",
}

LOCKED_HAR_DOWNSIDE_FEATURES = [
    "log_rv_1d",
    "log_rv_std_5d",
    "log_rv_std_21d",
    "log_rv_std_63d",
    "log_downside_rv_5d",
    "log_downside_rv_21d",
    "log_downside_rv_63d",
    "downside_share_21d",
    "negative_return_count_21d",
]

FORBIDDEN_MODEL_FEATURE_KEYWORDS = [
    "implied", "vix", "vrp", "z_", "rsi", "actual_dte", "win", "pnl",
    "credit", "strike", "expiry", "expiration", "option", "delta", "bucket", "tenor",
    "target", "forward", "last_forward", "trade_outcome",
]

BASELINE_FORECAST_VARIANCE_COL = "forecast_variance_har_downside_v1"
BASELINE_FORECAST_VOL_COL = "forecast_vol_har_downside_v1"
BASELINE_PREDICTED_LOG_COL = "predicted_log_variance_har_downside_v1"
TARGET_LOG_COL = "target_log_variance"

MIN_REASONABLE_PANEL_ROWS = 1_000
MIN_REASONABLE_DATE = pd.Timestamp("2018-01-01")

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def parse_project_date_series(s: pd.Series, source_name: str, column_name: str) -> pd.Series:
    """Robustly parse project date columns.

    The project often stores dates as YYYYMMDD integers. Plain pd.to_datetime on an
    integer interprets it as nanoseconds since 1970, which collapses 20180924 to
    1970-01-01 after normalize(). This parser handles YYYYMMDD explicitly.
    """
    raw = s.copy()
    out = pd.Series(pd.NaT, index=raw.index, dtype="datetime64[ns]")

    # Already datetime-like values.
    datetime_mask = raw.map(lambda x: isinstance(x, (pd.Timestamp, datetime, np.datetime64)))
    if datetime_mask.any():
        out.loc[datetime_mask] = pd.to_datetime(raw.loc[datetime_mask], errors="coerce")

    # Parse remaining values as either YYYYMMDD or generic date strings.
    remaining = ~datetime_mask
    if remaining.any():
        text = raw.loc[remaining].astype("string").str.strip()
        text = text.str.replace(r"\.0$", "", regex=True)
        text = text.replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "NaT": pd.NA})

        yyyymmdd_mask = text.str.fullmatch(r"\d{8}", na=False)
        if yyyymmdd_mask.any():
            idx = text.index[yyyymmdd_mask]
            out.loc[idx] = pd.to_datetime(text.loc[idx], format="%Y%m%d", errors="coerce")

        generic_mask = (~yyyymmdd_mask) & text.notna()
        if generic_mask.any():
            idx = text.index[generic_mask]
            out.loc[idx] = pd.to_datetime(text.loc[idx], errors="coerce")

    parsed = pd.to_datetime(out, errors="coerce").dt.normalize()

    non_null_raw = raw.notna().sum()
    parsed_non_null = parsed.notna().sum()
    if non_null_raw > 0 and parsed_non_null == 0:
        sample = raw.dropna().head(5).tolist()
        raise ValueError(
            f"Could not parse any dates for {source_name}.{column_name}; sample={sample}"
        )
    return parsed


def canonicalize_trade_date(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    df = df.copy()
    if "trade_date" not in df.columns:
        if "date" in df.columns:
            df["trade_date"] = df["date"]
        else:
            raise KeyError(f"{source_name} must include trade_date or date")
    df["trade_date"] = parse_project_date_series(df["trade_date"], source_name, "trade_date")
    if "date" not in df.columns:
        df["date"] = df["trade_date"]
    else:
        df["date"] = parse_project_date_series(df["date"], source_name, "date")
    return df


def canonicalize_tenor(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    df = df.copy()
    if "tenor" not in df.columns:
        if "target_days" in df.columns:
            df["tenor"] = df["target_days"]
        elif "dte" in df.columns:
            df["tenor"] = df["dte"]
        else:
            raise KeyError(f"{source_name} must include tenor, target_days, or dte")
    df["tenor"] = pd.to_numeric(df["tenor"], errors="coerce").astype("Int64")
    if df["tenor"].isna().any():
        bad_count = int(df["tenor"].isna().sum())
        raise ValueError(f"{source_name} has {bad_count:,} rows with unparseable tenor")
    df["tenor"] = df["tenor"].astype(int)
    return df


def add_target_log_variance(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    df = df.copy()
    if TARGET_LOG_COL in df.columns:
        df[TARGET_LOG_COL] = pd.to_numeric(df[TARGET_LOG_COL], errors="coerce")
        return df
    if "log_forward_realized_variance_corsi" in df.columns:
        df[TARGET_LOG_COL] = pd.to_numeric(df["log_forward_realized_variance_corsi"], errors="coerce")
        return df
    if "forward_realized_variance_corsi" in df.columns:
        x = pd.to_numeric(df["forward_realized_variance_corsi"], errors="coerce")
        df[TARGET_LOG_COL] = np.where(x > 0, np.log(x), np.nan)
        return df
    raise KeyError(
        f"{source_name} must include target_log_variance, "
        "log_forward_realized_variance_corsi, or forward_realized_variance_corsi"
    )


def forbidden_feature_hits(cols: list[str], forbidden_keywords: list[str]) -> list[dict]:
    hits = []
    for col in cols:
        col_lower = col.lower()
        for kw in forbidden_keywords:
            if kw in col_lower:
                hits.append({"feature": col, "forbidden_keyword": kw})
    return hits


def date_range_detail(df: pd.DataFrame) -> str:
    if len(df) == 0:
        return "empty"
    return f"rows={len(df):,}; min={df['trade_date'].min()}; max={df['trade_date'].max()}"

# ======================================================================================
# 2. Load upstream files
# ======================================================================================

denominator_path = latest_file(
    SOURCE_CORE_BUCKET_PROCESSED_DIR,
    "02B_cross_tenor_har_downside_denominator_panel_*.parquet",
    required=True,
)
signal_base_path = latest_file(
    SOURCE_CORE_BUCKET_PROCESSED_DIR,
    "02C_cross_tenor_core_signal_base_panel_*.parquet",
    required=True,
)
target_path = latest_file(
    SOURCE_CORSI_PROCESSED_DIR,
    "corsi_forward_realized_variance_targets_v1_*.parquet",
    required=True,
)
feature_30d_path = latest_file(
    SOURCE_30D_PROCESSED_DIR,
    "02B_30d_corsi_extended_feature_target_panel_*.parquet",
    required=True,
)

print("=" * 100)
print("Cell 2 — Loading upstream files")
print("=" * 100)
print(f"Denominator source:       {denominator_path}")
print(f"Signal base source:       {signal_base_path}")
print(f"Target source:            {target_path}")
print(f"Locked feature source:    {feature_30d_path}")

# Load.
den = pd.read_parquet(denominator_path)
sig = pd.read_parquet(signal_base_path)
target = pd.read_parquet(target_path)
features_30d = pd.read_parquet(feature_30d_path)

# Canonicalize identifiers.
den = canonicalize_tenor(canonicalize_trade_date(den, "denominator"), "denominator")
sig = canonicalize_tenor(canonicalize_trade_date(sig, "signal_base"), "signal_base")
target = canonicalize_tenor(canonicalize_trade_date(target, "target"), "target")
features_30d = canonicalize_trade_date(features_30d, "features_30d")

target = add_target_log_variance(target, "target")
if "last_forward_rv_date" not in target.columns:
    raise KeyError("Target source must include last_forward_rv_date for leakage diagnostics")
target["last_forward_rv_date"] = parse_project_date_series(
    target["last_forward_rv_date"], "target", "last_forward_rv_date"
)

# ======================================================================================
# 3. Validate exact grid and required feature columns
# ======================================================================================

validation_rows = []

def add_validation(check: str, status: str, detail: str):
    validation_rows.append({"check": check, "status": status, "detail": str(detail)})

for name, df in [("denominator", den), ("signal_base", sig), ("target", target)]:
    found_tenors = sorted(df["tenor"].dropna().astype(int).unique().tolist())
    status = "PASS" if found_tenors == EXACT_TENOR_GRID else "FAIL"
    add_validation(f"exact_grid_{name}", status, f"found={found_tenors}")

for name, df in [
    ("denominator", den),
    ("signal_base", sig),
    ("target", target),
    ("features_30d", features_30d),
]:
    min_date = df["trade_date"].min()
    max_date = df["trade_date"].max()
    date_ok = pd.notna(min_date) and pd.notna(max_date) and max_date >= MIN_REASONABLE_DATE
    add_validation(
        f"reasonable_date_range_{name}",
        "PASS" if date_ok else "FAIL",
        date_range_detail(df),
    )

missing_locked_features = [c for c in LOCKED_HAR_DOWNSIDE_FEATURES if c not in features_30d.columns]
add_validation(
    "locked_feature_columns_present",
    "PASS" if not missing_locked_features else "FAIL",
    f"missing={missing_locked_features}",
)
if missing_locked_features:
    raise KeyError(f"Missing locked feature columns: {missing_locked_features}")

for col in [BASELINE_FORECAST_VARIANCE_COL]:
    add_validation(
        f"baseline_column_present_{col}",
        "PASS" if col in den.columns else "FAIL",
        "present" if col in den.columns else "missing",
    )
    if col not in den.columns:
        raise KeyError(f"Denominator panel missing {col}")

# ======================================================================================
# 4. Build base target × feature panel
# ======================================================================================

# Keep exact grid only.
target = target[target["tenor"].isin(EXACT_TENOR_GRID)].copy()
den = den[den["tenor"].isin(EXACT_TENOR_GRID)].copy()
sig = sig[sig["tenor"].isin(EXACT_TENOR_GRID)].copy()

# Target columns to retain.
target_keep_cols = [
    "trade_date",
    "date",
    "tenor",
    TARGET_LOG_COL,
    "last_forward_rv_date",
]
for c in ["forward_realized_variance_corsi", "log_forward_realized_variance_corsi"]:
    if c in target.columns and c not in target_keep_cols:
        target_keep_cols.append(c)

target_small = target[target_keep_cols].drop_duplicates(["trade_date", "tenor"])

feature_keep_cols = ["trade_date"] + LOCKED_HAR_DOWNSIDE_FEATURES
features_small = features_30d[feature_keep_cols].drop_duplicates("trade_date")

panel = target_small.merge(features_small, on="trade_date", how="left", validate="many_to_one")

# ======================================================================================
# 5. Join baseline forecasts and diagnostic context
# ======================================================================================

den_keep_cols = [
    "trade_date",
    "tenor",
    BASELINE_FORECAST_VARIANCE_COL,
]
for c in [
    BASELINE_FORECAST_VOL_COL,
    BASELINE_PREDICTED_LOG_COL,
    "implied_variance",
    "vix_style_vol",
    "forecast_vol_pct",
    "fit_status",
    "selected_alpha",
    "train_rows_used",
    "test_year",
]:
    if c in den.columns and c not in den_keep_cols:
        den_keep_cols.append(c)

den_small = den[den_keep_cols].drop_duplicates(["trade_date", "tenor"])
panel = panel.merge(den_small, on=["trade_date", "tenor"], how="left", validate="one_to_one")

# Outcome/context fields come from the clean signal base. These are not model features.
signal_context_cols = ["trade_date", "tenor"]
optional_context_cols = [
    "tenor_bucket",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "actual_dte",
    "win",
    "normalized_pnl_dollars",
    "normalized_pnl_pct",
    "entry_credit_points",
    "short_option_pnl_points",
    "expiry",
    "expiration",
    "strike",
]
for c in optional_context_cols:
    if c in sig.columns and c not in signal_context_cols:
        signal_context_cols.append(c)

sig_small = sig[signal_context_cols].drop_duplicates(["trade_date", "tenor"])
# Avoid suffix collision if a column came from denominator already.
overlap = [c for c in sig_small.columns if c in panel.columns and c not in ["trade_date", "tenor"]]
if overlap:
    sig_small = sig_small.drop(columns=overlap)

panel = panel.merge(sig_small, on=["trade_date", "tenor"], how="left", validate="one_to_one")

# Standard bucket fields.
panel["tenor_bucket"] = panel["tenor"].map(TENOR_BUCKET_MAP)
panel["is_research_tenor"] = panel["tenor"].isin(RESEARCH_TENORS)
panel["is_back_frozen"] = panel["tenor"].isin(BACK_TENORS)
panel["forecast_repair_scope"] = np.select(
    [panel["tenor"].isin(FRONT_TENORS), panel["tenor"].isin(MIDDLE_TENORS), panel["tenor"].isin(BACK_TENORS)],
    ["RESEARCH_FRONT", "RESEARCH_MIDDLE", "FROZEN_BACK_DIAGNOSTIC"],
    default="OUT_OF_SCOPE",
)

# If predicted log was not saved, derive it from variance.
if BASELINE_PREDICTED_LOG_COL not in panel.columns:
    panel[BASELINE_PREDICTED_LOG_COL] = np.nan
mask = panel[BASELINE_PREDICTED_LOG_COL].isna() & (panel[BASELINE_FORECAST_VARIANCE_COL] > 0)
panel.loc[mask, BASELINE_PREDICTED_LOG_COL] = np.log(panel.loc[mask, BASELINE_FORECAST_VARIANCE_COL])

# Derived target/forecast diagnostics for later cells.
panel["target_realized_variance"] = np.exp(panel[TARGET_LOG_COL])
panel["baseline_forecast_variance"] = panel[BASELINE_FORECAST_VARIANCE_COL]
panel["baseline_forecast_vol_pct"] = np.sqrt(panel["baseline_forecast_variance"].where(panel["baseline_forecast_variance"] > 0)) * 100
panel["target_realized_vol_pct"] = np.sqrt(panel["target_realized_variance"].where(panel["target_realized_variance"] > 0)) * 100
panel["baseline_log_error_pred_minus_actual"] = panel[BASELINE_PREDICTED_LOG_COL] - panel[TARGET_LOG_COL]
panel["baseline_underforecast_flag"] = panel[BASELINE_PREDICTED_LOG_COL] < panel[TARGET_LOG_COL]
panel["baseline_actual_to_forecast_var_ratio"] = panel["target_realized_variance"] / panel["baseline_forecast_variance"]

# ======================================================================================
# 6. Leakage and forbidden-feature audits
# ======================================================================================

# This panel is diagnostic, but future model fitting must use these fields.
panel["target_window_overlaps_trade_date"] = panel["last_forward_rv_date"] >= panel["trade_date"]
panel["target_window_missing"] = panel["last_forward_rv_date"].isna()

# Only the locked feature list is approved as model-input columns in Cells 1-3.
approved_model_feature_cols = LOCKED_HAR_DOWNSIDE_FEATURES.copy()
forbidden_hits = forbidden_feature_hits(approved_model_feature_cols, FORBIDDEN_MODEL_FEATURE_KEYWORDS)
forbidden_audit = pd.DataFrame(forbidden_hits)
if forbidden_audit.empty:
    forbidden_audit = pd.DataFrame(columns=["feature", "forbidden_keyword"])

add_validation(
    "approved_model_features_no_forbidden_keywords",
    "PASS" if forbidden_audit.empty else "FAIL",
    forbidden_audit.to_dict("records") if not forbidden_audit.empty else "no forbidden hits",
)

# Missingness checks.
required_panel_cols = [
    "trade_date", "tenor", TARGET_LOG_COL, "last_forward_rv_date",
    BASELINE_FORECAST_VARIANCE_COL, BASELINE_PREDICTED_LOG_COL,
] + LOCKED_HAR_DOWNSIDE_FEATURES
missing_required_cols = [c for c in required_panel_cols if c not in panel.columns]
add_validation(
    "research_panel_required_columns_present",
    "PASS" if not missing_required_cols else "FAIL",
    f"missing={missing_required_cols}",
)
if missing_required_cols:
    raise KeyError(f"Research panel missing required columns: {missing_required_cols}")

nonfinite_counts = {}
for c in [TARGET_LOG_COL, BASELINE_FORECAST_VARIANCE_COL, BASELINE_PREDICTED_LOG_COL] + LOCKED_HAR_DOWNSIDE_FEATURES:
    nonfinite_counts[c] = int((~np.isfinite(pd.to_numeric(panel[c], errors="coerce"))).sum())

valid_diag_mask = (
    np.isfinite(pd.to_numeric(panel[TARGET_LOG_COL], errors="coerce"))
    & np.isfinite(pd.to_numeric(panel[BASELINE_FORECAST_VARIANCE_COL], errors="coerce"))
    & np.isfinite(pd.to_numeric(panel[BASELINE_PREDICTED_LOG_COL], errors="coerce"))
    & (pd.to_numeric(panel[BASELINE_FORECAST_VARIANCE_COL], errors="coerce") > 0)
)
valid_diag_rows = int(valid_diag_mask.sum())

add_validation("panel_rows", "INFO", str(len(panel)))
add_validation("panel_date_range", "INFO", f"{panel['trade_date'].min().date()} to {panel['trade_date'].max().date()}")
add_validation("nonfinite_counts", "INFO", json.dumps(nonfinite_counts))
add_validation(
    "panel_row_count_reasonable",
    "PASS" if len(panel) >= MIN_REASONABLE_PANEL_ROWS else "FAIL",
    f"rows={len(panel):,}; minimum_expected={MIN_REASONABLE_PANEL_ROWS:,}",
)
add_validation(
    "valid_diagnostic_rows_positive",
    "PASS" if valid_diag_rows >= MIN_REASONABLE_PANEL_ROWS else "FAIL",
    f"valid_diag_rows={valid_diag_rows:,}; minimum_expected={MIN_REASONABLE_PANEL_ROWS:,}",
)
add_validation(
    "panel_not_1970_date_parse_failure",
    "PASS" if panel["trade_date"].max() >= MIN_REASONABLE_DATE else "FAIL",
    f"date_min={panel['trade_date'].min()}; date_max={panel['trade_date'].max()}",
)

# Duplicate check.
dup_count = int(panel.duplicated(["trade_date", "tenor"]).sum())
add_validation("no_duplicate_trade_date_tenor", "PASS" if dup_count == 0 else "FAIL", f"duplicates={dup_count}")

# Back rows are present, but only as diagnostics.
back_rows = int(panel["is_back_frozen"].sum())
research_rows = int(panel["is_research_tenor"].sum())
add_validation("back_rows_frozen_diagnostic_only", "PASS" if back_rows > 0 else "FAIL", f"back_rows={back_rows}")
add_validation("front_middle_research_rows", "PASS" if research_rows > 0 else "FAIL", f"research_rows={research_rows}")

# ======================================================================================
# 7. Column and feature inventory audits
# ======================================================================================

column_audit = pd.DataFrame([
    {
        "column": c,
        "dtype": str(panel[c].dtype),
        "non_null_rows": int(panel[c].notna().sum()),
        "null_rows": int(panel[c].isna().sum()),
        "is_approved_model_feature_cell_1_3": c in approved_model_feature_cols,
        "is_forbidden_as_model_feature": any(kw in c.lower() for kw in FORBIDDEN_MODEL_FEATURE_KEYWORDS),
    }
    for c in panel.columns
]).sort_values(["is_approved_model_feature_cell_1_3", "column"], ascending=[False, True])

feature_inventory = pd.DataFrame([
    {
        "feature": c,
        "approved_for_cells_1_3": c in approved_model_feature_cols,
        "source": "locked_30d_feature_source" if c in LOCKED_HAR_DOWNSIDE_FEATURES else "diagnostic_context",
        "forbidden_hit_as_model_feature": any(kw in c.lower() for kw in FORBIDDEN_MODEL_FEATURE_KEYWORDS),
        "non_null_rows": int(panel[c].notna().sum()) if c in panel.columns else 0,
    }
    for c in sorted(set(approved_model_feature_cols + list(panel.columns)))
])

validation = pd.DataFrame(validation_rows)
failures = validation[validation["status"] == "FAIL"]
if not failures.empty:
    print("Cell 2 validation failed before saving research panel:")
    display(validation)
    raise RuntimeError("Cell 2 validation failed. See validation rows above.")

# ======================================================================================
# 8. Save outputs
# ======================================================================================

panel_path = OUT_PROCESSED_DIR / f"02_front_middle_forecast_research_panel_{panel['trade_date'].min():%Y%m%d}_{panel['trade_date'].max():%Y%m%d}_{TIMESTAMP}.parquet"
validation_path = OUT_AUDIT_DIR / f"02_front_middle_forecast_research_panel_validation_{TIMESTAMP}.csv"
column_audit_path = OUT_AUDIT_DIR / f"02_front_middle_forecast_research_panel_column_audit_{TIMESTAMP}.csv"
feature_inventory_path = OUT_AUDIT_DIR / f"02_front_middle_forecast_research_panel_feature_inventory_{TIMESTAMP}.csv"
forbidden_audit_path = OUT_AUDIT_DIR / f"02_front_middle_forecast_research_panel_forbidden_feature_audit_{TIMESTAMP}.csv"

panel.to_parquet(panel_path, index=False)
validation.to_csv(validation_path, index=False)
column_audit.to_csv(column_audit_path, index=False)
feature_inventory.to_csv(feature_inventory_path, index=False)
forbidden_audit.to_csv(forbidden_audit_path, index=False)

print("=" * 100)
print("Cell 2 — Forecast research panel built")
print("=" * 100)
print(f"Rows:                    {len(panel):,}")
print(f"Date range:              {panel['trade_date'].min().date()} to {panel['trade_date'].max().date()}")
print(f"Tenors:                  {sorted(panel['tenor'].dropna().astype(int).unique().tolist())}")
print(f"Valid diagnostic rows:   {valid_diag_rows:,}")
print(f"Research rows F/M:       {research_rows:,}")
print(f"Frozen Back rows:        {back_rows:,}")
print(f"Approved model features: {approved_model_feature_cols}")
print()
print("Rows by forecast repair scope:")
print(panel["forecast_repair_scope"].value_counts(dropna=False).to_string())
print()
print("Rows by bucket:")
print(panel["tenor_bucket"].value_counts(dropna=False).to_string())
print()
print(f"Saved panel:             {panel_path}")
print(f"Saved validation:        {validation_path}")
print(f"Saved column audit:      {column_audit_path}")
print(f"Saved feature inventory: {feature_inventory_path}")
print(f"Saved forbidden audit:   {forbidden_audit_path}")


In [ ]:
# ======================================================================================
# Cell 3 — Baseline har_downside_v1 diagnostics by tenor / bucket / year
# ======================================================================================
#
# Objective:
#   Diagnose exactly how the current locked har_downside_v1 denominator behaves by
#   tenor and bucket before researching any Front/Middle repairs.
#
# Scope:
#   - Load latest Cell 2 research panel.
#   - Compare baseline forecast variance vs realized forward Corsi variance.
#   - Report diagnostics by tenor, bucket, and year.
#   - Identify underforecasting concentration and worst underforecast rows.
#   - Compare diagnostics with optional outcome/P&L context when available.
#   - Keep Back as frozen diagnostic comparator only.
#
# Runtime repair in this version:
#   - Fail fast if the loaded panel has no valid diagnostic rows or still appears to be
#     affected by a 1970-style date parse collapse.
#
# Explicitly not in scope:
#   - No new model fitting.
#   - No feature engineering.
#   - No threshold optimization.
#   - No Back refit or retest.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

# ======================================================================================
# 0. Configuration
# ======================================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
NEW_BRANCH = "vrp_front_middle_corsi_forecast_repair_v1"
OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

EXACT_TENOR_GRID = [9, 12, 15, 18, 21, 24, 27, 30, 33]
FRONT_TENORS = [9, 12, 15]
MIDDLE_TENORS = [18, 21, 24]
BACK_TENORS = [27, 30, 33]

TARGET_LOG_COL = "target_log_variance"
PRED_LOG_COL = "predicted_log_variance_har_downside_v1"
FORECAST_VAR_COL = "baseline_forecast_variance"
TARGET_VAR_COL = "target_realized_variance"

UNDERFORECAST_RATIO_THRESHOLDS = [1.25, 1.50, 2.00]
WORST_UNDERFORECAST_ROWS = 100
WORST_PNL_ROWS = 100
MIN_REASONABLE_VALID_ROWS = 1_000
MIN_REASONABLE_DATE = pd.Timestamp("2018-01-01")

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def safe_rmse(x: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce").dropna()
    return float(np.sqrt(np.mean(np.square(x)))) if len(x) else np.nan


def safe_mae(x: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce").dropna()
    return float(np.mean(np.abs(x))) if len(x) else np.nan


def summarize_group(g: pd.DataFrame) -> pd.Series:
    e = pd.to_numeric(g["baseline_log_error_pred_minus_actual"], errors="coerce")
    ratio = pd.to_numeric(g["baseline_actual_to_forecast_var_ratio"], errors="coerce")
    actual_vol = pd.to_numeric(g["target_realized_vol_pct"], errors="coerce")
    forecast_vol = pd.to_numeric(g["baseline_forecast_vol_pct"], errors="coerce")
    out = {
        "rows": int(len(g)),
        "date_min": g["trade_date"].min(),
        "date_max": g["trade_date"].max(),
        "rmse_log": safe_rmse(e),
        "mae_log": safe_mae(e),
        "bias_log_pred_minus_actual": float(e.mean()) if e.notna().any() else np.nan,
        "median_log_error": float(e.median()) if e.notna().any() else np.nan,
        "underforecast_rate": float((e < 0).mean()) if e.notna().any() else np.nan,
        "actual_to_forecast_var_ratio_mean": float(ratio.mean()) if ratio.notna().any() else np.nan,
        "actual_to_forecast_var_ratio_median": float(ratio.median()) if ratio.notna().any() else np.nan,
        "target_realized_vol_pct_mean": float(actual_vol.mean()) if actual_vol.notna().any() else np.nan,
        "baseline_forecast_vol_pct_mean": float(forecast_vol.mean()) if forecast_vol.notna().any() else np.nan,
        "large_underforecast_1p5x_rate": float((ratio > 1.50).mean()) if ratio.notna().any() else np.nan,
        "large_underforecast_2p0x_rate": float((ratio > 2.00).mean()) if ratio.notna().any() else np.nan,
        "actual_top_decile_rows": int(g.get("actual_top_decile_by_tenor", pd.Series(dtype=bool)).sum()) if "actual_top_decile_by_tenor" in g else 0,
        "forecast_top_decile_rows": int(g.get("forecast_top_decile_by_tenor", pd.Series(dtype=bool)).sum()) if "forecast_top_decile_by_tenor" in g else 0,
        "missed_top_decile_actual_rate": float(g["top_decile_actual_missed_by_forecast"].mean()) if "top_decile_actual_missed_by_forecast" in g and len(g) else np.nan,
    }
    if "win" in g.columns:
        win = pd.to_numeric(g["win"], errors="coerce")
        out["win_rate_context"] = float(win.mean()) if win.notna().any() else np.nan
    if "normalized_pnl_dollars" in g.columns:
        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        out["avg_normalized_pnl_dollars"] = float(pnl.mean()) if pnl.notna().any() else np.nan
        out["worst_normalized_pnl_dollars"] = float(pnl.min()) if pnl.notna().any() else np.nan
    if "normalized_pnl_pct" in g.columns:
        pnl_pct = pd.to_numeric(g["normalized_pnl_pct"], errors="coerce")
        out["avg_normalized_pnl_pct"] = float(pnl_pct.mean()) if pnl_pct.notna().any() else np.nan
        out["worst_normalized_pnl_pct"] = float(pnl_pct.min()) if pnl_pct.notna().any() else np.nan
    return pd.Series(out)

# ======================================================================================
# 2. Load research panel
# ======================================================================================

panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "02_front_middle_forecast_research_panel_*.parquet",
    required=True,
)
panel = pd.read_parquet(panel_path)
panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce").dt.normalize()
panel["year"] = panel["trade_date"].dt.year

required_cols = [
    "trade_date", "tenor", "tenor_bucket", TARGET_LOG_COL, PRED_LOG_COL,
    FORECAST_VAR_COL, TARGET_VAR_COL, "baseline_log_error_pred_minus_actual",
    "baseline_actual_to_forecast_var_ratio", "baseline_forecast_vol_pct",
    "target_realized_vol_pct", "forecast_repair_scope", "is_back_frozen",
]
missing = [c for c in required_cols if c not in panel.columns]
if missing:
    raise KeyError(f"Research panel missing required columns for Cell 3: {missing}")

if len(panel) < MIN_REASONABLE_VALID_ROWS or panel["trade_date"].max() < MIN_REASONABLE_DATE:
    raise RuntimeError(
        "Cell 3 refusing to run: latest Cell 2 panel is too small or has bad dates. "
        f"panel_rows={len(panel):,}; date_min={panel['trade_date'].min()}; date_max={panel['trade_date'].max()}. "
        "Rerun patched Cell 2 and confirm it builds a full 2018–2026 panel."
    )

# Keep rows with finite target and forecast diagnostics.
for c in [TARGET_LOG_COL, PRED_LOG_COL, FORECAST_VAR_COL, TARGET_VAR_COL, "baseline_log_error_pred_minus_actual"]:
    panel[c] = pd.to_numeric(panel[c], errors="coerce")

valid = panel[
    np.isfinite(panel[TARGET_LOG_COL])
    & np.isfinite(panel[PRED_LOG_COL])
    & np.isfinite(panel[FORECAST_VAR_COL])
    & np.isfinite(panel[TARGET_VAR_COL])
    & (panel[FORECAST_VAR_COL] > 0)
    & (panel[TARGET_VAR_COL] > 0)
].copy()

if len(valid) < MIN_REASONABLE_VALID_ROWS:
    raise RuntimeError(
        "Cell 3 refusing to pass with too few valid diagnostics. "
        f"valid_rows={len(valid):,}; panel_rows={len(panel):,}. "
        "Check Cell 2 date parsing and joins before interpreting model diagnostics."
    )

# Percentile diagnostics by tenor.
valid["actual_var_pct_rank_by_tenor"] = valid.groupby("tenor")[TARGET_VAR_COL].rank(pct=True)
valid["forecast_var_pct_rank_by_tenor"] = valid.groupby("tenor")[FORECAST_VAR_COL].rank(pct=True)
valid["actual_top_decile_by_tenor"] = valid["actual_var_pct_rank_by_tenor"] >= 0.90
valid["forecast_top_decile_by_tenor"] = valid["forecast_var_pct_rank_by_tenor"] >= 0.90
valid["top_decile_actual_missed_by_forecast"] = valid["actual_top_decile_by_tenor"] & (~valid["forecast_top_decile_by_tenor"])
valid["large_underforecast_1p5x"] = valid["baseline_actual_to_forecast_var_ratio"] > 1.50
valid["large_underforecast_2p0x"] = valid["baseline_actual_to_forecast_var_ratio"] > 2.00

# ======================================================================================
# 3. Main diagnostics
# ======================================================================================

by_tenor = (
    valid.groupby(["tenor", "tenor_bucket", "forecast_repair_scope"], dropna=False)
    .apply(summarize_group, include_groups=False)
    .reset_index()
    .sort_values("tenor")
)

by_bucket = (
    valid.groupby(["tenor_bucket", "forecast_repair_scope"], dropna=False)
    .apply(summarize_group, include_groups=False)
    .reset_index()
)

by_bucket_year = (
    valid.groupby(["year", "tenor_bucket", "forecast_repair_scope"], dropna=False)
    .apply(summarize_group, include_groups=False)
    .reset_index()
    .sort_values(["year", "tenor_bucket"])
)

by_tenor_year = (
    valid.groupby(["year", "tenor", "tenor_bucket", "forecast_repair_scope"], dropna=False)
    .apply(summarize_group, include_groups=False)
    .reset_index()
    .sort_values(["year", "tenor"])
)

# High-realized-variance capture diagnostics.
top_decile_capture_by_tenor = (
    valid[valid["actual_top_decile_by_tenor"]]
    .groupby(["tenor", "tenor_bucket", "forecast_repair_scope"], dropna=False)
    .agg(
        top_decile_rows=("trade_date", "size"),
        forecast_top_decile_capture_rate=("forecast_top_decile_by_tenor", "mean"),
        missed_top_decile_actual_rate=("top_decile_actual_missed_by_forecast", "mean"),
        underforecast_rate_in_actual_top_decile=("baseline_underforecast_flag", "mean"),
        large_underforecast_1p5x_rate=("large_underforecast_1p5x", "mean"),
        large_underforecast_2p0x_rate=("large_underforecast_2p0x", "mean"),
        avg_actual_to_forecast_var_ratio=("baseline_actual_to_forecast_var_ratio", "mean"),
        median_actual_to_forecast_var_ratio=("baseline_actual_to_forecast_var_ratio", "median"),
    )
    .reset_index()
    .sort_values("tenor")
)

# Worst underforecasts.
worst_underforecasts = (
    valid.sort_values("baseline_actual_to_forecast_var_ratio", ascending=False)
    .head(WORST_UNDERFORECAST_ROWS)
    .copy()
)
worst_under_cols = [
    "trade_date", "year", "tenor", "tenor_bucket", "forecast_repair_scope",
    "target_realized_vol_pct", "baseline_forecast_vol_pct",
    "baseline_actual_to_forecast_var_ratio", "baseline_log_error_pred_minus_actual",
    "actual_top_decile_by_tenor", "forecast_top_decile_by_tenor",
]
for c in ["win", "normalized_pnl_dollars", "normalized_pnl_pct", "model_vrp_log", "model_vrp_z_3m", "model_vrp_z_1y", "RSI14"]:
    if c in worst_underforecasts.columns and c not in worst_under_cols:
        worst_under_cols.append(c)
worst_underforecasts = worst_underforecasts[worst_under_cols]

# Optional bad-outcome overlap diagnostics.
bad_outcome_overlap = pd.DataFrame()
worst_pnl_rows = pd.DataFrame()
if "normalized_pnl_dollars" in valid.columns:
    pnl_valid = valid[pd.to_numeric(valid["normalized_pnl_dollars"], errors="coerce").notna()].copy()
    pnl_valid["normalized_pnl_dollars"] = pd.to_numeric(pnl_valid["normalized_pnl_dollars"], errors="coerce")
    if len(pnl_valid):
        pnl_valid["pnl_bottom_decile_by_tenor"] = pnl_valid.groupby("tenor")["normalized_pnl_dollars"].rank(pct=True) <= 0.10
        worst_pnl_rows = pnl_valid.sort_values("normalized_pnl_dollars", ascending=True).head(WORST_PNL_ROWS).copy()
        bad_outcome_overlap = (
            pnl_valid[pnl_valid["pnl_bottom_decile_by_tenor"]]
            .groupby(["tenor", "tenor_bucket", "forecast_repair_scope"], dropna=False)
            .agg(
                bottom_decile_pnl_rows=("trade_date", "size"),
                avg_bottom_decile_pnl=("normalized_pnl_dollars", "mean"),
                worst_bottom_decile_pnl=("normalized_pnl_dollars", "min"),
                underforecast_rate_in_bottom_decile_pnl=("baseline_underforecast_flag", "mean"),
                large_underforecast_1p5x_rate=("large_underforecast_1p5x", "mean"),
                large_underforecast_2p0x_rate=("large_underforecast_2p0x", "mean"),
                avg_actual_to_forecast_var_ratio=("baseline_actual_to_forecast_var_ratio", "mean"),
                median_actual_to_forecast_var_ratio=("baseline_actual_to_forecast_var_ratio", "median"),
            )
            .reset_index()
            .sort_values("tenor")
        )
        worst_pnl_cols = [
            "trade_date", "year", "tenor", "tenor_bucket", "forecast_repair_scope",
            "normalized_pnl_dollars", "normalized_pnl_pct" if "normalized_pnl_pct" in worst_pnl_rows.columns else None,
            "win" if "win" in worst_pnl_rows.columns else None,
            "target_realized_vol_pct", "baseline_forecast_vol_pct",
            "baseline_actual_to_forecast_var_ratio", "baseline_log_error_pred_minus_actual",
            "large_underforecast_1p5x", "large_underforecast_2p0x",
        ]
        worst_pnl_cols = [c for c in worst_pnl_cols if c is not None and c in worst_pnl_rows.columns]
        worst_pnl_rows = worst_pnl_rows[worst_pnl_cols]

# Bucket failure summary focused on Front/Middle vs Back.
front_middle_vs_back = (
    valid.groupby("tenor_bucket", dropna=False)
    .agg(
        rows=("trade_date", "size"),
        rmse_log=("baseline_log_error_pred_minus_actual", lambda x: safe_rmse(x)),
        mae_log=("baseline_log_error_pred_minus_actual", lambda x: safe_mae(x)),
        bias_log_pred_minus_actual=("baseline_log_error_pred_minus_actual", "mean"),
        underforecast_rate=("baseline_underforecast_flag", "mean"),
        large_underforecast_1p5x_rate=("large_underforecast_1p5x", "mean"),
        large_underforecast_2p0x_rate=("large_underforecast_2p0x", "mean"),
        actual_top_decile_rows=("actual_top_decile_by_tenor", "sum"),
        missed_top_decile_actual_rate=("top_decile_actual_missed_by_forecast", "mean"),
        target_realized_vol_pct_mean=("target_realized_vol_pct", "mean"),
        baseline_forecast_vol_pct_mean=("baseline_forecast_vol_pct", "mean"),
    )
    .reset_index()
)

# ======================================================================================
# 4. Save outputs
# ======================================================================================

by_tenor_path = OUT_AUDIT_DIR / f"03_baseline_har_downside_diagnostics_by_tenor_{TIMESTAMP}.csv"
by_bucket_path = OUT_AUDIT_DIR / f"03_baseline_har_downside_diagnostics_by_bucket_{TIMESTAMP}.csv"
by_bucket_year_path = OUT_AUDIT_DIR / f"03_baseline_har_downside_diagnostics_by_bucket_year_{TIMESTAMP}.csv"
by_tenor_year_path = OUT_AUDIT_DIR / f"03_baseline_har_downside_diagnostics_by_tenor_year_{TIMESTAMP}.csv"
top_decile_path = OUT_AUDIT_DIR / f"03_baseline_har_downside_top_decile_capture_by_tenor_{TIMESTAMP}.csv"
worst_under_path = OUT_AUDIT_DIR / f"03_baseline_har_downside_worst_underforecasts_{TIMESTAMP}.csv"
bucket_failure_path = OUT_AUDIT_DIR / f"03_baseline_har_downside_front_middle_vs_back_summary_{TIMESTAMP}.csv"
bad_overlap_path = OUT_AUDIT_DIR / f"03_baseline_har_downside_bad_outcome_overlap_{TIMESTAMP}.csv"
worst_pnl_path = OUT_AUDIT_DIR / f"03_baseline_har_downside_worst_pnl_rows_{TIMESTAMP}.csv"

by_tenor.to_csv(by_tenor_path, index=False)
by_bucket.to_csv(by_bucket_path, index=False)
by_bucket_year.to_csv(by_bucket_year_path, index=False)
by_tenor_year.to_csv(by_tenor_year_path, index=False)
top_decile_capture_by_tenor.to_csv(top_decile_path, index=False)
worst_underforecasts.to_csv(worst_under_path, index=False)
front_middle_vs_back.to_csv(bucket_failure_path, index=False)
if not bad_outcome_overlap.empty:
    bad_outcome_overlap.to_csv(bad_overlap_path, index=False)
if not worst_pnl_rows.empty:
    worst_pnl_rows.to_csv(worst_pnl_path, index=False)

# ======================================================================================
# 5. Display concise review tables
# ======================================================================================

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

print("=" * 100)
print("Cell 3 — Baseline har_downside_v1 diagnostics")
print("=" * 100)
print(f"Research panel source:    {panel_path}")
print(f"Valid diagnostic rows:    {len(valid):,} / {len(panel):,}")
print(f"Date range:               {valid['trade_date'].min().date()} to {valid['trade_date'].max().date()}")
print()
print("Front/Middle vs Back summary:")
display(front_middle_vs_back)
print()
print("By tenor diagnostics:")
display(by_tenor)
print()
print("Top-decile realized variance capture by tenor:")
display(top_decile_capture_by_tenor)
print()
print("Worst underforecast rows:")
display(worst_underforecasts.head(25))
if not bad_outcome_overlap.empty:
    print()
    print("Bad-outcome overlap diagnostics:")
    display(bad_outcome_overlap)
if not worst_pnl_rows.empty:
    print()
    print("Worst P&L rows with forecast diagnostics:")
    display(worst_pnl_rows.head(25))

print()
print("Saved diagnostics:")
for p in [
    by_tenor_path,
    by_bucket_path,
    by_bucket_year_path,
    by_tenor_year_path,
    top_decile_path,
    worst_under_path,
    bucket_failure_path,
]:
    print(f"  {p}")
if not bad_outcome_overlap.empty:
    print(f"  {bad_overlap_path}")
if not worst_pnl_rows.empty:
    print(f"  {worst_pnl_path}")
print("\nCELL 3 PASSED")


In [ ]:
# ======================================================================================
# Cell 4 — Front/Middle fast-state and shock feature construction audit
# ======================================================================================
#
# Objective:
#   Build and audit candidate realized-price-only features designed to improve
#   Front/Middle forecast-denominator behavior in high-forward-realized-variance states.
#
# Scope:
#   - Load the latest Cell 2 research panel.
#   - Locate a daily SPX close/log-return source.
#   - Construct trailing, prior/through-trade-date realized-price-only features:
#       fast realized variance
#       fast downside realized variance
#       return shock features
#       volatility/downside acceleration features
#       negative-return/downside clustering features
#   - Join candidate date-level features to the date × tenor research panel.
#   - Audit missingness, forbidden-feature terms, SPX source selection, and
#     association with actual top-decile forward realized variance.
#   - Save an enriched candidate-feature panel and audit CSVs.
#
# Explicitly not in scope:
#   - No model fitting.
#   - No alpha selection.
#   - No signal threshold optimization.
#   - No Back refit or Back overwrite.
#   - No Secondary.
#   - No sizing.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

# ======================================================================================
# 0. Configuration
# ======================================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
NEW_BRANCH = "vrp_front_middle_corsi_forecast_repair_v1"
SOURCE_30D_BRANCH = "vrp_30d_corsi_v1"

OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
SOURCE_30D_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / SOURCE_30D_BRANCH
OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

EXACT_TENOR_GRID = [9, 12, 15, 18, 21, 24, 27, 30, 33]
FRONT_TENORS = [9, 12, 15]
MIDDLE_TENORS = [18, 21, 24]
BACK_TENORS = [27, 30, 33]
RESEARCH_TENORS = FRONT_TENORS + MIDDLE_TENORS

TARGET_LOG_COL = "target_log_variance"
TARGET_VAR_COL = "target_realized_variance"
MIN_REASONABLE_PANEL_ROWS = 1_000
MIN_REASONABLE_DATE = pd.Timestamp("2018-01-01")
MAX_ALLOWED_FEATURE_MISSING_RATE = 0.20
EPS = 1e-12

FORBIDDEN_MODEL_FEATURE_KEYWORDS = [
    "implied", "vix", "vrp", "z_", "rsi", "actual_dte", "win", "pnl",
    "credit", "strike", "expiry", "expiration", "option", "delta", "bucket", "tenor",
    "target", "forward", "last_forward", "trade_outcome",
]

SPX_SOURCE_CANDIDATES = [
    (PROJECT_ROOT / "data" / "external", "spx_daily.parquet"),
    (PROJECT_ROOT / "data" / "external", "spx_daily*.parquet"),
    (PROJECT_ROOT / "data" / "external", "*spx*.parquet"),
    (PROJECT_ROOT / "data" / "processed" / "staging", "production_feature_panel*.parquet"),
    (SOURCE_30D_PROCESSED_DIR, "02B_30d_corsi_extended_feature_target_panel_*.parquet"),
]

CLOSE_COLUMN_CANDIDATES = [
    "spx_close", "spx", "close", "Close", "adj_close", "Adj Close", "px_last",
    "last", "price", "spx_price", "underlying_price", "index_close", "official_close",
]
RETURN_COLUMN_CANDIDATES = [
    "log_return", "log_ret", "spx_log_return", "spx_log_ret", "daily_log_return",
    "return_log", "ret_log", "spx_return", "return", "ret", "daily_return",
    "spx_return_pct", "return_pct",
]

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def parse_project_date_series(s: pd.Series, source_name: str, column_name: str) -> pd.Series:
    """Parse YYYYMMDD integers/strings and normal date-like values safely."""
    raw = s.copy()
    out = pd.Series(pd.NaT, index=raw.index, dtype="datetime64[ns]")

    datetime_mask = raw.map(lambda x: isinstance(x, (pd.Timestamp, datetime, np.datetime64)))
    if datetime_mask.any():
        out.loc[datetime_mask] = pd.to_datetime(raw.loc[datetime_mask], errors="coerce")

    remaining = ~datetime_mask
    if remaining.any():
        text = raw.loc[remaining].astype("string").str.strip()
        text = text.str.replace(r"\.0$", "", regex=True)
        text = text.replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "NaT": pd.NA})

        yyyymmdd_mask = text.str.fullmatch(r"\d{8}", na=False)
        if yyyymmdd_mask.any():
            idx = text.index[yyyymmdd_mask]
            out.loc[idx] = pd.to_datetime(text.loc[idx], format="%Y%m%d", errors="coerce")

        generic_mask = (~yyyymmdd_mask) & text.notna()
        if generic_mask.any():
            idx = text.index[generic_mask]
            out.loc[idx] = pd.to_datetime(text.loc[idx], errors="coerce")

    parsed = pd.to_datetime(out, errors="coerce").dt.normalize()
    if raw.notna().sum() > 0 and parsed.notna().sum() == 0:
        sample = raw.dropna().head(5).tolist()
        raise ValueError(f"Could not parse any dates for {source_name}.{column_name}; sample={sample}")
    return parsed


def infer_column(columns: list[str], candidates: list[str]) -> str | None:
    lower_map = {str(c).lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def canonicalize_spx_daily(raw_df: pd.DataFrame, source_name: str) -> tuple[pd.DataFrame, dict]:
    df = raw_df.copy()

    if "trade_date" not in df.columns:
        if "date" in df.columns:
            df["trade_date"] = df["date"]
        else:
            return pd.DataFrame(), {"usable": False, "reason": "missing trade_date/date"}

    df["trade_date"] = parse_project_date_series(df["trade_date"], source_name, "trade_date")
    df = df[df["trade_date"].notna()].copy()

    if df.empty:
        return pd.DataFrame(), {"usable": False, "reason": "no parseable dates"}

    close_col = infer_column(list(df.columns), CLOSE_COLUMN_CANDIDATES)
    return_col = infer_column(list(df.columns), RETURN_COLUMN_CANDIDATES)

    keep = ["trade_date"]
    if close_col is not None:
        keep.append(close_col)
    if return_col is not None and return_col not in keep:
        keep.append(return_col)

    df = df[keep].drop_duplicates("trade_date", keep="last").sort_values("trade_date")

    out = pd.DataFrame({"trade_date": df["trade_date"]})
    reason = []

    if close_col is not None:
        close = pd.to_numeric(df[close_col], errors="coerce")
        out["spx_close_for_features"] = close
        out["spx_log_return"] = np.log(close / close.shift(1))
        reason.append(f"close_col={close_col}")

    elif return_col is not None:
        r = pd.to_numeric(df[return_col], errors="coerce")
        return_col_lower = str(return_col).lower()

        if "pct" in return_col_lower or r.abs().quantile(0.99) > 0.50:
            r = r / 100.0

        if "log" in return_col_lower:
            out["spx_log_return"] = r
        else:
            out["spx_log_return"] = np.where(r > -1.0, np.log1p(r), np.nan)

        out["spx_close_for_features"] = np.nan
        reason.append(f"return_col={return_col}")

    else:
        return pd.DataFrame(), {"usable": False, "reason": "no close or return column found"}

    out = out.replace([np.inf, -np.inf], np.nan)
    usable_returns = int(out["spx_log_return"].notna().sum())

    if usable_returns < 100:
        return pd.DataFrame(), {
            "usable": False,
            "reason": f"too few usable returns: {usable_returns}",
            "close_col": close_col,
            "return_col": return_col,
        }

    meta = {
        "usable": True,
        "reason": "; ".join(reason),
        "close_col": close_col,
        "return_col": return_col,
        "rows": int(len(out)),
        "usable_return_rows": usable_returns,
        "date_min": out["trade_date"].min(),
        "date_max": out["trade_date"].max(),
    }
    return out, meta


def find_spx_daily_source() -> tuple[pd.DataFrame, Path, pd.DataFrame]:
    audit_rows = []

    for directory, pattern in SPX_SOURCE_CANDIDATES:
        for path in sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True):
            try:
                raw = pd.read_parquet(path)
                daily, meta = canonicalize_spx_daily(raw, str(path))
            except Exception as e:
                daily = pd.DataFrame()
                meta = {"usable": False, "reason": f"read/parse error: {type(e).__name__}: {e}"}

            row = {
                "path": str(path),
                "pattern": pattern,
                "usable": bool(meta.get("usable", False)),
                "reason": meta.get("reason"),
                "close_col": meta.get("close_col"),
                "return_col": meta.get("return_col"),
                "rows": meta.get("rows"),
                "usable_return_rows": meta.get("usable_return_rows"),
                "date_min": meta.get("date_min"),
                "date_max": meta.get("date_max"),
            }
            audit_rows.append(row)

            if meta.get("usable", False):
                return daily, path, pd.DataFrame(audit_rows)

    audit = pd.DataFrame(audit_rows)
    raise FileNotFoundError(
        "No usable SPX daily close/log-return source found. Reviewed candidates:\n"
        + (audit.to_string(index=False) if not audit.empty else "no candidate files found")
    )


def safe_corr(a: pd.Series, b: pd.Series) -> float:
    x = pd.to_numeric(a, errors="coerce")
    y = pd.to_numeric(b, errors="coerce")
    m = np.isfinite(x) & np.isfinite(y)

    if int(m.sum()) < 30:
        return np.nan
    if float(x[m].std()) == 0.0 or float(y[m].std()) == 0.0:
        return np.nan

    return float(np.corrcoef(x[m], y[m])[0, 1])


def forbidden_feature_hits(cols: list[str], forbidden_keywords: list[str]) -> pd.DataFrame:
    hits = []
    for col in cols:
        col_lower = col.lower()
        for kw in forbidden_keywords:
            if kw in col_lower:
                hits.append({"feature": col, "forbidden_keyword": kw})
    return pd.DataFrame(hits, columns=["feature", "forbidden_keyword"])


# ======================================================================================
# 2. Load latest Cell 2 panel and SPX daily source
# ======================================================================================

panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "02_front_middle_forecast_research_panel_*.parquet",
    required=True,
)

panel = pd.read_parquet(panel_path)
panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce").dt.normalize()
panel = panel[panel["tenor"].isin(EXACT_TENOR_GRID)].copy()

if len(panel) < MIN_REASONABLE_PANEL_ROWS or panel["trade_date"].max() < MIN_REASONABLE_DATE:
    raise RuntimeError(
        "Cell 4 refusing to run: latest Cell 2 panel is too small or has bad dates. "
        f"panel_rows={len(panel):,}; date_min={panel['trade_date'].min()}; date_max={panel['trade_date'].max()}."
    )

required_panel_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    TARGET_LOG_COL,
    TARGET_VAR_COL,
    "forecast_repair_scope",
    "is_back_frozen",
]
missing_panel_cols = [c for c in required_panel_cols if c not in panel.columns]
if missing_panel_cols:
    raise KeyError(f"Cell 4 missing required panel columns: {missing_panel_cols}")

spx_daily, spx_source_path, spx_source_audit = find_spx_daily_source()
spx_daily = spx_daily.sort_values("trade_date").reset_index(drop=True)


# ======================================================================================
# 3. Construct candidate date-level features
# ======================================================================================

feature_daily = spx_daily.copy()
feature_daily["ret_sq"] = feature_daily["spx_log_return"] ** 2
feature_daily["downside_ret_sq"] = np.where(
    feature_daily["spx_log_return"] < 0,
    feature_daily["spx_log_return"] ** 2,
    0.0,
)
feature_daily["abs_return_1d_raw"] = feature_daily["spx_log_return"].abs()
feature_daily["is_negative_return"] = (feature_daily["spx_log_return"] < 0).astype(float)

rv_windows = [2, 3, 5, 10, 21, 63]
for w in rv_windows:
    rv = feature_daily["ret_sq"].rolling(w, min_periods=w).mean() * 252.0
    downside_rv = feature_daily["downside_ret_sq"].rolling(w, min_periods=w).mean() * 252.0

    feature_daily[f"candidate_log_rv_{w}d"] = np.log(rv.clip(lower=EPS))
    feature_daily[f"candidate_log_downside_rv_{w}d"] = np.log(downside_rv.clip(lower=EPS))

shock_windows = [3, 5, 10]
feature_daily["candidate_abs_return_1d"] = feature_daily["abs_return_1d_raw"]

for w in shock_windows:
    feature_daily[f"candidate_max_abs_return_{w}d"] = (
        feature_daily["spx_log_return"].abs().rolling(w, min_periods=w).max()
    )
    feature_daily[f"candidate_min_return_{w}d"] = (
        feature_daily["spx_log_return"].rolling(w, min_periods=w).min()
    )

cluster_windows = [5, 10]
for w in cluster_windows:
    ret_sq_sum = feature_daily["ret_sq"].rolling(w, min_periods=w).sum()
    downside_sum = feature_daily["downside_ret_sq"].rolling(w, min_periods=w).sum()

    feature_daily[f"candidate_negative_return_count_{w}d"] = (
        feature_daily["is_negative_return"].rolling(w, min_periods=w).sum()
    )
    feature_daily[f"candidate_downside_share_{w}d"] = downside_sum / ret_sq_sum.replace(0.0, np.nan)

feature_daily["candidate_log_rv_3d_minus_21d"] = (
    feature_daily["candidate_log_rv_3d"] - feature_daily["candidate_log_rv_21d"]
)
feature_daily["candidate_log_rv_5d_minus_21d"] = (
    feature_daily["candidate_log_rv_5d"] - feature_daily["candidate_log_rv_21d"]
)
feature_daily["candidate_log_rv_10d_minus_63d"] = (
    feature_daily["candidate_log_rv_10d"] - feature_daily["candidate_log_rv_63d"]
)
feature_daily["candidate_log_downside_rv_5d_minus_21d"] = (
    feature_daily["candidate_log_downside_rv_5d"] - feature_daily["candidate_log_downside_rv_21d"]
)
feature_daily["candidate_log_downside_rv_10d_minus_63d"] = (
    feature_daily["candidate_log_downside_rv_10d"] - feature_daily["candidate_log_downside_rv_63d"]
)

FEATURE_GROUPS = {
    "fast_realized_variance": [
        "candidate_log_rv_2d",
        "candidate_log_rv_3d",
        "candidate_log_rv_5d",
        "candidate_log_rv_10d",
        "candidate_log_rv_21d",
        "candidate_log_rv_63d",
    ],
    "fast_downside_realized_variance": [
        "candidate_log_downside_rv_2d",
        "candidate_log_downside_rv_3d",
        "candidate_log_downside_rv_5d",
        "candidate_log_downside_rv_10d",
        "candidate_log_downside_rv_21d",
        "candidate_log_downside_rv_63d",
    ],
    "shock_features": [
        "candidate_abs_return_1d",
        "candidate_max_abs_return_3d",
        "candidate_max_abs_return_5d",
        "candidate_max_abs_return_10d",
        "candidate_min_return_3d",
        "candidate_min_return_5d",
        "candidate_min_return_10d",
    ],
    "acceleration_features": [
        "candidate_log_rv_3d_minus_21d",
        "candidate_log_rv_5d_minus_21d",
        "candidate_log_rv_10d_minus_63d",
        "candidate_log_downside_rv_5d_minus_21d",
        "candidate_log_downside_rv_10d_minus_63d",
    ],
    "clustering_features": [
        "candidate_negative_return_count_5d",
        "candidate_negative_return_count_10d",
        "candidate_downside_share_5d",
        "candidate_downside_share_10d",
    ],
}

CANDIDATE_FEATURE_COLS = [c for cols in FEATURE_GROUPS.values() for c in cols]
FEATURE_TO_GROUP = {c: g for g, cols in FEATURE_GROUPS.items() for c in cols}
FEATURE_DANGER_DIRECTION = {c: "high" for c in CANDIDATE_FEATURE_COLS}

for c in ["candidate_min_return_3d", "candidate_min_return_5d", "candidate_min_return_10d"]:
    FEATURE_DANGER_DIRECTION[c] = "low"

missing_candidate_cols = [c for c in CANDIDATE_FEATURE_COLS if c not in feature_daily.columns]
if missing_candidate_cols:
    raise KeyError(f"Candidate feature construction failed; missing columns: {missing_candidate_cols}")

feature_keep_cols = ["trade_date", "spx_close_for_features", "spx_log_return"] + CANDIDATE_FEATURE_COLS
feature_daily_small = feature_daily[feature_keep_cols].copy()


# ======================================================================================
# 4. Join features to research panel
# ======================================================================================

enriched = panel.merge(feature_daily_small, on="trade_date", how="left", validate="many_to_one")

if enriched.duplicated(["trade_date", "tenor"]).any():
    dup_count = int(enriched.duplicated(["trade_date", "tenor"]).sum())
    raise RuntimeError(f"Enriched panel has duplicate trade_date × tenor rows: {dup_count}")

enriched[TARGET_LOG_COL] = pd.to_numeric(enriched[TARGET_LOG_COL], errors="coerce")
enriched[TARGET_VAR_COL] = pd.to_numeric(enriched[TARGET_VAR_COL], errors="coerce")

valid = enriched[
    enriched["tenor"].isin(EXACT_TENOR_GRID)
    & np.isfinite(enriched[TARGET_LOG_COL])
    & np.isfinite(enriched[TARGET_VAR_COL])
    & (enriched[TARGET_VAR_COL] > 0)
].copy()

valid["actual_var_pct_rank_by_tenor"] = valid.groupby("tenor")[TARGET_VAR_COL].rank(pct=True)
valid["actual_top_decile_by_tenor"] = valid["actual_var_pct_rank_by_tenor"] >= 0.90


# ======================================================================================
# 5. Feature audits and top-decile association diagnostics
# ======================================================================================

feature_missingness = []
for c in CANDIDATE_FEATURE_COLS:
    non_null = int(enriched[c].notna().sum())
    missing = int(enriched[c].isna().sum())

    feature_missingness.append({
        "feature_group": FEATURE_TO_GROUP[c],
        "feature": c,
        "danger_direction": FEATURE_DANGER_DIRECTION[c],
        "rows": int(len(enriched)),
        "non_null_rows": non_null,
        "missing_rows": missing,
        "missing_rate": float(missing / len(enriched)) if len(enriched) else np.nan,
        "min": float(pd.to_numeric(enriched[c], errors="coerce").min()) if non_null else np.nan,
        "median": float(pd.to_numeric(enriched[c], errors="coerce").median()) if non_null else np.nan,
        "max": float(pd.to_numeric(enriched[c], errors="coerce").max()) if non_null else np.nan,
    })

feature_missingness = pd.DataFrame(feature_missingness).sort_values(["feature_group", "feature"])
forbidden_audit = forbidden_feature_hits(CANDIDATE_FEATURE_COLS, FORBIDDEN_MODEL_FEATURE_KEYWORDS)

association_rows = []
group_keys = ["tenor_bucket", "forecast_repair_scope"]

for feature in CANDIDATE_FEATURE_COLS:
    direction = FEATURE_DANGER_DIRECTION[feature]

    tmp = valid[
        [
            "trade_date",
            "tenor",
            "tenor_bucket",
            "forecast_repair_scope",
            TARGET_LOG_COL,
            TARGET_VAR_COL,
            "actual_top_decile_by_tenor",
            feature,
        ]
    ].copy()

    tmp[feature] = pd.to_numeric(tmp[feature], errors="coerce")
    tmp = tmp[np.isfinite(tmp[feature])].copy()

    if tmp.empty:
        continue

    pct_rank = tmp.groupby("tenor")[feature].rank(pct=True)

    if direction == "low":
        tmp["feature_extreme_decile_by_tenor"] = pct_rank <= 0.10
    else:
        tmp["feature_extreme_decile_by_tenor"] = pct_rank >= 0.90

    for keys, g in tmp.groupby(group_keys, dropna=False):
        bucket, scope = keys
        actual_top = g[g["actual_top_decile_by_tenor"]]
        non_top = g[~g["actual_top_decile_by_tenor"]]

        feature_mean_top = float(actual_top[feature].mean()) if len(actual_top) else np.nan
        feature_mean_non_top = float(non_top[feature].mean()) if len(non_top) else np.nan
        feature_std = float(g[feature].std()) if g[feature].notna().sum() > 1 else np.nan

        mean_diff = (
            feature_mean_top - feature_mean_non_top
            if np.isfinite(feature_mean_top) and np.isfinite(feature_mean_non_top)
            else np.nan
        )
        standardized_lift = (
            mean_diff / feature_std
            if np.isfinite(mean_diff) and np.isfinite(feature_std) and feature_std != 0
            else np.nan
        )
        mean_ratio = (
            feature_mean_top / feature_mean_non_top
            if np.isfinite(feature_mean_top) and np.isfinite(feature_mean_non_top) and feature_mean_non_top != 0
            else np.nan
        )
        capture_rate = float(actual_top["feature_extreme_decile_by_tenor"].mean()) if len(actual_top) else np.nan
        false_positive_rate = float(non_top["feature_extreme_decile_by_tenor"].mean()) if len(non_top) else np.nan

        association_rows.append({
            "feature_group": FEATURE_TO_GROUP[feature],
            "feature": feature,
            "danger_direction": direction,
            "tenor_bucket": bucket,
            "forecast_repair_scope": scope,
            "rows": int(len(g)),
            "actual_top_decile_rows": int(len(actual_top)),
            "corr_with_target_log_variance": safe_corr(g[feature], g[TARGET_LOG_COL]),
            "feature_mean_in_actual_top_decile": feature_mean_top,
            "feature_mean_outside_actual_top_decile": feature_mean_non_top,
            "feature_mean_top_minus_non_top": mean_diff,
            "feature_mean_ratio_top_vs_non_top": mean_ratio,
            "standardized_lift_top_vs_non_top": standardized_lift,
            "feature_extreme_decile_capture_rate": capture_rate,
            "feature_extreme_decile_false_positive_rate": false_positive_rate,
        })

association = pd.DataFrame(association_rows).sort_values(
    ["forecast_repair_scope", "feature_extreme_decile_capture_rate", "corr_with_target_log_variance"],
    ascending=[True, False, False],
)

front_middle_association = association[
    association["forecast_repair_scope"].isin(["RESEARCH_FRONT", "RESEARCH_MIDDLE"])
].copy()

front_middle_top_features = (
    front_middle_association
    .sort_values(
        ["forecast_repair_scope", "feature_extreme_decile_capture_rate", "standardized_lift_top_vs_non_top"],
        ascending=[True, False, False],
    )
    .groupby("forecast_repair_scope", group_keys=False)
    .head(20)
)


# ======================================================================================
# 6. Validation
# ======================================================================================

validation_rows = []

def add_validation(check: str, status: str, detail: str):
    validation_rows.append({"check": check, "status": status, "detail": detail})


spx_missing_rate = float(enriched["spx_log_return"].isna().mean()) if len(enriched) else np.nan
max_candidate_missing = float(feature_missingness["missing_rate"].max()) if not feature_missingness.empty else np.nan
features_over_missing_limit = feature_missingness[
    feature_missingness["missing_rate"] > MAX_ALLOWED_FEATURE_MISSING_RATE
]

add_validation("cell2_panel_loaded", "PASS", f"path={panel_path}; rows={len(panel):,}")
add_validation("spx_daily_source_selected", "PASS", str(spx_source_path))

add_validation(
    "spx_daily_source_covers_panel_dates",
    "PASS"
    if spx_daily["trade_date"].min() <= panel["trade_date"].min()
    and spx_daily["trade_date"].max() >= panel["trade_date"].max()
    else "WARN",
    (
        f"spx_range={spx_daily['trade_date'].min()} to {spx_daily['trade_date'].max()}; "
        f"panel_range={panel['trade_date'].min()} to {panel['trade_date'].max()}"
    ),
)

add_validation(
    "candidate_feature_columns_present",
    "PASS" if not missing_candidate_cols else "FAIL",
    f"missing={missing_candidate_cols}",
)
add_validation(
    "candidate_features_no_forbidden_keywords",
    "PASS" if forbidden_audit.empty else "FAIL",
    forbidden_audit.to_dict("records") if not forbidden_audit.empty else "no forbidden hits",
)
add_validation("enriched_panel_no_duplicate_trade_date_tenor", "PASS", "duplicates=0")
add_validation(
    "spx_log_return_missing_rate",
    "PASS" if spx_missing_rate <= MAX_ALLOWED_FEATURE_MISSING_RATE else "WARN",
    f"missing_rate={spx_missing_rate:.2%}",
)
add_validation(
    "candidate_feature_missing_rate",
    "PASS" if max_candidate_missing <= MAX_ALLOWED_FEATURE_MISSING_RATE else "WARN",
    f"max_missing_rate={max_candidate_missing:.2%}; over_limit={features_over_missing_limit['feature'].tolist()}",
)
add_validation("no_model_fitting_performed", "PASS", "Cell 4 only constructs and audits candidate features")
add_validation(
    "back_tenors_not_refit",
    "PASS",
    "Back rows carried in enriched panel for diagnostics only; no Back model columns overwritten",
)

validation = pd.DataFrame(validation_rows)
failures = validation[validation["status"] == "FAIL"]

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 4 validation failed. See validation table above.")


# ======================================================================================
# 7. Save outputs
# ======================================================================================

panel_min = enriched["trade_date"].min()
panel_max = enriched["trade_date"].max()

feature_panel_path = OUT_PROCESSED_DIR / (
    f"04_front_middle_candidate_feature_panel_{panel_min:%Y%m%d}_{panel_max:%Y%m%d}_{TIMESTAMP}.parquet"
)
spx_source_audit_path = OUT_AUDIT_DIR / f"04_candidate_feature_spx_source_audit_{TIMESTAMP}.csv"
validation_path = OUT_AUDIT_DIR / f"04_candidate_feature_validation_{TIMESTAMP}.csv"
missingness_path = OUT_AUDIT_DIR / f"04_candidate_feature_missingness_{TIMESTAMP}.csv"
forbidden_path = OUT_AUDIT_DIR / f"04_candidate_feature_forbidden_feature_audit_{TIMESTAMP}.csv"
association_path = OUT_AUDIT_DIR / f"04_candidate_feature_top_decile_association_by_bucket_{TIMESTAMP}.csv"
front_middle_top_path = OUT_AUDIT_DIR / f"04_candidate_feature_front_middle_top_features_{TIMESTAMP}.csv"
feature_group_path = OUT_AUDIT_DIR / f"04_candidate_feature_group_manifest_{TIMESTAMP}.json"

with open(feature_group_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "feature_groups": FEATURE_GROUPS,
            "feature_danger_direction": FEATURE_DANGER_DIRECTION,
            "spx_source_path": str(spx_source_path),
            "source_panel_path": str(panel_path),
            "notes": "Candidate features are realized-price-only and are not yet approved as final forecast-model inputs.",
        },
        f,
        indent=2,
    )

enriched.to_parquet(feature_panel_path, index=False)
spx_source_audit.to_csv(spx_source_audit_path, index=False)
validation.to_csv(validation_path, index=False)
feature_missingness.to_csv(missingness_path, index=False)
forbidden_audit.to_csv(forbidden_path, index=False)
association.to_csv(association_path, index=False)
front_middle_top_features.to_csv(front_middle_top_path, index=False)


# ======================================================================================
# 8. Display concise review
# ======================================================================================

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)

print("=" * 100)
print("Cell 4 — Front/Middle candidate feature construction audit")
print("=" * 100)
print(f"Source Cell 2 panel:       {panel_path}")
print(f"SPX daily source:          {spx_source_path}")
print(f"Enriched panel rows:       {len(enriched):,}")
print(f"Date range:                {panel_min.date()} to {panel_max.date()}")
print(f"Candidate feature count:   {len(CANDIDATE_FEATURE_COLS)}")
print(f"Front/Middle rows:         {int(enriched['tenor'].isin(RESEARCH_TENORS).sum()):,}")
print(f"Frozen Back rows:          {int(enriched['tenor'].isin(BACK_TENORS).sum()):,}")

print()
print("Validation:")
display(validation)

print()
print("Candidate feature missingness:")
display(feature_missingness)

print()
print("Front/Middle top candidate feature association diagnostics:")
display(front_middle_top_features)

print()
print("Saved outputs:")
for p in [
    feature_panel_path,
    spx_source_audit_path,
    validation_path,
    missingness_path,
    forbidden_path,
    association_path,
    front_middle_top_path,
    feature_group_path,
]:
    print(f"  {p}")

print("\nCELL 4 PASSED")

In [ ]:
# ======================================================================================
# Cell 5 — Front/Middle OOS forecast candidate sweep
# ======================================================================================
#
# Objective:
#   Fit and evaluate candidate realized-price-only forecast denominators for Front/Middle
#   tenors, using expanding yearly OOS Ridge models with the same leakage guard as the
#   locked har_downside_v1 methodology.
#
# Scope:
#   - Load latest Cell 4 enriched candidate-feature panel.
#   - Keep Back tenors frozen / diagnostic only.
#   - Fit candidate model families only for Front/Middle tenors:
#       9, 12, 15, 18, 21, 24
#   - Compare candidate OOS forecasts against existing har_downside_v1 baseline.
#   - Evaluate forecast quality, underforecast behavior, top-decile realized-variance
#     capture, and bottom-P&L overlap.
#
# Explicitly not in scope:
#   - No Back refit.
#   - No signal threshold sweep.
#   - No VRP/z-score rebuild yet.
#   - No final model selection.
#   - No Secondary.
#   - No sizing.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ======================================================================================
# 0. Configuration
# ======================================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
NEW_BRANCH = "vrp_front_middle_corsi_forecast_repair_v1"

OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

EXACT_TENOR_GRID = [9, 12, 15, 18, 21, 24, 27, 30, 33]
FRONT_TENORS = [9, 12, 15]
MIDDLE_TENORS = [18, 21, 24]
BACK_TENORS = [27, 30, 33]
RESEARCH_TENORS = FRONT_TENORS + MIDDLE_TENORS

TARGET_LOG_COL = "target_log_variance"
TARGET_VAR_COL = "target_realized_variance"

FORECAST_VAR_BASELINE_COL = "forecast_variance_har_downside_v1"
PRED_LOG_BASELINE_COL = "predicted_log_variance_har_downside_v1"

ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]
FALLBACK_ALPHA = 1.0

MIN_OUTER_TRAIN_ROWS = 250
MIN_INNER_TRAIN_ROWS = 250
MIN_INNER_VAL_ROWS = 30
EPS = 1e-12

FORBIDDEN_MODEL_FEATURE_KEYWORDS = [
    "implied", "vix", "vrp", "z_", "rsi", "actual_dte", "win", "pnl",
    "credit", "strike", "expiry", "expiration", "option", "delta", "bucket", "tenor",
    "target", "forward", "last_forward", "trade_outcome",
]

LOCKED_HAR_DOWNSIDE_FEATURES = [
    "log_rv_1d",
    "log_rv_std_5d",
    "log_rv_std_21d",
    "log_rv_std_63d",
    "log_downside_rv_5d",
    "log_downside_rv_21d",
    "log_downside_rv_63d",
    "downside_share_21d",
    "negative_return_count_21d",
]

FAST_RV_CORE = [
    "candidate_log_rv_3d",
    "candidate_log_rv_5d",
    "candidate_log_rv_10d",
    "candidate_log_rv_21d",
    "candidate_log_rv_63d",
]

FAST_DOWNSIDE_CORE = [
    "candidate_log_downside_rv_5d",
    "candidate_log_downside_rv_10d",
    "candidate_log_downside_rv_21d",
    "candidate_log_downside_rv_63d",
    "candidate_downside_share_5d",
    "candidate_downside_share_10d",
]

SHOCK_CORE = [
    "candidate_max_abs_return_3d",
    "candidate_max_abs_return_5d",
    "candidate_max_abs_return_10d",
    "candidate_min_return_3d",
    "candidate_min_return_5d",
    "candidate_min_return_10d",
]

ACCELERATION_CORE = [
    "candidate_log_rv_3d_minus_21d",
    "candidate_log_rv_5d_minus_21d",
    "candidate_log_rv_10d_minus_63d",
    "candidate_log_downside_rv_5d_minus_21d",
    "candidate_log_downside_rv_10d_minus_63d",
]

MODEL_FAMILIES = {
    "baseline_locked_features_refit": LOCKED_HAR_DOWNSIDE_FEATURES,
    "fast_rv_core": FAST_RV_CORE,
    "fast_downside_core": FAST_DOWNSIDE_CORE,
    "shock_core": SHOCK_CORE,
    "fast_rv_plus_shock": FAST_RV_CORE + SHOCK_CORE,
    "fast_downside_plus_shock": FAST_DOWNSIDE_CORE + SHOCK_CORE,
    "full_fast_state": FAST_RV_CORE + FAST_DOWNSIDE_CORE + SHOCK_CORE + ACCELERATION_CORE,
}

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def forbidden_feature_hits(cols: list[str], forbidden_keywords: list[str]) -> pd.DataFrame:
    hits = []
    for col in cols:
        col_lower = col.lower()
        for kw in forbidden_keywords:
            if kw in col_lower:
                hits.append({"feature": col, "forbidden_keyword": kw})
    return pd.DataFrame(hits, columns=["feature", "forbidden_keyword"])


def fit_ridge_predict(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    features: list[str],
    target_col: str,
    alpha: float,
) -> np.ndarray:
    pipe = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=alpha)),
        ]
    )
    pipe.fit(train_df[features].to_numpy(), train_df[target_col].to_numpy())
    return pipe.predict(test_df[features].to_numpy())


def choose_alpha_inner_yearly_cv(
    train_pool: pd.DataFrame,
    features: list[str],
    target_col: str,
    alpha_grid: list[float],
    fallback_alpha: float,
) -> tuple[float, pd.DataFrame]:
    """
    Train-only inner expanding yearly CV.

    For each validation year inside the outer training pool:
        inner_train rows must satisfy:
            trade_date < validation-year start
            last_forward_rv_date < validation-year start
        inner_val rows are rows from the validation year.

    This avoids letting realized windows that overlap the validation-year start leak
    into inner-training fits.
    """
    cv_rows = []

    available_years = sorted(train_pool["trade_date"].dt.year.dropna().astype(int).unique().tolist())

    for val_year in available_years:
        val_start = pd.Timestamp(f"{val_year}-01-01")
        val_end = pd.Timestamp(f"{val_year + 1}-01-01")

        inner_train = train_pool[
            (train_pool["trade_date"] < val_start)
            & (train_pool["last_forward_rv_date"] < val_start)
        ].copy()

        inner_val = train_pool[
            (train_pool["trade_date"] >= val_start)
            & (train_pool["trade_date"] < val_end)
        ].copy()

        inner_train = inner_train.dropna(subset=features + [target_col])
        inner_val = inner_val.dropna(subset=features + [target_col])

        if len(inner_train) < MIN_INNER_TRAIN_ROWS or len(inner_val) < MIN_INNER_VAL_ROWS:
            continue

        for alpha in alpha_grid:
            try:
                pred = fit_ridge_predict(inner_train, inner_val, features, target_col, alpha)
                rmse = float(np.sqrt(mean_squared_error(inner_val[target_col], pred)))
                mae = float(mean_absolute_error(inner_val[target_col], pred))
                bias = float(np.mean(pred - inner_val[target_col].to_numpy()))
                cv_rows.append({
                    "val_year": val_year,
                    "alpha": alpha,
                    "inner_train_rows": int(len(inner_train)),
                    "inner_val_rows": int(len(inner_val)),
                    "rmse_log": rmse,
                    "mae_log": mae,
                    "bias_log_pred_minus_actual": bias,
                })
            except Exception as e:
                cv_rows.append({
                    "val_year": val_year,
                    "alpha": alpha,
                    "inner_train_rows": int(len(inner_train)),
                    "inner_val_rows": int(len(inner_val)),
                    "rmse_log": np.nan,
                    "mae_log": np.nan,
                    "bias_log_pred_minus_actual": np.nan,
                    "error": f"{type(e).__name__}: {e}",
                })

    cv = pd.DataFrame(cv_rows)

    if cv.empty or cv["rmse_log"].notna().sum() == 0:
        return fallback_alpha, cv

    alpha_summary = (
        cv.dropna(subset=["rmse_log"])
        .groupby("alpha", as_index=False)
        .agg(
            cv_folds=("val_year", "nunique"),
            mean_rmse_log=("rmse_log", "mean"),
            mean_mae_log=("mae_log", "mean"),
            mean_bias_log_pred_minus_actual=("bias_log_pred_minus_actual", "mean"),
        )
        .sort_values(["mean_rmse_log", "mean_mae_log", "alpha"], ascending=[True, True, True])
    )

    if alpha_summary.empty:
        return fallback_alpha, cv

    return float(alpha_summary.iloc[0]["alpha"]), cv


def summarize_metrics(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    rows = []

    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_cols, keys))

        valid_g = g[
            np.isfinite(g["predicted_log_variance_candidate"])
            & np.isfinite(g[TARGET_LOG_COL])
            & np.isfinite(g["forecast_variance_candidate"])
            & np.isfinite(g[TARGET_VAR_COL])
            & (g["forecast_variance_candidate"] > 0)
            & (g[TARGET_VAR_COL] > 0)
        ].copy()

        row["rows"] = int(len(valid_g))

        if len(valid_g) == 0:
            row.update({
                "rmse_log": np.nan,
                "mae_log": np.nan,
                "bias_log_pred_minus_actual": np.nan,
                "median_log_error": np.nan,
                "underforecast_rate": np.nan,
                "large_underforecast_1p5x_rate": np.nan,
                "large_underforecast_2p0x_rate": np.nan,
                "actual_top_decile_rows": 0,
                "forecast_top_decile_capture_rate": np.nan,
                "missed_top_decile_actual_rate": np.nan,
                "underforecast_rate_in_actual_top_decile": np.nan,
                "avg_actual_to_forecast_var_ratio": np.nan,
                "median_actual_to_forecast_var_ratio": np.nan,
                "target_realized_vol_pct_mean": np.nan,
                "candidate_forecast_vol_pct_mean": np.nan,
                "win_rate_context": np.nan,
                "avg_normalized_pnl_dollars": np.nan,
                "worst_normalized_pnl_dollars": np.nan,
            })
            rows.append(row)
            continue

        log_error = valid_g["predicted_log_variance_candidate"] - valid_g[TARGET_LOG_COL]
        ratio = valid_g[TARGET_VAR_COL] / valid_g["forecast_variance_candidate"]

        actual_top = valid_g[valid_g["actual_top_decile_by_tenor"]]

        row["rmse_log"] = float(np.sqrt(mean_squared_error(valid_g[TARGET_LOG_COL], valid_g["predicted_log_variance_candidate"])))
        row["mae_log"] = float(mean_absolute_error(valid_g[TARGET_LOG_COL], valid_g["predicted_log_variance_candidate"]))
        row["bias_log_pred_minus_actual"] = float(log_error.mean())
        row["median_log_error"] = float(log_error.median())
        row["underforecast_rate"] = float((valid_g["forecast_variance_candidate"] < valid_g[TARGET_VAR_COL]).mean())
        row["large_underforecast_1p5x_rate"] = float((ratio > 1.5).mean())
        row["large_underforecast_2p0x_rate"] = float((ratio > 2.0).mean())

        row["actual_top_decile_rows"] = int(len(actual_top))
        row["forecast_top_decile_capture_rate"] = (
            float(actual_top["forecast_top_decile_by_model_tenor"].mean()) if len(actual_top) else np.nan
        )
        row["missed_top_decile_actual_rate"] = (
            float((~actual_top["forecast_top_decile_by_model_tenor"]).mean()) if len(actual_top) else np.nan
        )
        row["underforecast_rate_in_actual_top_decile"] = (
            float((actual_top["forecast_variance_candidate"] < actual_top[TARGET_VAR_COL]).mean()) if len(actual_top) else np.nan
        )

        row["avg_actual_to_forecast_var_ratio"] = float(ratio.mean())
        row["median_actual_to_forecast_var_ratio"] = float(ratio.median())
        row["target_realized_vol_pct_mean"] = float(valid_g["target_realized_vol_pct"].mean())
        row["candidate_forecast_vol_pct_mean"] = float(valid_g["candidate_forecast_vol_pct"].mean())

        if "win" in valid_g.columns:
            row["win_rate_context"] = float(pd.to_numeric(valid_g["win"], errors="coerce").mean())
        else:
            row["win_rate_context"] = np.nan

        if "normalized_pnl_dollars" in valid_g.columns:
            pnl = pd.to_numeric(valid_g["normalized_pnl_dollars"], errors="coerce")
            row["avg_normalized_pnl_dollars"] = float(pnl.mean())
            row["worst_normalized_pnl_dollars"] = float(pnl.min())
        else:
            row["avg_normalized_pnl_dollars"] = np.nan
            row["worst_normalized_pnl_dollars"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows)


def summarize_bottom_pnl_overlap(df: pd.DataFrame) -> pd.DataFrame:
    if "normalized_pnl_dollars" not in df.columns:
        return pd.DataFrame()

    d = df.copy()
    d["normalized_pnl_dollars"] = pd.to_numeric(d["normalized_pnl_dollars"], errors="coerce")
    d = d[
        d["normalized_pnl_dollars"].notna()
        & np.isfinite(d["forecast_variance_candidate"])
        & np.isfinite(d[TARGET_VAR_COL])
        & (d["forecast_variance_candidate"] > 0)
        & (d[TARGET_VAR_COL] > 0)
    ].copy()

    if d.empty:
        return pd.DataFrame()

    d["pnl_pct_rank_by_model_tenor"] = d.groupby(["model_family", "tenor"])["normalized_pnl_dollars"].rank(pct=True)
    d["bottom_decile_pnl_by_model_tenor"] = d["pnl_pct_rank_by_model_tenor"] <= 0.10
    d["actual_to_forecast_var_ratio"] = d[TARGET_VAR_COL] / d["forecast_variance_candidate"]
    d["underforecast_flag"] = d["forecast_variance_candidate"] < d[TARGET_VAR_COL]
    d["large_underforecast_1p5x"] = d["actual_to_forecast_var_ratio"] > 1.5
    d["large_underforecast_2p0x"] = d["actual_to_forecast_var_ratio"] > 2.0

    rows = []
    for keys, g in d.groupby(["model_family", "tenor", "tenor_bucket", "forecast_repair_scope"], dropna=False):
        model_family, tenor, tenor_bucket, scope = keys
        bottom = g[g["bottom_decile_pnl_by_model_tenor"]].copy()

        rows.append({
            "model_family": model_family,
            "tenor": tenor,
            "tenor_bucket": tenor_bucket,
            "forecast_repair_scope": scope,
            "bottom_decile_pnl_rows": int(len(bottom)),
            "avg_bottom_decile_pnl": float(bottom["normalized_pnl_dollars"].mean()) if len(bottom) else np.nan,
            "worst_bottom_decile_pnl": float(bottom["normalized_pnl_dollars"].min()) if len(bottom) else np.nan,
            "underforecast_rate_in_bottom_decile_pnl": float(bottom["underforecast_flag"].mean()) if len(bottom) else np.nan,
            "large_underforecast_1p5x_rate": float(bottom["large_underforecast_1p5x"].mean()) if len(bottom) else np.nan,
            "large_underforecast_2p0x_rate": float(bottom["large_underforecast_2p0x"].mean()) if len(bottom) else np.nan,
            "avg_actual_to_forecast_var_ratio": float(bottom["actual_to_forecast_var_ratio"].mean()) if len(bottom) else np.nan,
            "median_actual_to_forecast_var_ratio": float(bottom["actual_to_forecast_var_ratio"].median()) if len(bottom) else np.nan,
        })

    return pd.DataFrame(rows)


# ======================================================================================
# 2. Load latest Cell 4 feature panel
# ======================================================================================

cell4_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "04_front_middle_candidate_feature_panel_*.parquet",
    required=True,
)

panel = pd.read_parquet(cell4_panel_path)
panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce").dt.normalize()
panel["last_forward_rv_date"] = pd.to_datetime(panel["last_forward_rv_date"], errors="coerce").dt.normalize()

panel = panel[panel["tenor"].isin(EXACT_TENOR_GRID)].copy()

required_cols = [
    "trade_date",
    "last_forward_rv_date",
    "tenor",
    "tenor_bucket",
    "forecast_repair_scope",
    TARGET_LOG_COL,
    TARGET_VAR_COL,
    FORECAST_VAR_BASELINE_COL,
]

missing_required = [c for c in required_cols if c not in panel.columns]
if missing_required:
    raise KeyError(f"Cell 5 missing required columns from Cell 4 panel: {missing_required}")

panel[TARGET_LOG_COL] = pd.to_numeric(panel[TARGET_LOG_COL], errors="coerce")
panel[TARGET_VAR_COL] = pd.to_numeric(panel[TARGET_VAR_COL], errors="coerce")
panel[FORECAST_VAR_BASELINE_COL] = pd.to_numeric(panel[FORECAST_VAR_BASELINE_COL], errors="coerce")

# Validate and filter target rows.
valid_target_panel = panel[
    panel["trade_date"].notna()
    & panel["last_forward_rv_date"].notna()
    & np.isfinite(panel[TARGET_LOG_COL])
    & np.isfinite(panel[TARGET_VAR_COL])
    & (panel[TARGET_VAR_COL] > 0)
].copy()

if valid_target_panel.empty:
    raise RuntimeError("No valid target rows available for Cell 5.")

# Ensure all model-family feature columns exist.
feature_validation_rows = []
usable_model_families = {}

for family, features in MODEL_FAMILIES.items():
    missing = [c for c in features if c not in panel.columns]
    forbidden_hits = forbidden_feature_hits(features, FORBIDDEN_MODEL_FEATURE_KEYWORDS)

    status = "PASS" if not missing and forbidden_hits.empty else "FAIL"

    feature_validation_rows.append({
        "model_family": family,
        "status": status,
        "feature_count": len(features),
        "missing_features": missing,
        "forbidden_hits": forbidden_hits.to_dict("records") if not forbidden_hits.empty else [],
        "features": features,
    })

    if status == "PASS":
        usable_model_families[family] = features

feature_validation = pd.DataFrame(feature_validation_rows)

if feature_validation["status"].eq("FAIL").any():
    display(feature_validation)
    raise RuntimeError("One or more model families failed feature validation.")

# ======================================================================================
# 3. Add existing har_downside_v1 baseline rows
# ======================================================================================

baseline = valid_target_panel.copy()
baseline["model_family"] = "existing_har_downside_v1"
baseline["model_source"] = "existing_locked_oos_forecast"
baseline["selected_alpha_candidate"] = pd.to_numeric(baseline.get("selected_alpha", np.nan), errors="coerce")
baseline["train_rows_used_candidate"] = pd.to_numeric(baseline.get("train_rows_used", np.nan), errors="coerce")
baseline["test_year_candidate"] = baseline["trade_date"].dt.year.astype("Int64")
baseline["fit_status_candidate"] = "existing_forecast"

if PRED_LOG_BASELINE_COL in baseline.columns:
    baseline["predicted_log_variance_candidate"] = pd.to_numeric(baseline[PRED_LOG_BASELINE_COL], errors="coerce")
else:
    baseline["predicted_log_variance_candidate"] = np.log(baseline[FORECAST_VAR_BASELINE_COL].clip(lower=EPS))

baseline["forecast_variance_candidate"] = baseline[FORECAST_VAR_BASELINE_COL]
baseline["candidate_forecast_vol_pct"] = np.sqrt(baseline["forecast_variance_candidate"].clip(lower=0)) * 100.0
baseline["target_realized_vol_pct"] = np.sqrt(baseline[TARGET_VAR_COL].clip(lower=0)) * 100.0

baseline_keep_cols = list(valid_target_panel.columns) + [
    "model_family",
    "model_source",
    "selected_alpha_candidate",
    "train_rows_used_candidate",
    "test_year_candidate",
    "fit_status_candidate",
    "predicted_log_variance_candidate",
    "forecast_variance_candidate",
    "candidate_forecast_vol_pct",
    "target_realized_vol_pct",
]

baseline = baseline[baseline_keep_cols].copy()

# ======================================================================================
# 4. Fit candidate models OOS for Front/Middle only
# ======================================================================================

forecast_rows = []
fit_log_rows = []
inner_cv_rows_all = []

for model_family, features in usable_model_families.items():
    for tenor in RESEARCH_TENORS:
        tenor_df = valid_target_panel[valid_target_panel["tenor"] == tenor].copy()
        tenor_df = tenor_df.sort_values("trade_date").reset_index(drop=True)

        test_years = sorted(tenor_df["trade_date"].dt.year.dropna().astype(int).unique().tolist())

        for test_year in test_years:
            test_start = pd.Timestamp(f"{test_year}-01-01")
            test_end = pd.Timestamp(f"{test_year + 1}-01-01")

            train_pool = tenor_df[
                (tenor_df["trade_date"] < test_start)
                & (tenor_df["last_forward_rv_date"] < test_start)
            ].copy()

            test_rows = tenor_df[
                (tenor_df["trade_date"] >= test_start)
                & (tenor_df["trade_date"] < test_end)
            ].copy()

            train_fit = train_pool.dropna(subset=features + [TARGET_LOG_COL]).copy()
            test_fit = test_rows.dropna(subset=features + [TARGET_LOG_COL]).copy()

            fit_status = "candidate_fit"

            if len(test_fit) == 0:
                fit_log_rows.append({
                    "model_family": model_family,
                    "tenor": tenor,
                    "test_year": test_year,
                    "fit_status": "skipped_no_test_rows",
                    "selected_alpha": np.nan,
                    "train_rows_used": int(len(train_fit)),
                    "test_rows_scored": 0,
                    "feature_count": len(features),
                })
                continue

            if len(train_fit) < MIN_OUTER_TRAIN_ROWS:
                fit_log_rows.append({
                    "model_family": model_family,
                    "tenor": tenor,
                    "test_year": test_year,
                    "fit_status": "skipped_insufficient_train_rows",
                    "selected_alpha": np.nan,
                    "train_rows_used": int(len(train_fit)),
                    "test_rows_scored": 0,
                    "feature_count": len(features),
                })
                continue

            selected_alpha, inner_cv = choose_alpha_inner_yearly_cv(
                train_pool=train_fit,
                features=features,
                target_col=TARGET_LOG_COL,
                alpha_grid=ALPHA_GRID,
                fallback_alpha=FALLBACK_ALPHA,
            )

            if not inner_cv.empty:
                inner_cv["outer_model_family"] = model_family
                inner_cv["outer_tenor"] = tenor
                inner_cv["outer_test_year"] = test_year
                inner_cv_rows_all.append(inner_cv)

            try:
                pred = fit_ridge_predict(
                    train_df=train_fit,
                    test_df=test_fit,
                    features=features,
                    target_col=TARGET_LOG_COL,
                    alpha=selected_alpha,
                )

                scored = test_fit.copy()
                scored["model_family"] = model_family
                scored["model_source"] = "candidate_oos_refit"
                scored["selected_alpha_candidate"] = selected_alpha
                scored["train_rows_used_candidate"] = int(len(train_fit))
                scored["test_year_candidate"] = int(test_year)
                scored["fit_status_candidate"] = fit_status
                scored["predicted_log_variance_candidate"] = pred
                scored["forecast_variance_candidate"] = np.exp(pred)
                scored["candidate_forecast_vol_pct"] = np.sqrt(scored["forecast_variance_candidate"].clip(lower=0)) * 100.0
                scored["target_realized_vol_pct"] = np.sqrt(scored[TARGET_VAR_COL].clip(lower=0)) * 100.0

                forecast_rows.append(scored)

                fit_log_rows.append({
                    "model_family": model_family,
                    "tenor": tenor,
                    "test_year": test_year,
                    "fit_status": fit_status,
                    "selected_alpha": selected_alpha,
                    "train_rows_used": int(len(train_fit)),
                    "test_rows_scored": int(len(scored)),
                    "feature_count": len(features),
                })

            except Exception as e:
                fit_log_rows.append({
                    "model_family": model_family,
                    "tenor": tenor,
                    "test_year": test_year,
                    "fit_status": "fit_error",
                    "selected_alpha": selected_alpha,
                    "train_rows_used": int(len(train_fit)),
                    "test_rows_scored": 0,
                    "feature_count": len(features),
                    "error": f"{type(e).__name__}: {e}",
                })

candidate_forecasts = pd.concat(forecast_rows, ignore_index=True) if forecast_rows else pd.DataFrame()
fit_log = pd.DataFrame(fit_log_rows)
inner_cv_all = pd.concat(inner_cv_rows_all, ignore_index=True) if inner_cv_rows_all else pd.DataFrame()

if candidate_forecasts.empty:
    display(fit_log)
    raise RuntimeError("No candidate forecasts were produced. See fit_log.")

# Use same column set where possible.
missing_from_candidate = [c for c in baseline.columns if c not in candidate_forecasts.columns]
for c in missing_from_candidate:
    candidate_forecasts[c] = np.nan

candidate_forecasts = candidate_forecasts[baseline.columns].copy()

all_forecasts = pd.concat([baseline, candidate_forecasts], ignore_index=True)

# ======================================================================================
# 5. Forecast diagnostic flags
# ======================================================================================

all_forecasts["forecast_variance_candidate"] = pd.to_numeric(all_forecasts["forecast_variance_candidate"], errors="coerce")
all_forecasts["predicted_log_variance_candidate"] = pd.to_numeric(all_forecasts["predicted_log_variance_candidate"], errors="coerce")
all_forecasts[TARGET_VAR_COL] = pd.to_numeric(all_forecasts[TARGET_VAR_COL], errors="coerce")
all_forecasts[TARGET_LOG_COL] = pd.to_numeric(all_forecasts[TARGET_LOG_COL], errors="coerce")

all_forecasts = all_forecasts[
    np.isfinite(all_forecasts["forecast_variance_candidate"])
    & np.isfinite(all_forecasts["predicted_log_variance_candidate"])
    & np.isfinite(all_forecasts[TARGET_VAR_COL])
    & np.isfinite(all_forecasts[TARGET_LOG_COL])
    & (all_forecasts["forecast_variance_candidate"] > 0)
    & (all_forecasts[TARGET_VAR_COL] > 0)
].copy()

all_forecasts["target_realized_vol_pct"] = np.sqrt(all_forecasts[TARGET_VAR_COL].clip(lower=0)) * 100.0
all_forecasts["candidate_forecast_vol_pct"] = np.sqrt(all_forecasts["forecast_variance_candidate"].clip(lower=0)) * 100.0
all_forecasts["log_error_pred_minus_actual"] = (
    all_forecasts["predicted_log_variance_candidate"] - all_forecasts[TARGET_LOG_COL]
)
all_forecasts["actual_to_forecast_var_ratio"] = (
    all_forecasts[TARGET_VAR_COL] / all_forecasts["forecast_variance_candidate"]
)
all_forecasts["underforecast_flag"] = (
    all_forecasts["forecast_variance_candidate"] < all_forecasts[TARGET_VAR_COL]
)
all_forecasts["large_underforecast_1p5x"] = all_forecasts["actual_to_forecast_var_ratio"] > 1.5
all_forecasts["large_underforecast_2p0x"] = all_forecasts["actual_to_forecast_var_ratio"] > 2.0

# Actual top decile is model-independent, computed by tenor using unique date × tenor target rows.
actual_rank_source = (
    valid_target_panel[["trade_date", "tenor", TARGET_VAR_COL]]
    .drop_duplicates(["trade_date", "tenor"])
    .copy()
)
actual_rank_source["actual_var_pct_rank_by_tenor"] = actual_rank_source.groupby("tenor")[TARGET_VAR_COL].rank(pct=True)
actual_rank_source["actual_top_decile_by_tenor"] = actual_rank_source["actual_var_pct_rank_by_tenor"] >= 0.90

all_forecasts = all_forecasts.drop(
    columns=[c for c in ["actual_var_pct_rank_by_tenor", "actual_top_decile_by_tenor"] if c in all_forecasts.columns],
    errors="ignore",
).merge(
    actual_rank_source[["trade_date", "tenor", "actual_var_pct_rank_by_tenor", "actual_top_decile_by_tenor"]],
    on=["trade_date", "tenor"],
    how="left",
    validate="many_to_one",
)

all_forecasts["forecast_var_pct_rank_by_model_tenor"] = (
    all_forecasts.groupby(["model_family", "tenor"])["forecast_variance_candidate"].rank(pct=True)
)
all_forecasts["forecast_top_decile_by_model_tenor"] = (
    all_forecasts["forecast_var_pct_rank_by_model_tenor"] >= 0.90
)

# --------------------------------------------------------------------------------------
# Runtime repair:
# Some upstream panels may already contain diagnostic columns such as
# target_realized_vol_pct or candidate_forecast_vol_pct. After concatenating baseline and
# candidate rows, duplicate column labels can cause df["col"] to return a DataFrame
# instead of a Series. De-duplicate before summary metrics.
# --------------------------------------------------------------------------------------

duplicate_cols = all_forecasts.columns[all_forecasts.columns.duplicated()].tolist()

if duplicate_cols:
    print("Dropping duplicate column labels before metric summaries:")
    print(sorted(set(duplicate_cols)))

all_forecasts = all_forecasts.loc[:, ~all_forecasts.columns.duplicated(keep="last")].copy()

# Recompute these after de-duplicating so the metric function sees clean Series.
all_forecasts["target_realized_vol_pct"] = (
    np.sqrt(pd.to_numeric(all_forecasts[TARGET_VAR_COL], errors="coerce").clip(lower=0)) * 100.0
)

all_forecasts["candidate_forecast_vol_pct"] = (
    np.sqrt(pd.to_numeric(all_forecasts["forecast_variance_candidate"], errors="coerce").clip(lower=0)) * 100.0
)

# ======================================================================================
# 6. Metric tables
# ======================================================================================

metrics_by_bucket = summarize_metrics(
    all_forecasts,
    ["model_family", "tenor_bucket", "forecast_repair_scope"],
)

metrics_by_tenor = summarize_metrics(
    all_forecasts,
    ["model_family", "tenor", "tenor_bucket", "forecast_repair_scope"],
)

metrics_by_bucket_year = summarize_metrics(
    all_forecasts,
    ["model_family", "tenor_bucket", "forecast_repair_scope", "test_year_candidate"],
)

metrics_by_tenor_year = summarize_metrics(
    all_forecasts,
    ["model_family", "tenor", "tenor_bucket", "forecast_repair_scope", "test_year_candidate"],
)

bad_outcome_overlap = summarize_bottom_pnl_overlap(all_forecasts)

# Front/Middle-only leaderboard for review, not final selection.
front_middle_bucket_metrics = metrics_by_bucket[
    metrics_by_bucket["forecast_repair_scope"].isin(["RESEARCH_FRONT", "RESEARCH_MIDDLE"])
].copy()

existing_base = front_middle_bucket_metrics[
    front_middle_bucket_metrics["model_family"] == "existing_har_downside_v1"
][
    [
        "tenor_bucket",
        "forecast_repair_scope",
        "rmse_log",
        "forecast_top_decile_capture_rate",
        "missed_top_decile_actual_rate",
        "underforecast_rate_in_actual_top_decile",
        "large_underforecast_2p0x_rate",
        "candidate_forecast_vol_pct_mean",
    ]
].rename(
    columns={
        "rmse_log": "baseline_rmse_log",
        "forecast_top_decile_capture_rate": "baseline_forecast_top_decile_capture_rate",
        "missed_top_decile_actual_rate": "baseline_missed_top_decile_actual_rate",
        "underforecast_rate_in_actual_top_decile": "baseline_underforecast_rate_in_actual_top_decile",
        "large_underforecast_2p0x_rate": "baseline_large_underforecast_2p0x_rate",
        "candidate_forecast_vol_pct_mean": "baseline_forecast_vol_pct_mean",
    }
)

candidate_leaderboard = front_middle_bucket_metrics.merge(
    existing_base,
    on=["tenor_bucket", "forecast_repair_scope"],
    how="left",
)

candidate_leaderboard["capture_improvement_vs_existing"] = (
    candidate_leaderboard["forecast_top_decile_capture_rate"]
    - candidate_leaderboard["baseline_forecast_top_decile_capture_rate"]
)
candidate_leaderboard["missed_top_decile_reduction_vs_existing"] = (
    candidate_leaderboard["baseline_missed_top_decile_actual_rate"]
    - candidate_leaderboard["missed_top_decile_actual_rate"]
)
candidate_leaderboard["rmse_change_vs_existing"] = (
    candidate_leaderboard["rmse_log"] - candidate_leaderboard["baseline_rmse_log"]
)
candidate_leaderboard["forecast_vol_mean_change_vs_existing"] = (
    candidate_leaderboard["candidate_forecast_vol_pct_mean"]
    - candidate_leaderboard["baseline_forecast_vol_pct_mean"]
)

candidate_leaderboard["review_score_not_selection"] = (
    candidate_leaderboard["capture_improvement_vs_existing"].fillna(0.0) * 3.0
    + candidate_leaderboard["missed_top_decile_reduction_vs_existing"].fillna(0.0) * 2.0
    - candidate_leaderboard["rmse_change_vs_existing"].clip(lower=0).fillna(0.0) * 0.5
    - candidate_leaderboard["forecast_vol_mean_change_vs_existing"].abs().fillna(0.0) * 0.01
)

candidate_leaderboard = candidate_leaderboard.sort_values(
    ["forecast_repair_scope", "review_score_not_selection"],
    ascending=[True, False],
)

alpha_summary = (
    fit_log[fit_log["fit_status"].eq("candidate_fit")]
    .groupby(["model_family", "tenor"], as_index=False)
    .agg(
        fit_years=("test_year", "nunique"),
        total_test_rows_scored=("test_rows_scored", "sum"),
        mean_train_rows_used=("train_rows_used", "mean"),
        median_selected_alpha=("selected_alpha", "median"),
        min_selected_alpha=("selected_alpha", "min"),
        max_selected_alpha=("selected_alpha", "max"),
    )
)

# ======================================================================================
# 7. Validation
# ======================================================================================

validation_rows = []

def add_validation(check: str, status: str, detail: str):
    validation_rows.append({"check": check, "status": status, "detail": detail})

candidate_back_refit_rows = candidate_forecasts[
    (candidate_forecasts["model_source"] == "candidate_oos_refit")
    & (candidate_forecasts["tenor"].isin(BACK_TENORS))
]

add_validation("cell4_panel_loaded", "PASS", f"path={cell4_panel_path}; rows={len(panel):,}")
add_validation("feature_validation", "PASS", "all model families passed feature availability and forbidden-keyword audit")
add_validation(
    "candidate_forecasts_produced",
    "PASS" if len(candidate_forecasts) > 0 else "FAIL",
    f"candidate rows={len(candidate_forecasts):,}",
)
add_validation(
    "no_candidate_back_refit_rows",
    "PASS" if candidate_back_refit_rows.empty else "FAIL",
    f"candidate_back_refit_rows={len(candidate_back_refit_rows):,}",
)
add_validation(
    "existing_back_rows_carried_only",
    "PASS",
    "existing_har_downside_v1 rows include Back only as frozen diagnostics; no Back candidate models fitted",
)
add_validation(
    "fit_errors",
    "PASS" if not fit_log["fit_status"].eq("fit_error").any() else "WARN",
    f"fit_error_rows={int(fit_log['fit_status'].eq('fit_error').sum())}",
)
add_validation(
    "insufficient_train_skips_expected_early_years",
    "PASS",
    f"skipped_insufficient_train_rows={int(fit_log['fit_status'].eq('skipped_insufficient_train_rows').sum())}",
)
add_validation(
    "no_signal_or_threshold_sweep",
    "PASS",
    "Cell 5 only fits/evaluates forecast denominators",
)

validation = pd.DataFrame(validation_rows)
failures = validation[validation["status"] == "FAIL"]

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 5 validation failed. See validation table above.")

# ======================================================================================
# 8. Save outputs
# ======================================================================================

date_min = all_forecasts["trade_date"].min()
date_max = all_forecasts["trade_date"].max()

forecast_panel_path = OUT_PROCESSED_DIR / (
    f"05_front_middle_oos_forecast_candidate_panel_{date_min:%Y%m%d}_{date_max:%Y%m%d}_{TIMESTAMP}.parquet"
)

metrics_by_bucket_path = OUT_AUDIT_DIR / f"05_candidate_model_metrics_by_bucket_{TIMESTAMP}.csv"
metrics_by_tenor_path = OUT_AUDIT_DIR / f"05_candidate_model_metrics_by_tenor_{TIMESTAMP}.csv"
metrics_by_bucket_year_path = OUT_AUDIT_DIR / f"05_candidate_model_metrics_by_bucket_year_{TIMESTAMP}.csv"
metrics_by_tenor_year_path = OUT_AUDIT_DIR / f"05_candidate_model_metrics_by_tenor_year_{TIMESTAMP}.csv"
bad_outcome_overlap_path = OUT_AUDIT_DIR / f"05_candidate_model_bad_outcome_overlap_{TIMESTAMP}.csv"
candidate_leaderboard_path = OUT_AUDIT_DIR / f"05_candidate_model_front_middle_review_leaderboard_{TIMESTAMP}.csv"
fit_log_path = OUT_AUDIT_DIR / f"05_candidate_model_fit_log_{TIMESTAMP}.csv"
inner_cv_path = OUT_AUDIT_DIR / f"05_candidate_model_inner_cv_{TIMESTAMP}.csv"
alpha_summary_path = OUT_AUDIT_DIR / f"05_candidate_model_alpha_summary_{TIMESTAMP}.csv"
feature_validation_path = OUT_AUDIT_DIR / f"05_candidate_model_feature_validation_{TIMESTAMP}.csv"
validation_path = OUT_AUDIT_DIR / f"05_candidate_model_validation_{TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"05_candidate_model_family_manifest_{TIMESTAMP}.json"

all_forecasts.to_parquet(forecast_panel_path, index=False)
metrics_by_bucket.to_csv(metrics_by_bucket_path, index=False)
metrics_by_tenor.to_csv(metrics_by_tenor_path, index=False)
metrics_by_bucket_year.to_csv(metrics_by_bucket_year_path, index=False)
metrics_by_tenor_year.to_csv(metrics_by_tenor_year_path, index=False)
bad_outcome_overlap.to_csv(bad_outcome_overlap_path, index=False)
candidate_leaderboard.to_csv(candidate_leaderboard_path, index=False)
fit_log.to_csv(fit_log_path, index=False)
alpha_summary.to_csv(alpha_summary_path, index=False)
feature_validation.to_csv(feature_validation_path, index=False)
validation.to_csv(validation_path, index=False)

if not inner_cv_all.empty:
    inner_cv_all.to_csv(inner_cv_path, index=False)
else:
    pd.DataFrame().to_csv(inner_cv_path, index=False)

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "cell": "Cell 5 — Front/Middle OOS forecast candidate sweep",
            "timestamp": TIMESTAMP,
            "source_cell4_panel": str(cell4_panel_path),
            "research_tenors_refit": RESEARCH_TENORS,
            "back_tenors_frozen": BACK_TENORS,
            "alpha_grid": ALPHA_GRID,
            "fallback_alpha": FALLBACK_ALPHA,
            "min_outer_train_rows": MIN_OUTER_TRAIN_ROWS,
            "min_inner_train_rows": MIN_INNER_TRAIN_ROWS,
            "min_inner_val_rows": MIN_INNER_VAL_ROWS,
            "model_families": MODEL_FAMILIES,
            "notes": [
                "Candidate models are fitted only for Front/Middle tenors.",
                "Back tenors are carried only through existing_har_downside_v1 diagnostics.",
                "No signal threshold sweep or final denominator selection is performed in this cell.",
            ],
        },
        f,
        indent=2,
    )

# ======================================================================================
# 9. Display concise review
# ======================================================================================

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 260)

print("=" * 100)
print("Cell 5 — Front/Middle OOS forecast candidate sweep")
print("=" * 100)
print(f"Source Cell 4 panel:       {cell4_panel_path}")
print(f"Forecast rows saved:       {len(all_forecasts):,}")
print(f"Candidate refit rows:      {len(candidate_forecasts):,}")
print(f"Existing baseline rows:    {len(baseline):,}")
print(f"Date range:                {date_min.date()} to {date_max.date()}")
print(f"Research tenors refit:     {RESEARCH_TENORS}")
print(f"Back tenors frozen:        {BACK_TENORS}")
print(f"Model families fitted:     {list(usable_model_families.keys())}")

print()
print("Validation:")
display(validation)

print()
print("Fit status counts:")
display(fit_log["fit_status"].value_counts(dropna=False).to_frame("count"))

print()
print("Alpha summary:")
display(alpha_summary)

print()
print("Front/Middle candidate review leaderboard — NOT final selection:")
display(candidate_leaderboard[
    [
        "model_family",
        "tenor_bucket",
        "forecast_repair_scope",
        "rows",
        "rmse_log",
        "mae_log",
        "bias_log_pred_minus_actual",
        "underforecast_rate",
        "large_underforecast_1p5x_rate",
        "large_underforecast_2p0x_rate",
        "forecast_top_decile_capture_rate",
        "missed_top_decile_actual_rate",
        "underforecast_rate_in_actual_top_decile",
        "candidate_forecast_vol_pct_mean",
        "baseline_forecast_top_decile_capture_rate",
        "capture_improvement_vs_existing",
        "missed_top_decile_reduction_vs_existing",
        "rmse_change_vs_existing",
        "forecast_vol_mean_change_vs_existing",
        "review_score_not_selection",
    ]
].head(40))

print()
print("Metrics by tenor:")
display(metrics_by_tenor[
    metrics_by_tenor["forecast_repair_scope"].isin(["RESEARCH_FRONT", "RESEARCH_MIDDLE"])
].sort_values(["tenor", "forecast_top_decile_capture_rate"], ascending=[True, False]).head(80))

if not bad_outcome_overlap.empty:
    print()
    print("Bad-outcome overlap diagnostics:")
    display(bad_outcome_overlap[
        bad_outcome_overlap["forecast_repair_scope"].isin(["RESEARCH_FRONT", "RESEARCH_MIDDLE"])
    ].sort_values(["tenor", "underforecast_rate_in_bottom_decile_pnl"], ascending=[True, True]).head(80))

print()
print("Saved outputs:")
for p in [
    forecast_panel_path,
    metrics_by_bucket_path,
    metrics_by_tenor_path,
    metrics_by_bucket_year_path,
    metrics_by_tenor_year_path,
    bad_outcome_overlap_path,
    candidate_leaderboard_path,
    fit_log_path,
    inner_cv_path,
    alpha_summary_path,
    feature_validation_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 5 PASSED")

In [ ]:
# ======================================================================================
# Cell 6 — Common-row Front/Middle forecast candidate review
# ======================================================================================
#
# Objective:
#   Re-evaluate the leading Front/Middle forecast-denominator candidates from Cell 5
#   on identical date × tenor rows, so candidate models and the existing baseline are
#   compared apples-to-apples.
#
# Scope:
#   - Load latest Cell 5 candidate forecast panel.
#   - Restrict to Front/Middle tenors only.
#   - Restrict to reviewed model families only.
#   - Keep only common date × tenor rows where every reviewed model has a forecast.
#   - Recompute forecast diagnostics on the common-row universe.
#   - Review year-by-year stability, crash vs non-crash behavior, normal-period
#     overforecasting, top-decile capture, and bad-P&L overlap.
#   - Produce a narrowed candidate recommendation table.
#
# Explicitly not in scope:
#   - No new model fitting.
#   - No Back refit.
#   - No Back-unfreeze diagnostic.
#   - No VRP/z-score rebuild.
#   - No parameter threshold sweep.
#   - No final model lock.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ======================================================================================
# 0. Configuration
# ======================================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
NEW_BRANCH = "vrp_front_middle_corsi_forecast_repair_v1"

OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

FRONT_TENORS = [9, 12, 15]
MIDDLE_TENORS = [18, 21, 24]
BACK_TENORS = [27, 30, 33]
RESEARCH_TENORS = FRONT_TENORS + MIDDLE_TENORS

TARGET_LOG_COL = "target_log_variance"
TARGET_VAR_COL = "target_realized_variance"

EXISTING_MODEL = "existing_har_downside_v1"

REVIEW_MODELS = [
    "existing_har_downside_v1",
    "fast_rv_plus_shock",
    "fast_downside_plus_shock",
    "full_fast_state",
    "shock_core",
]

DROPPED_FROM_ACTIVE_REVIEW = [
    "baseline_locked_features_refit",
    "fast_rv_core",
    "fast_downside_core",
]

COVID_CRASH_START = pd.Timestamp("2020-02-18")
COVID_CRASH_END = pd.Timestamp("2020-04-30")

EPS = 1e-12

# Recommendation heuristics.
# These are not final model-lock rules. They are a structured narrowing guide.
MIN_AVG_CAPTURE_IMPROVEMENT_ADVANCE = 0.15
MAX_WORST_BUCKET_RMSE_DETERIORATION_ADVANCE = 0.05
MIN_AVG_CAPTURE_IMPROVEMENT_BACKUP = 0.05
MAX_WORST_BUCKET_RMSE_DETERIORATION_BACKUP = 0.10

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def summarize_metrics(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    rows = []

    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_cols, keys))

        valid_g = g[
            np.isfinite(g["predicted_log_variance_candidate"])
            & np.isfinite(g[TARGET_LOG_COL])
            & np.isfinite(g["forecast_variance_candidate"])
            & np.isfinite(g[TARGET_VAR_COL])
            & (g["forecast_variance_candidate"] > 0)
            & (g[TARGET_VAR_COL] > 0)
        ].copy()

        row["rows"] = int(len(valid_g))

        if len(valid_g) == 0:
            metric_cols = [
                "rmse_log",
                "mae_log",
                "bias_log_pred_minus_actual",
                "median_log_error",
                "underforecast_rate",
                "large_underforecast_1p5x_rate",
                "large_underforecast_2p0x_rate",
                "actual_top_decile_rows",
                "forecast_top_decile_capture_rate",
                "missed_top_decile_actual_rate",
                "underforecast_rate_in_actual_top_decile",
                "avg_actual_to_forecast_var_ratio",
                "median_actual_to_forecast_var_ratio",
                "target_realized_vol_pct_mean",
                "candidate_forecast_vol_pct_mean",
                "forecast_vol_minus_target_vol_mean",
                "normal_period_rows",
                "normal_period_overforecast_rate",
                "normal_period_mean_log_error",
                "normal_period_forecast_vol_mean",
                "normal_period_target_vol_mean",
                "win_rate_context",
                "avg_normalized_pnl_dollars",
                "worst_normalized_pnl_dollars",
            ]
            for c in metric_cols:
                row[c] = np.nan
            rows.append(row)
            continue

        log_error = valid_g["predicted_log_variance_candidate"] - valid_g[TARGET_LOG_COL]
        ratio = valid_g[TARGET_VAR_COL] / valid_g["forecast_variance_candidate"]

        actual_top = valid_g[valid_g["actual_top_decile_common_by_tenor"]].copy()
        normal_period = valid_g[valid_g["normal_period_common_by_tenor"]].copy()

        row["rmse_log"] = float(np.sqrt(mean_squared_error(valid_g[TARGET_LOG_COL], valid_g["predicted_log_variance_candidate"])))
        row["mae_log"] = float(mean_absolute_error(valid_g[TARGET_LOG_COL], valid_g["predicted_log_variance_candidate"]))
        row["bias_log_pred_minus_actual"] = float(log_error.mean())
        row["median_log_error"] = float(log_error.median())
        row["underforecast_rate"] = float((valid_g["forecast_variance_candidate"] < valid_g[TARGET_VAR_COL]).mean())
        row["large_underforecast_1p5x_rate"] = float((ratio > 1.5).mean())
        row["large_underforecast_2p0x_rate"] = float((ratio > 2.0).mean())

        row["actual_top_decile_rows"] = int(len(actual_top))
        row["forecast_top_decile_capture_rate"] = (
            float(actual_top["forecast_top_decile_common_by_model_tenor"].mean()) if len(actual_top) else np.nan
        )
        row["missed_top_decile_actual_rate"] = (
            float((~actual_top["forecast_top_decile_common_by_model_tenor"]).mean()) if len(actual_top) else np.nan
        )
        row["underforecast_rate_in_actual_top_decile"] = (
            float((actual_top["forecast_variance_candidate"] < actual_top[TARGET_VAR_COL]).mean()) if len(actual_top) else np.nan
        )

        row["avg_actual_to_forecast_var_ratio"] = float(ratio.mean())
        row["median_actual_to_forecast_var_ratio"] = float(ratio.median())
        row["target_realized_vol_pct_mean"] = float(valid_g["target_realized_vol_pct"].mean())
        row["candidate_forecast_vol_pct_mean"] = float(valid_g["candidate_forecast_vol_pct"].mean())
        row["forecast_vol_minus_target_vol_mean"] = (
            row["candidate_forecast_vol_pct_mean"] - row["target_realized_vol_pct_mean"]
        )

        row["normal_period_rows"] = int(len(normal_period))
        row["normal_period_overforecast_rate"] = (
            float((normal_period["forecast_variance_candidate"] > normal_period[TARGET_VAR_COL]).mean())
            if len(normal_period) else np.nan
        )
        row["normal_period_mean_log_error"] = (
            float((normal_period["predicted_log_variance_candidate"] - normal_period[TARGET_LOG_COL]).mean())
            if len(normal_period) else np.nan
        )
        row["normal_period_forecast_vol_mean"] = (
            float(normal_period["candidate_forecast_vol_pct"].mean()) if len(normal_period) else np.nan
        )
        row["normal_period_target_vol_mean"] = (
            float(normal_period["target_realized_vol_pct"].mean()) if len(normal_period) else np.nan
        )

        if "win" in valid_g.columns:
            row["win_rate_context"] = float(pd.to_numeric(valid_g["win"], errors="coerce").mean())
        else:
            row["win_rate_context"] = np.nan

        if "normalized_pnl_dollars" in valid_g.columns:
            pnl = pd.to_numeric(valid_g["normalized_pnl_dollars"], errors="coerce")
            row["avg_normalized_pnl_dollars"] = float(pnl.mean())
            row["worst_normalized_pnl_dollars"] = float(pnl.min())
        else:
            row["avg_normalized_pnl_dollars"] = np.nan
            row["worst_normalized_pnl_dollars"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows)


def summarize_bad_outcome_overlap(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    if "normalized_pnl_dollars" not in df.columns:
        return pd.DataFrame()

    d = df.copy()
    d["normalized_pnl_dollars"] = pd.to_numeric(d["normalized_pnl_dollars"], errors="coerce")
    d = d[
        d["normalized_pnl_dollars"].notna()
        & np.isfinite(d["forecast_variance_candidate"])
        & np.isfinite(d[TARGET_VAR_COL])
        & (d["forecast_variance_candidate"] > 0)
        & (d[TARGET_VAR_COL] > 0)
    ].copy()

    if d.empty:
        return pd.DataFrame()

    d["pnl_pct_rank_common_by_model_tenor"] = (
        d.groupby(["model_family", "tenor"])["normalized_pnl_dollars"].rank(pct=True)
    )
    d["bottom_decile_pnl_common_by_model_tenor"] = d["pnl_pct_rank_common_by_model_tenor"] <= 0.10
    d["actual_to_forecast_var_ratio"] = d[TARGET_VAR_COL] / d["forecast_variance_candidate"]
    d["underforecast_flag"] = d["forecast_variance_candidate"] < d[TARGET_VAR_COL]
    d["large_underforecast_1p5x"] = d["actual_to_forecast_var_ratio"] > 1.5
    d["large_underforecast_2p0x"] = d["actual_to_forecast_var_ratio"] > 2.0

    rows = []
    for keys, g in d.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_cols, keys))
        bottom = g[g["bottom_decile_pnl_common_by_model_tenor"]].copy()

        row.update({
            "bottom_decile_pnl_rows": int(len(bottom)),
            "avg_bottom_decile_pnl": float(bottom["normalized_pnl_dollars"].mean()) if len(bottom) else np.nan,
            "worst_bottom_decile_pnl": float(bottom["normalized_pnl_dollars"].min()) if len(bottom) else np.nan,
            "underforecast_rate_in_bottom_decile_pnl": float(bottom["underforecast_flag"].mean()) if len(bottom) else np.nan,
            "large_underforecast_1p5x_rate": float(bottom["large_underforecast_1p5x"].mean()) if len(bottom) else np.nan,
            "large_underforecast_2p0x_rate": float(bottom["large_underforecast_2p0x"].mean()) if len(bottom) else np.nan,
            "avg_actual_to_forecast_var_ratio": float(bottom["actual_to_forecast_var_ratio"].mean()) if len(bottom) else np.nan,
            "median_actual_to_forecast_var_ratio": float(bottom["actual_to_forecast_var_ratio"].median()) if len(bottom) else np.nan,
        })
        rows.append(row)

    return pd.DataFrame(rows)


def add_vs_existing(metrics: pd.DataFrame, join_cols: list[str]) -> pd.DataFrame:
    baseline = metrics[metrics["model_family"] == EXISTING_MODEL].copy()

    baseline_keep = join_cols + [
        "rmse_log",
        "mae_log",
        "bias_log_pred_minus_actual",
        "underforecast_rate",
        "large_underforecast_1p5x_rate",
        "large_underforecast_2p0x_rate",
        "forecast_top_decile_capture_rate",
        "missed_top_decile_actual_rate",
        "underforecast_rate_in_actual_top_decile",
        "candidate_forecast_vol_pct_mean",
        "normal_period_overforecast_rate",
        "normal_period_mean_log_error",
        "normal_period_forecast_vol_mean",
    ]

    rename_map = {
        c: f"existing_{c}"
        for c in baseline_keep
        if c not in join_cols
    }

    baseline = baseline[baseline_keep].rename(columns=rename_map)

    out = metrics.merge(baseline, on=join_cols, how="left")

    out["capture_improvement_vs_existing"] = (
        out["forecast_top_decile_capture_rate"] - out["existing_forecast_top_decile_capture_rate"]
    )
    out["missed_top_decile_reduction_vs_existing"] = (
        out["existing_missed_top_decile_actual_rate"] - out["missed_top_decile_actual_rate"]
    )
    out["rmse_change_vs_existing"] = out["rmse_log"] - out["existing_rmse_log"]
    out["mae_change_vs_existing"] = out["mae_log"] - out["existing_mae_log"]
    out["large_underforecast_2p0x_change_vs_existing"] = (
        out["large_underforecast_2p0x_rate"] - out["existing_large_underforecast_2p0x_rate"]
    )
    out["forecast_vol_mean_change_vs_existing"] = (
        out["candidate_forecast_vol_pct_mean"] - out["existing_candidate_forecast_vol_pct_mean"]
    )
    out["normal_period_overforecast_change_vs_existing"] = (
        out["normal_period_overforecast_rate"] - out["existing_normal_period_overforecast_rate"]
    )
    out["normal_period_log_error_change_vs_existing"] = (
        out["normal_period_mean_log_error"] - out["existing_normal_period_mean_log_error"]
    )

    return out


# ======================================================================================
# 2. Load latest Cell 5 forecast panel
# ======================================================================================

cell5_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "05_front_middle_oos_forecast_candidate_panel_*.parquet",
    required=True,
)

panel = pd.read_parquet(cell5_panel_path)

# Remove duplicate column labels defensively.
duplicate_cols = panel.columns[panel.columns.duplicated()].tolist()
if duplicate_cols:
    print("Dropping duplicate column labels in Cell 6 source panel:")
    print(sorted(set(duplicate_cols)))
panel = panel.loc[:, ~panel.columns.duplicated(keep="last")].copy()

panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce").dt.normalize()
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype("Int64")

required_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "forecast_repair_scope",
    "model_family",
    TARGET_LOG_COL,
    TARGET_VAR_COL,
    "predicted_log_variance_candidate",
    "forecast_variance_candidate",
]

missing_required = [c for c in required_cols if c not in panel.columns]
if missing_required:
    raise KeyError(f"Cell 6 missing required columns from Cell 5 panel: {missing_required}")

# Keep reviewed model families and Front/Middle only.
review = panel[
    panel["model_family"].isin(REVIEW_MODELS)
    & panel["tenor"].isin(RESEARCH_TENORS)
].copy()

for c in [
    TARGET_LOG_COL,
    TARGET_VAR_COL,
    "predicted_log_variance_candidate",
    "forecast_variance_candidate",
]:
    review[c] = pd.to_numeric(review[c], errors="coerce")

review = review[
    review["trade_date"].notna()
    & review["tenor"].notna()
    & np.isfinite(review[TARGET_LOG_COL])
    & np.isfinite(review[TARGET_VAR_COL])
    & np.isfinite(review["predicted_log_variance_candidate"])
    & np.isfinite(review["forecast_variance_candidate"])
    & (review[TARGET_VAR_COL] > 0)
    & (review["forecast_variance_candidate"] > 0)
].copy()

if review.empty:
    raise RuntimeError("No reviewed Front/Middle forecast rows found in Cell 5 panel.")

# ======================================================================================
# 3. Build common-row universe
# ======================================================================================

row_model_counts = (
    review.groupby(["trade_date", "tenor"], as_index=False)
    .agg(model_count=("model_family", "nunique"))
)

common_keys = row_model_counts[row_model_counts["model_count"] == len(REVIEW_MODELS)][["trade_date", "tenor"]].copy()

if common_keys.empty:
    raise RuntimeError("No common date × tenor rows found across all review models.")

common = review.merge(common_keys, on=["trade_date", "tenor"], how="inner", validate="many_to_one").copy()

# Validate one row per model/date/tenor.
dups = common.duplicated(["model_family", "trade_date", "tenor"]).sum()
if dups:
    raise RuntimeError(f"Common panel has duplicate model_family × trade_date × tenor rows: {dups}")

# Recompute vol diagnostics after common-row restriction.
common["target_realized_vol_pct"] = np.sqrt(common[TARGET_VAR_COL].clip(lower=0)) * 100.0
common["candidate_forecast_vol_pct"] = np.sqrt(common["forecast_variance_candidate"].clip(lower=0)) * 100.0
common["log_error_pred_minus_actual"] = common["predicted_log_variance_candidate"] - common[TARGET_LOG_COL]
common["actual_to_forecast_var_ratio"] = common[TARGET_VAR_COL] / common["forecast_variance_candidate"]

# Recompute common-row actual top-decile and normal-period flags by tenor.
actual_common_rank_source = (
    common[["trade_date", "tenor", TARGET_VAR_COL]]
    .drop_duplicates(["trade_date", "tenor"])
    .copy()
)

actual_common_rank_source["actual_var_pct_rank_common_by_tenor"] = (
    actual_common_rank_source.groupby("tenor")[TARGET_VAR_COL].rank(pct=True)
)
actual_common_rank_source["actual_top_decile_common_by_tenor"] = (
    actual_common_rank_source["actual_var_pct_rank_common_by_tenor"] >= 0.90
)
actual_common_rank_source["normal_period_common_by_tenor"] = (
    actual_common_rank_source["actual_var_pct_rank_common_by_tenor"] <= 0.75
)

common = common.drop(
    columns=[
        c for c in [
            "actual_var_pct_rank_common_by_tenor",
            "actual_top_decile_common_by_tenor",
            "normal_period_common_by_tenor",
        ]
        if c in common.columns
    ],
    errors="ignore",
).merge(
    actual_common_rank_source[
        [
            "trade_date",
            "tenor",
            "actual_var_pct_rank_common_by_tenor",
            "actual_top_decile_common_by_tenor",
            "normal_period_common_by_tenor",
        ]
    ],
    on=["trade_date", "tenor"],
    how="left",
    validate="many_to_one",
)

# Recompute forecast top-decile by model and tenor on common rows.
common["forecast_var_pct_rank_common_by_model_tenor"] = (
    common.groupby(["model_family", "tenor"])["forecast_variance_candidate"].rank(pct=True)
)
common["forecast_top_decile_common_by_model_tenor"] = (
    common["forecast_var_pct_rank_common_by_model_tenor"] >= 0.90
)

# Add period labels.
common["year"] = common["trade_date"].dt.year.astype("Int64")
common["is_covid_crash_window"] = common["trade_date"].between(COVID_CRASH_START, COVID_CRASH_END)
common["period_label"] = np.where(
    common["is_covid_crash_window"],
    "covid_crash_window",
    np.where(
        common["actual_top_decile_common_by_tenor"],
        "non_covid_actual_top_decile",
        "non_crash_non_top_decile",
    ),
)

# ======================================================================================
# 4. Metric tables on common rows
# ======================================================================================

metrics_by_bucket = summarize_metrics(
    common,
    ["model_family", "tenor_bucket", "forecast_repair_scope"],
)

metrics_by_tenor = summarize_metrics(
    common,
    ["model_family", "tenor", "tenor_bucket", "forecast_repair_scope"],
)

metrics_by_year = summarize_metrics(
    common,
    ["model_family", "year", "tenor_bucket", "forecast_repair_scope"],
)

metrics_crash_vs_noncrash = summarize_metrics(
    common,
    ["model_family", "period_label", "tenor_bucket", "forecast_repair_scope"],
)

normal_period_overforecast = summarize_metrics(
    common[common["normal_period_common_by_tenor"]].copy(),
    ["model_family", "tenor_bucket", "forecast_repair_scope"],
)

bad_outcome_overlap = summarize_bad_outcome_overlap(
    common,
    ["model_family", "tenor", "tenor_bucket", "forecast_repair_scope"],
)

metrics_by_bucket_vs_existing = add_vs_existing(
    metrics_by_bucket,
    ["tenor_bucket", "forecast_repair_scope"],
)

metrics_by_tenor_vs_existing = add_vs_existing(
    metrics_by_tenor,
    ["tenor", "tenor_bucket", "forecast_repair_scope"],
)

metrics_by_year_vs_existing = add_vs_existing(
    metrics_by_year,
    ["year", "tenor_bucket", "forecast_repair_scope"],
)

crash_vs_noncrash_vs_existing = add_vs_existing(
    metrics_crash_vs_noncrash,
    ["period_label", "tenor_bucket", "forecast_repair_scope"],
)

# ======================================================================================
# 5. Candidate recommendation table
# ======================================================================================

candidate_only = metrics_by_bucket_vs_existing[
    metrics_by_bucket_vs_existing["model_family"] != EXISTING_MODEL
].copy()

candidate_only["bucket_review_score_not_selection"] = (
    candidate_only["capture_improvement_vs_existing"].fillna(0.0) * 3.0
    + candidate_only["missed_top_decile_reduction_vs_existing"].fillna(0.0) * 2.0
    - candidate_only["rmse_change_vs_existing"].clip(lower=0).fillna(0.0) * 0.75
    - candidate_only["large_underforecast_2p0x_change_vs_existing"].clip(lower=0).fillna(0.0) * 0.50
    - candidate_only["normal_period_overforecast_change_vs_existing"].clip(lower=0).fillna(0.0) * 0.25
    - candidate_only["forecast_vol_mean_change_vs_existing"].abs().fillna(0.0) * 0.01
)

recommendation_rows = []

for model_family, g in candidate_only.groupby("model_family", dropna=False):
    front_row = g[g["forecast_repair_scope"] == "RESEARCH_FRONT"]
    middle_row = g[g["forecast_repair_scope"] == "RESEARCH_MIDDLE"]

    avg_capture_improvement = float(g["capture_improvement_vs_existing"].mean())
    min_capture_improvement = float(g["capture_improvement_vs_existing"].min())
    worst_rmse_change = float(g["rmse_change_vs_existing"].max())
    avg_rmse_change = float(g["rmse_change_vs_existing"].mean())
    avg_large_under_2x_change = float(g["large_underforecast_2p0x_change_vs_existing"].mean())
    worst_normal_overforecast_change = float(g["normal_period_overforecast_change_vs_existing"].max())
    avg_forecast_vol_change = float(g["forecast_vol_mean_change_vs_existing"].mean())
    avg_score = float(g["bucket_review_score_not_selection"].mean())

    if (
        avg_capture_improvement >= MIN_AVG_CAPTURE_IMPROVEMENT_ADVANCE
        and worst_rmse_change <= MAX_WORST_BUCKET_RMSE_DETERIORATION_ADVANCE
    ):
        recommendation = "ADVANCE_TO_SIGNAL_PANEL"
    elif (
        avg_capture_improvement >= MIN_AVG_CAPTURE_IMPROVEMENT_BACKUP
        and worst_rmse_change <= MAX_WORST_BUCKET_RMSE_DETERIORATION_BACKUP
    ):
        recommendation = "KEEP_FOR_BACKUP"
    else:
        recommendation = "DROP"

    recommendation_rows.append({
        "model_family": model_family,
        "recommendation": recommendation,
        "avg_review_score_not_selection": avg_score,
        "avg_capture_improvement_vs_existing": avg_capture_improvement,
        "min_capture_improvement_vs_existing": min_capture_improvement,
        "avg_rmse_change_vs_existing": avg_rmse_change,
        "worst_bucket_rmse_change_vs_existing": worst_rmse_change,
        "avg_large_underforecast_2p0x_change_vs_existing": avg_large_under_2x_change,
        "worst_normal_period_overforecast_change_vs_existing": worst_normal_overforecast_change,
        "avg_forecast_vol_mean_change_vs_existing": avg_forecast_vol_change,
        "front_capture_improvement": (
            float(front_row["capture_improvement_vs_existing"].iloc[0]) if len(front_row) else np.nan
        ),
        "middle_capture_improvement": (
            float(middle_row["capture_improvement_vs_existing"].iloc[0]) if len(middle_row) else np.nan
        ),
        "front_rmse_change": (
            float(front_row["rmse_change_vs_existing"].iloc[0]) if len(front_row) else np.nan
        ),
        "middle_rmse_change": (
            float(middle_row["rmse_change_vs_existing"].iloc[0]) if len(middle_row) else np.nan
        ),
    })

recommendation = pd.DataFrame(recommendation_rows).sort_values(
    ["recommendation", "avg_review_score_not_selection"],
    ascending=[True, False],
)

# Put ADVANCE first, then KEEP, then DROP.
recommendation_order = {
    "ADVANCE_TO_SIGNAL_PANEL": 0,
    "KEEP_FOR_BACKUP": 1,
    "DROP": 2,
}
recommendation["recommendation_sort"] = recommendation["recommendation"].map(recommendation_order).fillna(99)
recommendation = recommendation.sort_values(
    ["recommendation_sort", "avg_review_score_not_selection"],
    ascending=[True, False],
).drop(columns=["recommendation_sort"])

baseline_reference = metrics_by_bucket_vs_existing[
    metrics_by_bucket_vs_existing["model_family"] == EXISTING_MODEL
].copy()
baseline_reference["recommendation"] = "BASELINE_REFERENCE"

# ======================================================================================
# 6. Validation
# ======================================================================================

validation_rows = []

def add_validation(check: str, status: str, detail: str):
    validation_rows.append({"check": check, "status": status, "detail": detail})

model_counts_common = common.groupby(["trade_date", "tenor"])["model_family"].nunique()
bad_model_count_rows = int((model_counts_common != len(REVIEW_MODELS)).sum())

back_rows = int(common["tenor"].isin(BACK_TENORS).sum())
missing_review_models = sorted(set(REVIEW_MODELS) - set(common["model_family"].unique()))

add_validation("cell5_panel_loaded", "PASS", f"path={cell5_panel_path}; rows={len(panel):,}")
add_validation("review_models_present", "PASS" if not missing_review_models else "FAIL", f"missing={missing_review_models}")
add_validation("common_keys_non_empty", "PASS" if len(common_keys) > 0 else "FAIL", f"common date×tenor keys={len(common_keys):,}")
add_validation("common_panel_model_count", "PASS" if bad_model_count_rows == 0 else "FAIL", f"bad_model_count_rows={bad_model_count_rows}")
add_validation("front_middle_only", "PASS" if back_rows == 0 else "FAIL", f"back_rows={back_rows}")
add_validation("no_new_model_fitting", "PASS", "Cell 6 only reads Cell 5 outputs and recomputes diagnostics")
add_validation("no_signal_or_threshold_sweep", "PASS", "No VRP/z-score/signal threshold calculations performed")
add_validation("no_final_lock", "PASS", "Recommendation is narrowing guidance only")

validation = pd.DataFrame(validation_rows)
failures = validation[validation["status"] == "FAIL"]

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 6 validation failed. See validation table above.")

# ======================================================================================
# 7. Save outputs
# ======================================================================================

date_min = common["trade_date"].min()
date_max = common["trade_date"].max()

common_panel_path = OUT_PROCESSED_DIR / (
    f"06_common_row_candidate_panel_{date_min:%Y%m%d}_{date_max:%Y%m%d}_{TIMESTAMP}.parquet"
)

metrics_by_bucket_path = OUT_AUDIT_DIR / f"06_common_row_metrics_by_bucket_{TIMESTAMP}.csv"
metrics_by_tenor_path = OUT_AUDIT_DIR / f"06_common_row_metrics_by_tenor_{TIMESTAMP}.csv"
metrics_by_year_path = OUT_AUDIT_DIR / f"06_common_row_metrics_by_year_{TIMESTAMP}.csv"
crash_vs_noncrash_path = OUT_AUDIT_DIR / f"06_common_row_crash_vs_noncrash_{TIMESTAMP}.csv"
normal_overforecast_path = OUT_AUDIT_DIR / f"06_common_row_normal_period_overforecast_{TIMESTAMP}.csv"
bad_outcome_path = OUT_AUDIT_DIR / f"06_common_row_bad_outcome_overlap_{TIMESTAMP}.csv"
metrics_by_bucket_vs_existing_path = OUT_AUDIT_DIR / f"06_common_row_metrics_by_bucket_vs_existing_{TIMESTAMP}.csv"
metrics_by_tenor_vs_existing_path = OUT_AUDIT_DIR / f"06_common_row_metrics_by_tenor_vs_existing_{TIMESTAMP}.csv"
metrics_by_year_vs_existing_path = OUT_AUDIT_DIR / f"06_common_row_metrics_by_year_vs_existing_{TIMESTAMP}.csv"
crash_vs_noncrash_vs_existing_path = OUT_AUDIT_DIR / f"06_common_row_crash_vs_noncrash_vs_existing_{TIMESTAMP}.csv"
recommendation_path = OUT_AUDIT_DIR / f"06_narrowed_candidate_recommendation_{TIMESTAMP}.csv"
validation_path = OUT_AUDIT_DIR / f"06_validation_{TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"06_manifest_{TIMESTAMP}.json"

common.to_parquet(common_panel_path, index=False)
metrics_by_bucket.to_csv(metrics_by_bucket_path, index=False)
metrics_by_tenor.to_csv(metrics_by_tenor_path, index=False)
metrics_by_year.to_csv(metrics_by_year_path, index=False)
metrics_crash_vs_noncrash.to_csv(crash_vs_noncrash_path, index=False)
normal_period_overforecast.to_csv(normal_overforecast_path, index=False)
bad_outcome_overlap.to_csv(bad_outcome_path, index=False)
metrics_by_bucket_vs_existing.to_csv(metrics_by_bucket_vs_existing_path, index=False)
metrics_by_tenor_vs_existing.to_csv(metrics_by_tenor_vs_existing_path, index=False)
metrics_by_year_vs_existing.to_csv(metrics_by_year_vs_existing_path, index=False)
crash_vs_noncrash_vs_existing.to_csv(crash_vs_noncrash_vs_existing_path, index=False)
recommendation.to_csv(recommendation_path, index=False)
validation.to_csv(validation_path, index=False)

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "cell": "Cell 6 — Common-row Front/Middle forecast candidate review",
            "timestamp": TIMESTAMP,
            "source_cell5_panel": str(cell5_panel_path),
            "review_models": REVIEW_MODELS,
            "dropped_from_active_review": DROPPED_FROM_ACTIVE_REVIEW,
            "research_tenors": RESEARCH_TENORS,
            "back_tenors_excluded": BACK_TENORS,
            "covid_crash_window": {
                "start": str(COVID_CRASH_START.date()),
                "end": str(COVID_CRASH_END.date()),
            },
            "recommendation_heuristics": {
                "min_avg_capture_improvement_advance": MIN_AVG_CAPTURE_IMPROVEMENT_ADVANCE,
                "max_worst_bucket_rmse_deterioration_advance": MAX_WORST_BUCKET_RMSE_DETERIORATION_ADVANCE,
                "min_avg_capture_improvement_backup": MIN_AVG_CAPTURE_IMPROVEMENT_BACKUP,
                "max_worst_bucket_rmse_deterioration_backup": MAX_WORST_BUCKET_RMSE_DETERIORATION_BACKUP,
            },
            "notes": [
                "No new model fitting performed.",
                "No Back refit or Back diagnostic performed.",
                "No VRP/z-score signal panel construction performed.",
                "Recommendation table is narrowing guidance, not a final model lock.",
            ],
        },
        f,
        indent=2,
    )

# ======================================================================================
# 8. Display concise review
# ======================================================================================

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 280)

print("=" * 100)
print("Cell 6 — Common-row Front/Middle forecast candidate review")
print("=" * 100)
print(f"Source Cell 5 panel:        {cell5_panel_path}")
print(f"Reviewed models:            {REVIEW_MODELS}")
print(f"Dropped from active review: {DROPPED_FROM_ACTIVE_REVIEW}")
print(f"Common date×tenor keys:     {len(common_keys):,}")
print(f"Common forecast rows:       {len(common):,}")
print(f"Date range:                 {date_min.date()} to {date_max.date()}")
print(f"Research tenors:            {RESEARCH_TENORS}")
print(f"Back rows in common panel:  {back_rows}")

print()
print("Validation:")
display(validation)

print()
print("Narrowed candidate recommendation — NOT final model lock:")
display(recommendation)

bucket_display_cols = [
    "model_family",
    "tenor_bucket",
    "forecast_repair_scope",
    "rows",
    "rmse_log",
    "mae_log",
    "bias_log_pred_minus_actual",
    "underforecast_rate",
    "large_underforecast_1p5x_rate",
    "large_underforecast_2p0x_rate",
    "forecast_top_decile_capture_rate",
    "missed_top_decile_actual_rate",
    "underforecast_rate_in_actual_top_decile",
    "candidate_forecast_vol_pct_mean",
    "normal_period_overforecast_rate",
    "normal_period_mean_log_error",
    "existing_rmse_log",
    "existing_forecast_top_decile_capture_rate",
    "capture_improvement_vs_existing",
    "missed_top_decile_reduction_vs_existing",
    "rmse_change_vs_existing",
    "large_underforecast_2p0x_change_vs_existing",
    "forecast_vol_mean_change_vs_existing",
    "normal_period_overforecast_change_vs_existing",
]

bucket_display = (
    metrics_by_bucket_vs_existing
    .sort_values(
        ["forecast_repair_scope", "capture_improvement_vs_existing", "rmse_change_vs_existing"],
        ascending=[True, False, True],
    )
    .loc[:, bucket_display_cols]
)

print()
print("Common-row bucket metrics vs existing baseline:")
display(bucket_display)

tenor_display_cols = [
    "model_family",
    "tenor",
    "tenor_bucket",
    "forecast_repair_scope",
    "rows",
    "rmse_log",
    "mae_log",
    "bias_log_pred_minus_actual",
    "underforecast_rate",
    "large_underforecast_2p0x_rate",
    "forecast_top_decile_capture_rate",
    "missed_top_decile_actual_rate",
    "underforecast_rate_in_actual_top_decile",
    "candidate_forecast_vol_pct_mean",
    "normal_period_overforecast_rate",
    "existing_forecast_top_decile_capture_rate",
    "capture_improvement_vs_existing",
    "rmse_change_vs_existing",
    "forecast_vol_mean_change_vs_existing",
]

tenor_display = (
    metrics_by_tenor_vs_existing
    .sort_values(
        ["tenor", "capture_improvement_vs_existing", "rmse_change_vs_existing"],
        ascending=[True, False, True],
    )
    .loc[:, tenor_display_cols]
    .head(120)
)

print()
print("Common-row tenor metrics vs existing baseline:")
display(tenor_display)

crash_display_cols = [
    "model_family",
    "period_label",
    "tenor_bucket",
    "forecast_repair_scope",
    "rows",
    "rmse_log",
    "bias_log_pred_minus_actual",
    "underforecast_rate",
    "large_underforecast_2p0x_rate",
    "forecast_top_decile_capture_rate",
    "candidate_forecast_vol_pct_mean",
    "existing_forecast_top_decile_capture_rate",
    "capture_improvement_vs_existing",
    "rmse_change_vs_existing",
    "forecast_vol_mean_change_vs_existing",
]

crash_display = (
    crash_vs_noncrash_vs_existing
    .sort_values(
        ["period_label", "forecast_repair_scope", "capture_improvement_vs_existing"],
        ascending=[True, True, False],
    )
    .loc[:, crash_display_cols]
    .head(120)
)

print()
print("Crash vs non-crash metrics vs existing baseline:")
display(crash_display)

if not bad_outcome_overlap.empty:
    bad_outcome_display = (
        bad_outcome_overlap
        .sort_values(
            ["tenor", "underforecast_rate_in_bottom_decile_pnl"],
            ascending=[True, True],
        )
        .head(120)
    )

    print()
    print("Common-row bad-outcome overlap diagnostics:")
    display(bad_outcome_display)

print()
print("Saved outputs:")
saved_paths = [
    common_panel_path,
    metrics_by_bucket_path,
    metrics_by_tenor_path,
    metrics_by_year_path,
    crash_vs_noncrash_path,
    normal_overforecast_path,
    bad_outcome_path,
    metrics_by_bucket_vs_existing_path,
    metrics_by_tenor_vs_existing_path,
    metrics_by_year_vs_existing_path,
    crash_vs_noncrash_vs_existing_path,
    recommendation_path,
    validation_path,
    manifest_path,
]

for p in saved_paths:
    print(f"  {p}")

print("\nCELL 6 PASSED")

In [ ]:
# ======================================================================================
# Cell 6B — Back unfreeze diagnostic only
# ======================================================================================
#
# Objective:
#   Test whether the same Front/Middle candidate forecast families can improve Back-tenor
#   consistency without damaging the already-strong locked Back trade-selection profile.
#
# Scope:
#   - Load latest Cell 4 enriched candidate-feature panel.
#   - Fit candidate models only for Back tenors: 27, 30, 33.
#   - Use the same expanding yearly OOS leakage guard:
#       train trade_date < test-year start
#       train last_forward_rv_date < test-year start
#   - Compare candidates vs existing har_downside_v1 on common Back rows.
#   - Rebuild candidate model_vrp_log and prior-only z-scores by model × tenor.
#   - Apply locked Core Back thresholds:
#       model_vrp_log > 0.70
#       model_vrp_z_3m > 0.70
#       model_vrp_z_1y > 0.70
#       RSI14 < 70
#       forecast_vol_pct > 8.5
#   - Summarize selected-trade preservation diagnostics.
#
# Explicitly not in scope:
#   - No production Back replacement.
#   - No Front/Middle refit.
#   - No threshold optimization.
#   - No Secondary.
#   - No sizing changes.
#   - No final model lock.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ======================================================================================
# 0. Configuration
# ======================================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
NEW_BRANCH = "vrp_front_middle_corsi_forecast_repair_v1"

OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

FRONT_TENORS = [9, 12, 15]
MIDDLE_TENORS = [18, 21, 24]
BACK_TENORS = [27, 30, 33]

TARGET_LOG_COL = "target_log_variance"
TARGET_VAR_COL = "target_realized_variance"

FORECAST_VAR_BASELINE_COL = "forecast_variance_har_downside_v1"
PRED_LOG_BASELINE_COL = "predicted_log_variance_har_downside_v1"

ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]
FALLBACK_ALPHA = 1.0

MIN_OUTER_TRAIN_ROWS = 250
MIN_INNER_TRAIN_ROWS = 250
MIN_INNER_VAL_ROWS = 30
EPS = 1e-12

# Locked Core Back thresholds.
LOCKED_BACK_THRESHOLDS = {
    "model_vrp_log": 0.70,
    "model_vrp_z_3m": 0.70,
    "model_vrp_z_1y": 0.70,
    "RSI14_max": 70.0,
    "forecast_vol_pct_min": 8.5,
}

EXISTING_MODEL = "existing_har_downside_v1"

REVIEW_MODELS = [
    "existing_har_downside_v1",
    "fast_rv_plus_shock",
    "fast_downside_plus_shock",
    "full_fast_state",
    "shock_core",
]

CANDIDATE_MODEL_FAMILIES = [
    "fast_rv_plus_shock",
    "fast_downside_plus_shock",
    "full_fast_state",
    "shock_core",
]

FORBIDDEN_MODEL_FEATURE_KEYWORDS = [
    "implied", "vix", "vrp", "z_", "rsi", "actual_dte", "win", "pnl",
    "credit", "strike", "expiry", "expiration", "option", "delta", "bucket", "tenor",
    "target", "forward", "last_forward", "trade_outcome",
]

FAST_RV_CORE = [
    "candidate_log_rv_3d",
    "candidate_log_rv_5d",
    "candidate_log_rv_10d",
    "candidate_log_rv_21d",
    "candidate_log_rv_63d",
]

FAST_DOWNSIDE_CORE = [
    "candidate_log_downside_rv_5d",
    "candidate_log_downside_rv_10d",
    "candidate_log_downside_rv_21d",
    "candidate_log_downside_rv_63d",
    "candidate_downside_share_5d",
    "candidate_downside_share_10d",
]

SHOCK_CORE = [
    "candidate_max_abs_return_3d",
    "candidate_max_abs_return_5d",
    "candidate_max_abs_return_10d",
    "candidate_min_return_3d",
    "candidate_min_return_5d",
    "candidate_min_return_10d",
]

ACCELERATION_CORE = [
    "candidate_log_rv_3d_minus_21d",
    "candidate_log_rv_5d_minus_21d",
    "candidate_log_rv_10d_minus_63d",
    "candidate_log_downside_rv_5d_minus_21d",
    "candidate_log_downside_rv_10d_minus_63d",
]

MODEL_FAMILIES = {
    "fast_rv_plus_shock": FAST_RV_CORE + SHOCK_CORE,
    "fast_downside_plus_shock": FAST_DOWNSIDE_CORE + SHOCK_CORE,
    "full_fast_state": FAST_RV_CORE + FAST_DOWNSIDE_CORE + SHOCK_CORE + ACCELERATION_CORE,
    "shock_core": SHOCK_CORE,
}

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def forbidden_feature_hits(cols: list[str], forbidden_keywords: list[str]) -> pd.DataFrame:
    hits = []
    for col in cols:
        col_lower = col.lower()
        for kw in forbidden_keywords:
            if kw in col_lower:
                hits.append({"feature": col, "forbidden_keyword": kw})
    return pd.DataFrame(hits, columns=["feature", "forbidden_keyword"])


def fit_ridge_predict(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    features: list[str],
    target_col: str,
    alpha: float,
) -> np.ndarray:
    pipe = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=alpha)),
        ]
    )
    pipe.fit(train_df[features].to_numpy(), train_df[target_col].to_numpy())
    return pipe.predict(test_df[features].to_numpy())


def choose_alpha_inner_yearly_cv(
    train_pool: pd.DataFrame,
    features: list[str],
    target_col: str,
    alpha_grid: list[float],
    fallback_alpha: float,
) -> tuple[float, pd.DataFrame]:
    cv_rows = []

    available_years = sorted(train_pool["trade_date"].dt.year.dropna().astype(int).unique().tolist())

    for val_year in available_years:
        val_start = pd.Timestamp(f"{val_year}-01-01")
        val_end = pd.Timestamp(f"{val_year + 1}-01-01")

        inner_train = train_pool[
            (train_pool["trade_date"] < val_start)
            & (train_pool["last_forward_rv_date"] < val_start)
        ].copy()

        inner_val = train_pool[
            (train_pool["trade_date"] >= val_start)
            & (train_pool["trade_date"] < val_end)
        ].copy()

        inner_train = inner_train.dropna(subset=features + [target_col])
        inner_val = inner_val.dropna(subset=features + [target_col])

        if len(inner_train) < MIN_INNER_TRAIN_ROWS or len(inner_val) < MIN_INNER_VAL_ROWS:
            continue

        for alpha in alpha_grid:
            try:
                pred = fit_ridge_predict(inner_train, inner_val, features, target_col, alpha)
                cv_rows.append({
                    "val_year": val_year,
                    "alpha": alpha,
                    "inner_train_rows": int(len(inner_train)),
                    "inner_val_rows": int(len(inner_val)),
                    "rmse_log": float(np.sqrt(mean_squared_error(inner_val[target_col], pred))),
                    "mae_log": float(mean_absolute_error(inner_val[target_col], pred)),
                    "bias_log_pred_minus_actual": float(np.mean(pred - inner_val[target_col].to_numpy())),
                })
            except Exception as e:
                cv_rows.append({
                    "val_year": val_year,
                    "alpha": alpha,
                    "inner_train_rows": int(len(inner_train)),
                    "inner_val_rows": int(len(inner_val)),
                    "rmse_log": np.nan,
                    "mae_log": np.nan,
                    "bias_log_pred_minus_actual": np.nan,
                    "error": f"{type(e).__name__}: {e}",
                })

    cv = pd.DataFrame(cv_rows)

    if cv.empty or cv["rmse_log"].notna().sum() == 0:
        return fallback_alpha, cv

    alpha_summary = (
        cv.dropna(subset=["rmse_log"])
        .groupby("alpha", as_index=False)
        .agg(
            cv_folds=("val_year", "nunique"),
            mean_rmse_log=("rmse_log", "mean"),
            mean_mae_log=("mae_log", "mean"),
            mean_bias_log_pred_minus_actual=("bias_log_pred_minus_actual", "mean"),
        )
        .sort_values(["mean_rmse_log", "mean_mae_log", "alpha"], ascending=[True, True, True])
    )

    if alpha_summary.empty:
        return fallback_alpha, cv

    return float(alpha_summary.iloc[0]["alpha"]), cv


def summarize_forecast_metrics(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    rows = []

    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_cols, keys))

        valid_g = g[
            np.isfinite(g["predicted_log_variance_candidate"])
            & np.isfinite(g[TARGET_LOG_COL])
            & np.isfinite(g["forecast_variance_candidate"])
            & np.isfinite(g[TARGET_VAR_COL])
            & (g["forecast_variance_candidate"] > 0)
            & (g[TARGET_VAR_COL] > 0)
        ].copy()

        row["rows"] = int(len(valid_g))

        if len(valid_g) == 0:
            rows.append(row)
            continue

        log_error = valid_g["predicted_log_variance_candidate"] - valid_g[TARGET_LOG_COL]
        ratio = valid_g[TARGET_VAR_COL] / valid_g["forecast_variance_candidate"]
        actual_top = valid_g[valid_g["actual_top_decile_common_by_tenor"]].copy()

        row["rmse_log"] = float(np.sqrt(mean_squared_error(valid_g[TARGET_LOG_COL], valid_g["predicted_log_variance_candidate"])))
        row["mae_log"] = float(mean_absolute_error(valid_g[TARGET_LOG_COL], valid_g["predicted_log_variance_candidate"]))
        row["bias_log_pred_minus_actual"] = float(log_error.mean())
        row["median_log_error"] = float(log_error.median())
        row["underforecast_rate"] = float((valid_g["forecast_variance_candidate"] < valid_g[TARGET_VAR_COL]).mean())
        row["large_underforecast_1p5x_rate"] = float((ratio > 1.5).mean())
        row["large_underforecast_2p0x_rate"] = float((ratio > 2.0).mean())
        row["actual_top_decile_rows"] = int(len(actual_top))
        row["forecast_top_decile_capture_rate"] = (
            float(actual_top["forecast_top_decile_common_by_model_tenor"].mean()) if len(actual_top) else np.nan
        )
        row["missed_top_decile_actual_rate"] = (
            float((~actual_top["forecast_top_decile_common_by_model_tenor"]).mean()) if len(actual_top) else np.nan
        )
        row["underforecast_rate_in_actual_top_decile"] = (
            float((actual_top["forecast_variance_candidate"] < actual_top[TARGET_VAR_COL]).mean()) if len(actual_top) else np.nan
        )
        row["avg_actual_to_forecast_var_ratio"] = float(ratio.mean())
        row["median_actual_to_forecast_var_ratio"] = float(ratio.median())
        row["target_realized_vol_pct_mean"] = float(valid_g["target_realized_vol_pct"].mean())
        row["candidate_forecast_vol_pct_mean"] = float(valid_g["candidate_forecast_vol_pct"].mean())

        rows.append(row)

    return pd.DataFrame(rows)


def add_vs_existing(metrics: pd.DataFrame, join_cols: list[str]) -> pd.DataFrame:
    baseline = metrics[metrics["model_family"] == EXISTING_MODEL].copy()

    baseline_keep = join_cols + [
        "rmse_log",
        "mae_log",
        "bias_log_pred_minus_actual",
        "underforecast_rate",
        "large_underforecast_1p5x_rate",
        "large_underforecast_2p0x_rate",
        "forecast_top_decile_capture_rate",
        "missed_top_decile_actual_rate",
        "underforecast_rate_in_actual_top_decile",
        "candidate_forecast_vol_pct_mean",
    ]

    rename_map = {
        c: f"existing_{c}"
        for c in baseline_keep
        if c not in join_cols
    }

    baseline = baseline[baseline_keep].rename(columns=rename_map)
    out = metrics.merge(baseline, on=join_cols, how="left")

    out["capture_improvement_vs_existing"] = (
        out["forecast_top_decile_capture_rate"] - out["existing_forecast_top_decile_capture_rate"]
    )
    out["missed_top_decile_reduction_vs_existing"] = (
        out["existing_missed_top_decile_actual_rate"] - out["missed_top_decile_actual_rate"]
    )
    out["rmse_change_vs_existing"] = out["rmse_log"] - out["existing_rmse_log"]
    out["mae_change_vs_existing"] = out["mae_log"] - out["existing_mae_log"]
    out["large_underforecast_2p0x_change_vs_existing"] = (
        out["large_underforecast_2p0x_rate"] - out["existing_large_underforecast_2p0x_rate"]
    )
    out["forecast_vol_mean_change_vs_existing"] = (
        out["candidate_forecast_vol_pct_mean"] - out["existing_candidate_forecast_vol_pct_mean"]
    )

    return out


def compute_prior_z_by_model_tenor(df: pd.DataFrame, value_col: str, window: int, min_periods: int) -> pd.Series:
    out = pd.Series(np.nan, index=df.index, dtype="float64")

    for _, g in df.sort_values(["model_family", "tenor", "trade_date"]).groupby(["model_family", "tenor"]):
        shifted = g[value_col].shift(1)
        roll_mean = shifted.rolling(window=window, min_periods=min_periods).mean()
        roll_std = shifted.rolling(window=window, min_periods=min_periods).std(ddof=1)
        z = (g[value_col] - roll_mean) / roll_std.replace(0.0, np.nan)
        out.loc[g.index] = z

    return out


def max_drawdown_from_pnl(pnl: pd.Series) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)
    cumulative = pnl.cumsum()
    running_max = cumulative.cummax()
    drawdown = cumulative - running_max
    return float(drawdown.min()) if len(drawdown) else np.nan


def worst_rolling_sum(pnl: pd.Series, window: int) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)
    if len(pnl) == 0:
        return np.nan
    if len(pnl) < window:
        return float(pnl.sum())
    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def summarize_trade_set(df: pd.DataFrame, universe_label: str) -> pd.DataFrame:
    rows = []

    if df.empty:
        return pd.DataFrame(columns=[
            "selection_universe",
            "model_family",
            "trades",
            "date_min",
            "date_max",
            "win_rate",
            "total_pnl",
            "avg_pnl",
            "median_pnl",
            "worst_trade",
            "best_trade",
            "profit_factor",
            "selected_trade_drawdown",
            "worst_20_trade_sum",
            "trades_le_neg_100k",
            "pnl_2025",
            "trades_2025",
            "worst_trade_2025",
        ])

    d = df.copy()
    d["normalized_pnl_dollars"] = pd.to_numeric(d["normalized_pnl_dollars"], errors="coerce")
    d["win"] = pd.to_numeric(d["win"], errors="coerce")
    d = d[d["normalized_pnl_dollars"].notna()].copy()
    d = d.sort_values(["model_family", "trade_date", "tenor"])

    for model_family, g in d.groupby("model_family", dropna=False):
        pnl = g["normalized_pnl_dollars"]
        wins = g["win"]

        gross_profit = float(pnl[pnl > 0].sum())
        gross_loss = float(pnl[pnl < 0].sum())
        profit_factor = gross_profit / abs(gross_loss) if gross_loss < 0 else np.inf

        y2025 = g[g["trade_date"].dt.year == 2025].copy()

        rows.append({
            "selection_universe": universe_label,
            "model_family": model_family,
            "trades": int(len(g)),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,
            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor,
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "trades_2025": int(len(y2025)),
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,
        })

    return pd.DataFrame(rows)


def compare_trade_summary_to_existing(summary: pd.DataFrame) -> pd.DataFrame:
    if summary.empty or EXISTING_MODEL not in set(summary["model_family"]):
        return summary.copy()

    out_rows = []

    for universe, g in summary.groupby("selection_universe", dropna=False):
        base_rows = g[g["model_family"] == EXISTING_MODEL]
        if base_rows.empty:
            continue

        base = base_rows.iloc[0]

        for _, row in g.iterrows():
            r = row.to_dict()

            r["trades_change_vs_existing"] = row["trades"] - base["trades"]
            r["win_rate_change_vs_existing"] = row["win_rate"] - base["win_rate"]
            r["total_pnl_change_vs_existing"] = row["total_pnl"] - base["total_pnl"]
            r["profit_factor_change_vs_existing"] = row["profit_factor"] - base["profit_factor"]
            r["worst_trade_change_vs_existing"] = row["worst_trade"] - base["worst_trade"]
            r["drawdown_change_vs_existing"] = row["selected_trade_drawdown"] - base["selected_trade_drawdown"]
            r["trades_le_neg_100k_change_vs_existing"] = row["trades_le_neg_100k"] - base["trades_le_neg_100k"]
            r["pnl_2025_change_vs_existing"] = row["pnl_2025"] - base["pnl_2025"]

            if row["model_family"] == EXISTING_MODEL:
                r["back_preservation_screen"] = "BASELINE_REFERENCE"
            else:
                pass_screen = (
                    (row["win_rate"] >= base["win_rate"] - 0.005)
                    and (row["profit_factor"] >= base["profit_factor"])
                    and (row["worst_trade"] >= base["worst_trade"])
                    and (row["selected_trade_drawdown"] >= base["selected_trade_drawdown"])
                    and (row["trades_le_neg_100k"] <= base["trades_le_neg_100k"])
                    and (row["pnl_2025"] >= base["pnl_2025"])
                )
                r["back_preservation_screen"] = "PASS" if pass_screen else "FAIL"

            out_rows.append(r)

    return pd.DataFrame(out_rows)


# ======================================================================================
# 2. Load latest Cell 4 feature panel
# ======================================================================================

cell4_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "04_front_middle_candidate_feature_panel_*.parquet",
    required=True,
)

panel = pd.read_parquet(cell4_panel_path)

# Remove duplicate column labels defensively.
duplicate_cols = panel.columns[panel.columns.duplicated()].tolist()
if duplicate_cols:
    print("Dropping duplicate column labels in Cell 6B source panel:")
    print(sorted(set(duplicate_cols)))
panel = panel.loc[:, ~panel.columns.duplicated(keep="last")].copy()

panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce").dt.normalize()
panel["last_forward_rv_date"] = pd.to_datetime(panel["last_forward_rv_date"], errors="coerce").dt.normalize()
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype("Int64")

required_cols = [
    "trade_date",
    "last_forward_rv_date",
    "tenor",
    "tenor_bucket",
    TARGET_LOG_COL,
    TARGET_VAR_COL,
    FORECAST_VAR_BASELINE_COL,
    "implied_variance",
    "RSI14",
    "win",
    "normalized_pnl_dollars",
]

missing_required = [c for c in required_cols if c not in panel.columns]
if missing_required:
    raise KeyError(f"Cell 6B missing required columns from Cell 4 panel: {missing_required}")

back_panel = panel[panel["tenor"].isin(BACK_TENORS)].copy()

for c in [TARGET_LOG_COL, TARGET_VAR_COL, FORECAST_VAR_BASELINE_COL, "implied_variance", "RSI14"]:
    back_panel[c] = pd.to_numeric(back_panel[c], errors="coerce")

back_panel = back_panel[
    back_panel["trade_date"].notna()
    & back_panel["last_forward_rv_date"].notna()
    & np.isfinite(back_panel[TARGET_LOG_COL])
    & np.isfinite(back_panel[TARGET_VAR_COL])
    & (back_panel[TARGET_VAR_COL] > 0)
    & np.isfinite(back_panel["implied_variance"])
    & (back_panel["implied_variance"] > 0)
].copy()

if back_panel.empty:
    raise RuntimeError("No valid Back rows found for Cell 6B.")

# Validate feature families.
feature_validation_rows = []
usable_model_families = {}

for family in CANDIDATE_MODEL_FAMILIES:
    features = MODEL_FAMILIES[family]
    missing = [c for c in features if c not in back_panel.columns]
    forbidden_hits = forbidden_feature_hits(features, FORBIDDEN_MODEL_FEATURE_KEYWORDS)
    status = "PASS" if not missing and forbidden_hits.empty else "FAIL"

    feature_validation_rows.append({
        "model_family": family,
        "status": status,
        "feature_count": len(features),
        "missing_features": missing,
        "forbidden_hits": forbidden_hits.to_dict("records") if not forbidden_hits.empty else [],
        "features": features,
    })

    if status == "PASS":
        usable_model_families[family] = features

feature_validation = pd.DataFrame(feature_validation_rows)

if feature_validation["status"].eq("FAIL").any():
    display(feature_validation)
    raise RuntimeError("One or more Back diagnostic model families failed feature validation.")

# ======================================================================================
# 3. Existing Back baseline rows
# ======================================================================================

baseline = back_panel.copy()
baseline["model_family"] = EXISTING_MODEL
baseline["model_source"] = "existing_locked_oos_forecast"
baseline["selected_alpha_candidate"] = pd.to_numeric(baseline.get("selected_alpha", np.nan), errors="coerce")
baseline["train_rows_used_candidate"] = pd.to_numeric(baseline.get("train_rows_used", np.nan), errors="coerce")
baseline["test_year_candidate"] = baseline["trade_date"].dt.year.astype("Int64")
baseline["fit_status_candidate"] = "existing_forecast"

if PRED_LOG_BASELINE_COL in baseline.columns:
    baseline["predicted_log_variance_candidate"] = pd.to_numeric(baseline[PRED_LOG_BASELINE_COL], errors="coerce")
else:
    baseline["predicted_log_variance_candidate"] = np.log(baseline[FORECAST_VAR_BASELINE_COL].clip(lower=EPS))

baseline["forecast_variance_candidate"] = baseline[FORECAST_VAR_BASELINE_COL]
baseline["candidate_forecast_vol_pct"] = np.sqrt(baseline["forecast_variance_candidate"].clip(lower=0)) * 100.0
baseline["target_realized_vol_pct"] = np.sqrt(baseline[TARGET_VAR_COL].clip(lower=0)) * 100.0

# ======================================================================================
# 4. Fit Back candidate models OOS
# ======================================================================================

forecast_rows = []
fit_log_rows = []
inner_cv_rows_all = []

for model_family, features in usable_model_families.items():
    for tenor in BACK_TENORS:
        tenor_df = back_panel[back_panel["tenor"] == tenor].copy()
        tenor_df = tenor_df.sort_values("trade_date").reset_index(drop=True)

        test_years = sorted(tenor_df["trade_date"].dt.year.dropna().astype(int).unique().tolist())

        for test_year in test_years:
            test_start = pd.Timestamp(f"{test_year}-01-01")
            test_end = pd.Timestamp(f"{test_year + 1}-01-01")

            train_pool = tenor_df[
                (tenor_df["trade_date"] < test_start)
                & (tenor_df["last_forward_rv_date"] < test_start)
            ].copy()

            test_rows = tenor_df[
                (tenor_df["trade_date"] >= test_start)
                & (tenor_df["trade_date"] < test_end)
            ].copy()

            train_fit = train_pool.dropna(subset=features + [TARGET_LOG_COL]).copy()
            test_fit = test_rows.dropna(subset=features + [TARGET_LOG_COL]).copy()

            if len(test_fit) == 0:
                fit_log_rows.append({
                    "model_family": model_family,
                    "tenor": tenor,
                    "test_year": test_year,
                    "fit_status": "skipped_no_test_rows",
                    "selected_alpha": np.nan,
                    "train_rows_used": int(len(train_fit)),
                    "test_rows_scored": 0,
                    "feature_count": len(features),
                })
                continue

            if len(train_fit) < MIN_OUTER_TRAIN_ROWS:
                fit_log_rows.append({
                    "model_family": model_family,
                    "tenor": tenor,
                    "test_year": test_year,
                    "fit_status": "skipped_insufficient_train_rows",
                    "selected_alpha": np.nan,
                    "train_rows_used": int(len(train_fit)),
                    "test_rows_scored": 0,
                    "feature_count": len(features),
                })
                continue

            selected_alpha, inner_cv = choose_alpha_inner_yearly_cv(
                train_pool=train_fit,
                features=features,
                target_col=TARGET_LOG_COL,
                alpha_grid=ALPHA_GRID,
                fallback_alpha=FALLBACK_ALPHA,
            )

            if not inner_cv.empty:
                inner_cv["outer_model_family"] = model_family
                inner_cv["outer_tenor"] = tenor
                inner_cv["outer_test_year"] = test_year
                inner_cv_rows_all.append(inner_cv)

            try:
                pred = fit_ridge_predict(
                    train_df=train_fit,
                    test_df=test_fit,
                    features=features,
                    target_col=TARGET_LOG_COL,
                    alpha=selected_alpha,
                )

                scored = test_fit.copy()
                scored["model_family"] = model_family
                scored["model_source"] = "back_unfreeze_candidate_oos_refit"
                scored["selected_alpha_candidate"] = selected_alpha
                scored["train_rows_used_candidate"] = int(len(train_fit))
                scored["test_year_candidate"] = int(test_year)
                scored["fit_status_candidate"] = "candidate_fit"
                scored["predicted_log_variance_candidate"] = pred
                scored["forecast_variance_candidate"] = np.exp(pred)
                scored["candidate_forecast_vol_pct"] = np.sqrt(scored["forecast_variance_candidate"].clip(lower=0)) * 100.0
                scored["target_realized_vol_pct"] = np.sqrt(scored[TARGET_VAR_COL].clip(lower=0)) * 100.0

                forecast_rows.append(scored)

                fit_log_rows.append({
                    "model_family": model_family,
                    "tenor": tenor,
                    "test_year": test_year,
                    "fit_status": "candidate_fit",
                    "selected_alpha": selected_alpha,
                    "train_rows_used": int(len(train_fit)),
                    "test_rows_scored": int(len(scored)),
                    "feature_count": len(features),
                })

            except Exception as e:
                fit_log_rows.append({
                    "model_family": model_family,
                    "tenor": tenor,
                    "test_year": test_year,
                    "fit_status": "fit_error",
                    "selected_alpha": selected_alpha,
                    "train_rows_used": int(len(train_fit)),
                    "test_rows_scored": 0,
                    "feature_count": len(features),
                    "error": f"{type(e).__name__}: {e}",
                })

candidate_forecasts = pd.concat(forecast_rows, ignore_index=True) if forecast_rows else pd.DataFrame()
fit_log = pd.DataFrame(fit_log_rows)
inner_cv_all = pd.concat(inner_cv_rows_all, ignore_index=True) if inner_cv_rows_all else pd.DataFrame()

if candidate_forecasts.empty:
    display(fit_log)
    raise RuntimeError("No Back candidate forecasts were produced. See fit_log.")

# Align columns.
for c in baseline.columns:
    if c not in candidate_forecasts.columns:
        candidate_forecasts[c] = np.nan

candidate_forecasts = candidate_forecasts[baseline.columns].copy()
all_forecasts = pd.concat([baseline, candidate_forecasts], ignore_index=True)

# Remove duplicate column labels defensively.
all_forecasts = all_forecasts.loc[:, ~all_forecasts.columns.duplicated(keep="last")].copy()

# Core forecast diagnostics.
all_forecasts["forecast_variance_candidate"] = pd.to_numeric(all_forecasts["forecast_variance_candidate"], errors="coerce")
all_forecasts["predicted_log_variance_candidate"] = pd.to_numeric(all_forecasts["predicted_log_variance_candidate"], errors="coerce")
all_forecasts[TARGET_VAR_COL] = pd.to_numeric(all_forecasts[TARGET_VAR_COL], errors="coerce")
all_forecasts[TARGET_LOG_COL] = pd.to_numeric(all_forecasts[TARGET_LOG_COL], errors="coerce")
all_forecasts["implied_variance"] = pd.to_numeric(all_forecasts["implied_variance"], errors="coerce")
all_forecasts["RSI14"] = pd.to_numeric(all_forecasts["RSI14"], errors="coerce")

all_forecasts = all_forecasts[
    np.isfinite(all_forecasts["forecast_variance_candidate"])
    & np.isfinite(all_forecasts["predicted_log_variance_candidate"])
    & np.isfinite(all_forecasts[TARGET_VAR_COL])
    & np.isfinite(all_forecasts[TARGET_LOG_COL])
    & np.isfinite(all_forecasts["implied_variance"])
    & (all_forecasts["forecast_variance_candidate"] > 0)
    & (all_forecasts[TARGET_VAR_COL] > 0)
    & (all_forecasts["implied_variance"] > 0)
].copy()

all_forecasts["target_realized_vol_pct"] = np.sqrt(all_forecasts[TARGET_VAR_COL].clip(lower=0)) * 100.0
all_forecasts["candidate_forecast_vol_pct"] = np.sqrt(all_forecasts["forecast_variance_candidate"].clip(lower=0)) * 100.0
all_forecasts["log_error_pred_minus_actual"] = all_forecasts["predicted_log_variance_candidate"] - all_forecasts[TARGET_LOG_COL]
all_forecasts["actual_to_forecast_var_ratio"] = all_forecasts[TARGET_VAR_COL] / all_forecasts["forecast_variance_candidate"]

# ======================================================================================
# 5. Common Back row forecast diagnostics
# ======================================================================================

review = all_forecasts[
    all_forecasts["model_family"].isin(REVIEW_MODELS)
    & all_forecasts["tenor"].isin(BACK_TENORS)
].copy()

row_model_counts = (
    review.groupby(["trade_date", "tenor"], as_index=False)
    .agg(model_count=("model_family", "nunique"))
)

common_keys = row_model_counts[row_model_counts["model_count"] == len(REVIEW_MODELS)][["trade_date", "tenor"]].copy()

if common_keys.empty:
    raise RuntimeError("No common Back date × tenor rows found across all review models.")

common = review.merge(common_keys, on=["trade_date", "tenor"], how="inner", validate="many_to_one").copy()

actual_common_rank_source = (
    common[["trade_date", "tenor", TARGET_VAR_COL]]
    .drop_duplicates(["trade_date", "tenor"])
    .copy()
)

actual_common_rank_source["actual_var_pct_rank_common_by_tenor"] = (
    actual_common_rank_source.groupby("tenor")[TARGET_VAR_COL].rank(pct=True)
)
actual_common_rank_source["actual_top_decile_common_by_tenor"] = (
    actual_common_rank_source["actual_var_pct_rank_common_by_tenor"] >= 0.90
)

common = common.merge(
    actual_common_rank_source[
        ["trade_date", "tenor", "actual_var_pct_rank_common_by_tenor", "actual_top_decile_common_by_tenor"]
    ],
    on=["trade_date", "tenor"],
    how="left",
    validate="many_to_one",
)

common["forecast_var_pct_rank_common_by_model_tenor"] = (
    common.groupby(["model_family", "tenor"])["forecast_variance_candidate"].rank(pct=True)
)
common["forecast_top_decile_common_by_model_tenor"] = (
    common["forecast_var_pct_rank_common_by_model_tenor"] >= 0.90
)

forecast_metrics_by_bucket = summarize_forecast_metrics(
    common,
    ["model_family", "tenor_bucket"],
)

forecast_metrics_by_tenor = summarize_forecast_metrics(
    common,
    ["model_family", "tenor", "tenor_bucket"],
)

forecast_metrics_by_year = summarize_forecast_metrics(
    common.assign(year=common["trade_date"].dt.year.astype("Int64")),
    ["model_family", "year", "tenor_bucket"],
)

forecast_metrics_by_bucket_vs_existing = add_vs_existing(
    forecast_metrics_by_bucket,
    ["tenor_bucket"],
)

forecast_metrics_by_tenor_vs_existing = add_vs_existing(
    forecast_metrics_by_tenor,
    ["tenor", "tenor_bucket"],
)

# ======================================================================================
# 6. Rebuild Back candidate VRP and prior-only z-scores
# ======================================================================================

signal_base = all_forecasts[
    all_forecasts["model_family"].isin(REVIEW_MODELS)
    & all_forecasts["tenor"].isin(BACK_TENORS)
].copy()

signal_base = signal_base.sort_values(["model_family", "tenor", "trade_date"]).reset_index(drop=True)

signal_base["model_vrp_log_candidate"] = np.log(
    signal_base["implied_variance"] / signal_base["forecast_variance_candidate"]
)

signal_base["model_vrp_z_3m_candidate"] = compute_prior_z_by_model_tenor(
    signal_base,
    value_col="model_vrp_log_candidate",
    window=63,
    min_periods=63,
)

signal_base["model_vrp_z_1y_candidate"] = compute_prior_z_by_model_tenor(
    signal_base,
    value_col="model_vrp_log_candidate",
    window=252,
    min_periods=252,
)

signal_base["locked_core_back_pass"] = (
    (signal_base["model_vrp_log_candidate"] > LOCKED_BACK_THRESHOLDS["model_vrp_log"])
    & (signal_base["model_vrp_z_3m_candidate"] > LOCKED_BACK_THRESHOLDS["model_vrp_z_3m"])
    & (signal_base["model_vrp_z_1y_candidate"] > LOCKED_BACK_THRESHOLDS["model_vrp_z_1y"])
    & (signal_base["RSI14"] < LOCKED_BACK_THRESHOLDS["RSI14_max"])
    & (signal_base["candidate_forecast_vol_pct"] > LOCKED_BACK_THRESHOLDS["forecast_vol_pct_min"])
)

# Restrict signal comparison to common rows where every model has the same date × tenor coverage.
common_signal = signal_base.merge(
    common_keys,
    on=["trade_date", "tenor"],
    how="inner",
    validate="many_to_one",
).copy()

common_signal["signal_inputs_available"] = (
    np.isfinite(common_signal["model_vrp_log_candidate"])
    & np.isfinite(common_signal["model_vrp_z_3m_candidate"])
    & np.isfinite(common_signal["model_vrp_z_1y_candidate"])
    & np.isfinite(common_signal["RSI14"])
    & np.isfinite(common_signal["candidate_forecast_vol_pct"])
    & common_signal["normalized_pnl_dollars"].notna()
)

qualified_all_rows = common_signal[
    common_signal["locked_core_back_pass"]
    & common_signal["signal_inputs_available"]
].copy()

qualified_30d_anchor = qualified_all_rows[qualified_all_rows["tenor"] == 30].copy()

# Diagnostic one-per-date Back bucket selection.
# This is NOT a production selection rule. It chooses the strongest candidate row per model/date
# by z_1y, then z_3m, then model_vrp_log, then tenor.
qualified_bucket_selected = (
    qualified_all_rows
    .sort_values(
        ["model_family", "trade_date", "model_vrp_z_1y_candidate", "model_vrp_z_3m_candidate", "model_vrp_log_candidate", "tenor"],
        ascending=[True, True, False, False, False, False],
    )
    .groupby(["model_family", "trade_date"], as_index=False)
    .head(1)
    .copy()
)

trade_summary_all_qualified = summarize_trade_set(qualified_all_rows, "all_qualified_back_rows")
trade_summary_30d_anchor = summarize_trade_set(qualified_30d_anchor, "thirty_day_anchor_only")
trade_summary_bucket_selected = summarize_trade_set(qualified_bucket_selected, "one_back_trade_per_date_best_vrp")

trade_summary = pd.concat(
    [trade_summary_all_qualified, trade_summary_30d_anchor, trade_summary_bucket_selected],
    ignore_index=True,
)

trade_summary_vs_existing = compare_trade_summary_to_existing(trade_summary)

# Signal pass-rate diagnostics.
signal_pass_diagnostics = (
    common_signal
    .groupby(["model_family", "tenor"], as_index=False)
    .agg(
        rows=("trade_date", "size"),
        signal_inputs_available_rows=("signal_inputs_available", "sum"),
        locked_core_back_pass_rows=("locked_core_back_pass", "sum"),
        pass_rate=("locked_core_back_pass", "mean"),
        avg_model_vrp_log=("model_vrp_log_candidate", "mean"),
        avg_z_3m=("model_vrp_z_3m_candidate", "mean"),
        avg_z_1y=("model_vrp_z_1y_candidate", "mean"),
        avg_forecast_vol_pct=("candidate_forecast_vol_pct", "mean"),
    )
)

# ======================================================================================
# 7. Validation
# ======================================================================================

validation_rows = []

def add_validation(check: str, status: str, detail: str):
    validation_rows.append({"check": check, "status": status, "detail": detail})

front_middle_candidate_rows = candidate_forecasts[
    candidate_forecasts["tenor"].isin(FRONT_TENORS + MIDDLE_TENORS)
]

bad_model_count_rows = int(
    (common.groupby(["trade_date", "tenor"])["model_family"].nunique() != len(REVIEW_MODELS)).sum()
)

add_validation("cell4_panel_loaded", "PASS", f"path={cell4_panel_path}; rows={len(panel):,}")
add_validation("back_panel_non_empty", "PASS" if len(back_panel) > 0 else "FAIL", f"back rows={len(back_panel):,}")
add_validation("feature_validation", "PASS", "all candidate model families passed feature availability and forbidden-keyword audit")
add_validation("candidate_back_forecasts_produced", "PASS", f"candidate rows={len(candidate_forecasts):,}")
add_validation(
    "no_front_middle_candidate_refit_rows",
    "PASS" if front_middle_candidate_rows.empty else "FAIL",
    f"front/middle candidate rows={len(front_middle_candidate_rows):,}",
)
add_validation("common_back_keys_non_empty", "PASS", f"common date×tenor keys={len(common_keys):,}")
add_validation(
    "common_back_model_count",
    "PASS" if bad_model_count_rows == 0 else "FAIL",
    f"bad_model_count_rows={bad_model_count_rows}",
)
add_validation("fit_errors", "PASS" if not fit_log["fit_status"].eq("fit_error").any() else "WARN", f"fit_error_rows={int(fit_log['fit_status'].eq('fit_error').sum())}")
add_validation("locked_thresholds_applied_without_optimization", "PASS", str(LOCKED_BACK_THRESHOLDS))
add_validation("no_final_back_unfreeze_lock", "PASS", "Cell 6B is diagnostic only")

validation = pd.DataFrame(validation_rows)
failures = validation[validation["status"] == "FAIL"]

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 6B validation failed. See validation table above.")

# ======================================================================================
# 8. Save outputs
# ======================================================================================

date_min = common["trade_date"].min()
date_max = common["trade_date"].max()

forecast_panel_path = OUT_PROCESSED_DIR / (
    f"06B_back_unfreeze_oos_forecast_panel_{date_min:%Y%m%d}_{date_max:%Y%m%d}_{TIMESTAMP}.parquet"
)
common_panel_path = OUT_PROCESSED_DIR / (
    f"06B_back_unfreeze_common_row_panel_{date_min:%Y%m%d}_{date_max:%Y%m%d}_{TIMESTAMP}.parquet"
)
signal_panel_path = OUT_PROCESSED_DIR / (
    f"06B_back_unfreeze_signal_threshold_panel_{date_min:%Y%m%d}_{date_max:%Y%m%d}_{TIMESTAMP}.parquet"
)

forecast_metrics_by_bucket_path = OUT_AUDIT_DIR / f"06B_back_forecast_metrics_by_bucket_{TIMESTAMP}.csv"
forecast_metrics_by_tenor_path = OUT_AUDIT_DIR / f"06B_back_forecast_metrics_by_tenor_{TIMESTAMP}.csv"
forecast_metrics_by_year_path = OUT_AUDIT_DIR / f"06B_back_forecast_metrics_by_year_{TIMESTAMP}.csv"
forecast_metrics_by_bucket_vs_existing_path = OUT_AUDIT_DIR / f"06B_back_forecast_metrics_by_bucket_vs_existing_{TIMESTAMP}.csv"
forecast_metrics_by_tenor_vs_existing_path = OUT_AUDIT_DIR / f"06B_back_forecast_metrics_by_tenor_vs_existing_{TIMESTAMP}.csv"
trade_summary_path = OUT_AUDIT_DIR / f"06B_back_locked_threshold_trade_summary_{TIMESTAMP}.csv"
trade_summary_vs_existing_path = OUT_AUDIT_DIR / f"06B_back_locked_threshold_trade_summary_vs_existing_{TIMESTAMP}.csv"
signal_pass_diagnostics_path = OUT_AUDIT_DIR / f"06B_back_signal_pass_diagnostics_{TIMESTAMP}.csv"
fit_log_path = OUT_AUDIT_DIR / f"06B_back_candidate_fit_log_{TIMESTAMP}.csv"
inner_cv_path = OUT_AUDIT_DIR / f"06B_back_candidate_inner_cv_{TIMESTAMP}.csv"
feature_validation_path = OUT_AUDIT_DIR / f"06B_back_feature_validation_{TIMESTAMP}.csv"
validation_path = OUT_AUDIT_DIR / f"06B_validation_{TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"06B_manifest_{TIMESTAMP}.json"

all_forecasts.to_parquet(forecast_panel_path, index=False)
common.to_parquet(common_panel_path, index=False)
common_signal.to_parquet(signal_panel_path, index=False)

forecast_metrics_by_bucket.to_csv(forecast_metrics_by_bucket_path, index=False)
forecast_metrics_by_tenor.to_csv(forecast_metrics_by_tenor_path, index=False)
forecast_metrics_by_year.to_csv(forecast_metrics_by_year_path, index=False)
forecast_metrics_by_bucket_vs_existing.to_csv(forecast_metrics_by_bucket_vs_existing_path, index=False)
forecast_metrics_by_tenor_vs_existing.to_csv(forecast_metrics_by_tenor_vs_existing_path, index=False)
trade_summary.to_csv(trade_summary_path, index=False)
trade_summary_vs_existing.to_csv(trade_summary_vs_existing_path, index=False)
signal_pass_diagnostics.to_csv(signal_pass_diagnostics_path, index=False)
fit_log.to_csv(fit_log_path, index=False)
feature_validation.to_csv(feature_validation_path, index=False)
validation.to_csv(validation_path, index=False)

if not inner_cv_all.empty:
    inner_cv_all.to_csv(inner_cv_path, index=False)
else:
    pd.DataFrame().to_csv(inner_cv_path, index=False)

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "cell": "Cell 6B — Back unfreeze diagnostic only",
            "timestamp": TIMESTAMP,
            "source_cell4_panel": str(cell4_panel_path),
            "review_models": REVIEW_MODELS,
            "candidate_model_families": MODEL_FAMILIES,
            "back_tenors_refit_for_diagnostic": BACK_TENORS,
            "front_middle_tenors_not_refit": FRONT_TENORS + MIDDLE_TENORS,
            "locked_back_thresholds": LOCKED_BACK_THRESHOLDS,
            "alpha_grid": ALPHA_GRID,
            "fallback_alpha": FALLBACK_ALPHA,
            "notes": [
                "Diagnostic only; does not unfreeze Back in production/research decision.",
                "No threshold optimization performed.",
                "No Front/Middle candidate refit performed.",
                "Signal summaries use locked Core Back thresholds.",
                "one_back_trade_per_date_best_vrp is a diagnostic one-per-date selector, not a locked production rule.",
            ],
        },
        f,
        indent=2,
    )

# ======================================================================================
# 9. Display concise review
# ======================================================================================

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 280)

print("=" * 100)
print("Cell 6B — Back unfreeze diagnostic only")
print("=" * 100)
print(f"Source Cell 4 panel:        {cell4_panel_path}")
print(f"Back tenors refit:          {BACK_TENORS}")
print(f"Front/Middle refit rows:    {len(front_middle_candidate_rows):,}")
print(f"Candidate Back rows:        {len(candidate_forecasts):,}")
print(f"Existing baseline rows:     {len(baseline):,}")
print(f"Common Back date×tenor:     {len(common_keys):,}")
print(f"Common Back forecast rows:  {len(common):,}")
print(f"Date range:                 {date_min.date()} to {date_max.date()}")
print(f"Locked thresholds:          {LOCKED_BACK_THRESHOLDS}")

print()
print("Validation:")
display(validation)

print()
print("Fit status counts:")
display(fit_log["fit_status"].value_counts(dropna=False).to_frame("count"))

print()
print("Back forecast metrics vs existing baseline:")
forecast_bucket_display_cols = [
    "model_family",
    "tenor_bucket",
    "rows",
    "rmse_log",
    "mae_log",
    "bias_log_pred_minus_actual",
    "underforecast_rate",
    "large_underforecast_1p5x_rate",
    "large_underforecast_2p0x_rate",
    "forecast_top_decile_capture_rate",
    "missed_top_decile_actual_rate",
    "underforecast_rate_in_actual_top_decile",
    "candidate_forecast_vol_pct_mean",
    "existing_rmse_log",
    "existing_forecast_top_decile_capture_rate",
    "capture_improvement_vs_existing",
    "missed_top_decile_reduction_vs_existing",
    "rmse_change_vs_existing",
    "large_underforecast_2p0x_change_vs_existing",
    "forecast_vol_mean_change_vs_existing",
]
display(
    forecast_metrics_by_bucket_vs_existing
    .sort_values(["capture_improvement_vs_existing", "rmse_change_vs_existing"], ascending=[False, True])
    .loc[:, forecast_bucket_display_cols]
)

print()
print("Back forecast metrics by tenor vs existing baseline:")
forecast_tenor_display_cols = [
    "model_family",
    "tenor",
    "tenor_bucket",
    "rows",
    "rmse_log",
    "mae_log",
    "bias_log_pred_minus_actual",
    "underforecast_rate",
    "large_underforecast_2p0x_rate",
    "forecast_top_decile_capture_rate",
    "missed_top_decile_actual_rate",
    "candidate_forecast_vol_pct_mean",
    "existing_forecast_top_decile_capture_rate",
    "capture_improvement_vs_existing",
    "rmse_change_vs_existing",
    "forecast_vol_mean_change_vs_existing",
]
display(
    forecast_metrics_by_tenor_vs_existing
    .sort_values(["tenor", "capture_improvement_vs_existing", "rmse_change_vs_existing"], ascending=[True, False, True])
    .loc[:, forecast_tenor_display_cols]
)

print()
print("Locked Core Back signal pass diagnostics:")
display(signal_pass_diagnostics)

print()
print("Locked Core Back threshold trade summary vs existing:")
trade_display_cols = [
    "selection_universe",
    "model_family",
    "back_preservation_screen",
    "trades",
    "win_rate",
    "profit_factor",
    "total_pnl",
    "worst_trade",
    "selected_trade_drawdown",
    "worst_20_trade_sum",
    "trades_le_neg_100k",
    "pnl_2025",
    "trades_2025",
    "worst_trade_2025",
    "trades_change_vs_existing",
    "win_rate_change_vs_existing",
    "total_pnl_change_vs_existing",
    "profit_factor_change_vs_existing",
    "worst_trade_change_vs_existing",
    "drawdown_change_vs_existing",
    "trades_le_neg_100k_change_vs_existing",
    "pnl_2025_change_vs_existing",
]
display(
    trade_summary_vs_existing
    .sort_values(["selection_universe", "back_preservation_screen", "total_pnl"], ascending=[True, True, False])
    .loc[:, [c for c in trade_display_cols if c in trade_summary_vs_existing.columns]]
)

print()
print("Saved outputs:")
saved_paths = [
    forecast_panel_path,
    common_panel_path,
    signal_panel_path,
    forecast_metrics_by_bucket_path,
    forecast_metrics_by_tenor_path,
    forecast_metrics_by_year_path,
    forecast_metrics_by_bucket_vs_existing_path,
    forecast_metrics_by_tenor_vs_existing_path,
    trade_summary_path,
    trade_summary_vs_existing_path,
    signal_pass_diagnostics_path,
    fit_log_path,
    inner_cv_path,
    feature_validation_path,
    validation_path,
    manifest_path,
]

for p in saved_paths:
    print(f"  {p}")

print("\nCELL 6B PASSED")

In [ ]:
# ======================================================================================
# Cell 6C — Pure Back unfreeze model-spec refinement
# ======================================================================================
#
# Objective:
#   Try to unfreeze Back using pure model-spec changes only, while keeping the locked
#   Back signal thresholds unchanged.
#
# Scope:
#   - Load latest Cell 4 enriched candidate-feature panel.
#   - Fit refined pure Back candidate model specs for tenors 27, 30, 33 only.
#   - No blending.
#   - No threshold grid.
#   - No Front/Middle refit.
#   - Rebuild Back model_vrp_log and prior-only z-scores.
#   - Apply the same locked Core Back thresholds:
#       model_vrp_log > 0.70
#       model_vrp_z_3m > 0.70
#       model_vrp_z_1y > 0.70
#       RSI14 < 70
#       forecast_vol_pct > 8.5
#   - Compare selected-trade preservation against existing har_downside_v1.
#
# Explicitly not in scope:
#   - No production Back replacement.
#   - No threshold optimization.
#   - No blending.
#   - No Front/Middle refit.
#   - No Secondary.
#   - No sizing changes.
#   - No final model lock.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, HuberRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ======================================================================================
# 0. Configuration
# ======================================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
NEW_BRANCH = "vrp_front_middle_corsi_forecast_repair_v1"

OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

FRONT_TENORS = [9, 12, 15]
MIDDLE_TENORS = [18, 21, 24]
BACK_TENORS = [27, 30, 33]

TARGET_LOG_COL = "target_log_variance"
TARGET_VAR_COL = "target_realized_variance"

FORECAST_VAR_BASELINE_COL = "forecast_variance_har_downside_v1"
PRED_LOG_BASELINE_COL = "predicted_log_variance_har_downside_v1"

MIN_OUTER_TRAIN_ROWS = 250
MIN_INNER_TRAIN_ROWS = 250
MIN_INNER_VAL_ROWS = 30
EPS = 1e-12

EXISTING_SPEC = "existing_har_downside_v1"

LOCKED_BACK_THRESHOLDS = {
    "model_vrp_log": 0.70,
    "model_vrp_z_3m": 0.70,
    "model_vrp_z_1y": 0.70,
    "RSI14_max": 70.0,
    "forecast_vol_pct_min": 8.5,
}

# Acceptance guide for consistency-driven Back unfreeze.
# This is intentionally looser than Cell 6B's strict preservation screen.
ACCEPTANCE_GUIDE = {
    "thirty_day_total_pnl_max_giveup_frac": 0.075,
    "thirty_day_min_profit_factor": 5.0,
    "thirty_day_min_win_rate_delta": -0.01,
    "thirty_day_min_worst_trade": -110_000.0,
    "thirty_day_min_drawdown": -300_000.0,
    "thirty_day_max_trades_le_neg_100k": 1,
    "thirty_day_min_2025_pnl_frac": 0.75,
    "bucket_selected_total_pnl_max_giveup_frac": 0.10,
    "bucket_selected_max_trades_le_neg_100k": 3,
}

FORBIDDEN_MODEL_FEATURE_KEYWORDS = [
    "implied", "vix", "vrp", "z_", "rsi", "actual_dte", "win", "pnl",
    "credit", "strike", "expiry", "expiration", "option", "delta", "bucket", "tenor",
    "target", "forward", "last_forward", "trade_outcome",
]

FAST_RV_CORE = [
    "candidate_log_rv_3d",
    "candidate_log_rv_5d",
    "candidate_log_rv_10d",
    "candidate_log_rv_21d",
    "candidate_log_rv_63d",
]

FAST_RV_SLOW = [
    "candidate_log_rv_10d",
    "candidate_log_rv_21d",
    "candidate_log_rv_63d",
]

FAST_DOWNSIDE_CORE = [
    "candidate_log_downside_rv_5d",
    "candidate_log_downside_rv_10d",
    "candidate_log_downside_rv_21d",
    "candidate_log_downside_rv_63d",
    "candidate_downside_share_5d",
    "candidate_downside_share_10d",
]

FAST_DOWNSIDE_SLOW = [
    "candidate_log_downside_rv_10d",
    "candidate_log_downside_rv_21d",
    "candidate_log_downside_rv_63d",
    "candidate_downside_share_10d",
]

SHOCK_MAX_ABS = [
    "candidate_max_abs_return_3d",
    "candidate_max_abs_return_5d",
    "candidate_max_abs_return_10d",
]

SHOCK_MIN_RETURN = [
    "candidate_min_return_3d",
    "candidate_min_return_5d",
    "candidate_min_return_10d",
]

SHOCK_CORE = SHOCK_MAX_ABS + SHOCK_MIN_RETURN

ACCELERATION_REDUCED = [
    "candidate_log_rv_10d_minus_63d",
    "candidate_log_downside_rv_10d_minus_63d",
]

MODEL_SPECS = {
    "fds_shock_raw_ridge": {
        "features": FAST_DOWNSIDE_CORE + SHOCK_CORE,
        "estimator": "ridge",
        "alpha_grid": [1.0, 10.0, 100.0, 300.0, 1000.0],
        "fallback_alpha": 100.0,
        "target_transform": "raw",
    },
    "fds_shock_strong_shrink": {
        "features": FAST_DOWNSIDE_CORE + SHOCK_CORE,
        "estimator": "ridge",
        "alpha_grid": [100.0, 300.0, 1000.0, 3000.0],
        "fallback_alpha": 300.0,
        "target_transform": "raw",
    },
    "fds_shock_winsor_1_99": {
        "features": FAST_DOWNSIDE_CORE + SHOCK_CORE,
        "estimator": "ridge",
        "alpha_grid": [1.0, 10.0, 100.0, 300.0, 1000.0],
        "fallback_alpha": 100.0,
        "target_transform": "winsor_1_99",
    },
    "fds_shock_winsor_2p5_97p5": {
        "features": FAST_DOWNSIDE_CORE + SHOCK_CORE,
        "estimator": "ridge",
        "alpha_grid": [1.0, 10.0, 100.0, 300.0, 1000.0],
        "fallback_alpha": 100.0,
        "target_transform": "winsor_2p5_97p5",
    },
    "fds_no_min_return": {
        "features": FAST_DOWNSIDE_CORE + SHOCK_MAX_ABS,
        "estimator": "ridge",
        "alpha_grid": [1.0, 10.0, 100.0, 300.0, 1000.0],
        "fallback_alpha": 100.0,
        "target_transform": "raw",
    },
    "fds_10d_21d_only": {
        "features": FAST_DOWNSIDE_SLOW + ["candidate_max_abs_return_10d", "candidate_min_return_10d"],
        "estimator": "ridge",
        "alpha_grid": [1.0, 10.0, 100.0, 300.0, 1000.0],
        "fallback_alpha": 100.0,
        "target_transform": "raw",
    },
    "fds_plus_rv_slow": {
        "features": FAST_DOWNSIDE_SLOW + FAST_RV_SLOW + ["candidate_max_abs_return_10d", "candidate_min_return_10d"],
        "estimator": "ridge",
        "alpha_grid": [1.0, 10.0, 100.0, 300.0, 1000.0],
        "fallback_alpha": 100.0,
        "target_transform": "raw",
    },
    "reduced_full_state": {
        "features": FAST_RV_SLOW + FAST_DOWNSIDE_SLOW + SHOCK_MAX_ABS + ["candidate_min_return_10d"] + ACCELERATION_REDUCED,
        "estimator": "ridge",
        "alpha_grid": [1.0, 10.0, 100.0, 300.0, 1000.0],
        "fallback_alpha": 100.0,
        "target_transform": "raw",
    },
    "huber_fds_shock": {
        "features": FAST_DOWNSIDE_CORE + SHOCK_CORE,
        "estimator": "huber",
        "alpha_grid": [0.0001, 0.001, 0.01, 0.1],
        "fallback_alpha": 0.001,
        "target_transform": "raw",
    },
}

REVIEW_SPECS = [EXISTING_SPEC] + list(MODEL_SPECS.keys())

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def forbidden_feature_hits(cols: list[str], forbidden_keywords: list[str]) -> pd.DataFrame:
    hits = []
    for col in cols:
        col_lower = col.lower()
        for kw in forbidden_keywords:
            if kw in col_lower:
                hits.append({"feature": col, "forbidden_keyword": kw})
    return pd.DataFrame(hits, columns=["feature", "forbidden_keyword"])


def transform_target_train_only(y_train: pd.Series, transform_name: str) -> tuple[np.ndarray, dict]:
    y = pd.to_numeric(y_train, errors="coerce").astype(float)

    if transform_name == "raw":
        return y.to_numpy(), {"target_transform": "raw"}

    if transform_name == "winsor_1_99":
        lo = float(y.quantile(0.01))
        hi = float(y.quantile(0.99))
        return y.clip(lower=lo, upper=hi).to_numpy(), {
            "target_transform": transform_name,
            "winsor_low": lo,
            "winsor_high": hi,
        }

    if transform_name == "winsor_2p5_97p5":
        lo = float(y.quantile(0.025))
        hi = float(y.quantile(0.975))
        return y.clip(lower=lo, upper=hi).to_numpy(), {
            "target_transform": transform_name,
            "winsor_low": lo,
            "winsor_high": hi,
        }

    raise ValueError(f"Unsupported target_transform: {transform_name}")


def make_estimator(estimator_name: str, alpha: float):
    if estimator_name == "ridge":
        return Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("model", Ridge(alpha=alpha)),
            ]
        )

    if estimator_name == "huber":
        return Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("model", HuberRegressor(alpha=alpha, epsilon=1.35, max_iter=2000)),
            ]
        )

    raise ValueError(f"Unsupported estimator_name: {estimator_name}")


def fit_model_predict(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    features: list[str],
    target_col: str,
    alpha: float,
    estimator_name: str,
    target_transform: str,
) -> tuple[np.ndarray, dict]:
    y_train_transformed, target_info = transform_target_train_only(train_df[target_col], target_transform)

    model = make_estimator(estimator_name, alpha)
    model.fit(train_df[features].to_numpy(), y_train_transformed)
    pred = model.predict(test_df[features].to_numpy())

    fit_info = {
        "estimator": estimator_name,
        "alpha": alpha,
        **target_info,
    }
    return pred, fit_info


def choose_alpha_inner_yearly_cv(
    train_pool: pd.DataFrame,
    features: list[str],
    target_col: str,
    alpha_grid: list[float],
    fallback_alpha: float,
    estimator_name: str,
    target_transform: str,
) -> tuple[float, pd.DataFrame]:
    cv_rows = []

    available_years = sorted(train_pool["trade_date"].dt.year.dropna().astype(int).unique().tolist())

    for val_year in available_years:
        val_start = pd.Timestamp(f"{val_year}-01-01")
        val_end = pd.Timestamp(f"{val_year + 1}-01-01")

        inner_train = train_pool[
            (train_pool["trade_date"] < val_start)
            & (train_pool["last_forward_rv_date"] < val_start)
        ].copy()

        inner_val = train_pool[
            (train_pool["trade_date"] >= val_start)
            & (train_pool["trade_date"] < val_end)
        ].copy()

        inner_train = inner_train.dropna(subset=features + [target_col])
        inner_val = inner_val.dropna(subset=features + [target_col])

        if len(inner_train) < MIN_INNER_TRAIN_ROWS or len(inner_val) < MIN_INNER_VAL_ROWS:
            continue

        for alpha in alpha_grid:
            try:
                pred, fit_info = fit_model_predict(
                    train_df=inner_train,
                    test_df=inner_val,
                    features=features,
                    target_col=target_col,
                    alpha=alpha,
                    estimator_name=estimator_name,
                    target_transform=target_transform,
                )
                cv_rows.append({
                    "val_year": val_year,
                    "alpha": alpha,
                    "estimator": estimator_name,
                    "target_transform": target_transform,
                    "inner_train_rows": int(len(inner_train)),
                    "inner_val_rows": int(len(inner_val)),
                    "rmse_log": float(np.sqrt(mean_squared_error(inner_val[target_col], pred))),
                    "mae_log": float(mean_absolute_error(inner_val[target_col], pred)),
                    "bias_log_pred_minus_actual": float(np.mean(pred - inner_val[target_col].to_numpy())),
                    "winsor_low": fit_info.get("winsor_low", np.nan),
                    "winsor_high": fit_info.get("winsor_high", np.nan),
                })
            except Exception as e:
                cv_rows.append({
                    "val_year": val_year,
                    "alpha": alpha,
                    "estimator": estimator_name,
                    "target_transform": target_transform,
                    "inner_train_rows": int(len(inner_train)),
                    "inner_val_rows": int(len(inner_val)),
                    "rmse_log": np.nan,
                    "mae_log": np.nan,
                    "bias_log_pred_minus_actual": np.nan,
                    "error": f"{type(e).__name__}: {e}",
                })

    cv = pd.DataFrame(cv_rows)

    if cv.empty or cv["rmse_log"].notna().sum() == 0:
        return fallback_alpha, cv

    alpha_summary = (
        cv.dropna(subset=["rmse_log"])
        .groupby("alpha", as_index=False)
        .agg(
            cv_folds=("val_year", "nunique"),
            mean_rmse_log=("rmse_log", "mean"),
            mean_mae_log=("mae_log", "mean"),
            mean_bias_log_pred_minus_actual=("bias_log_pred_minus_actual", "mean"),
        )
        .sort_values(["mean_rmse_log", "mean_mae_log", "alpha"], ascending=[True, True, True])
    )

    if alpha_summary.empty:
        return fallback_alpha, cv

    return float(alpha_summary.iloc[0]["alpha"]), cv


def summarize_forecast_metrics(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    rows = []

    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_cols, keys))

        valid_g = g[
            np.isfinite(g["predicted_log_variance_candidate"])
            & np.isfinite(g[TARGET_LOG_COL])
            & np.isfinite(g["forecast_variance_candidate"])
            & np.isfinite(g[TARGET_VAR_COL])
            & (g["forecast_variance_candidate"] > 0)
            & (g[TARGET_VAR_COL] > 0)
        ].copy()

        row["rows"] = int(len(valid_g))

        if len(valid_g) == 0:
            rows.append(row)
            continue

        log_error = valid_g["predicted_log_variance_candidate"] - valid_g[TARGET_LOG_COL]
        ratio = valid_g[TARGET_VAR_COL] / valid_g["forecast_variance_candidate"]
        actual_top = valid_g[valid_g["actual_top_decile_common_by_tenor"]].copy()

        row["rmse_log"] = float(np.sqrt(mean_squared_error(valid_g[TARGET_LOG_COL], valid_g["predicted_log_variance_candidate"])))
        row["mae_log"] = float(mean_absolute_error(valid_g[TARGET_LOG_COL], valid_g["predicted_log_variance_candidate"]))
        row["bias_log_pred_minus_actual"] = float(log_error.mean())
        row["median_log_error"] = float(log_error.median())
        row["underforecast_rate"] = float((valid_g["forecast_variance_candidate"] < valid_g[TARGET_VAR_COL]).mean())
        row["large_underforecast_1p5x_rate"] = float((ratio > 1.5).mean())
        row["large_underforecast_2p0x_rate"] = float((ratio > 2.0).mean())
        row["actual_top_decile_rows"] = int(len(actual_top))
        row["forecast_top_decile_capture_rate"] = (
            float(actual_top["forecast_top_decile_common_by_model_tenor"].mean()) if len(actual_top) else np.nan
        )
        row["missed_top_decile_actual_rate"] = (
            float((~actual_top["forecast_top_decile_common_by_model_tenor"]).mean()) if len(actual_top) else np.nan
        )
        row["underforecast_rate_in_actual_top_decile"] = (
            float((actual_top["forecast_variance_candidate"] < actual_top[TARGET_VAR_COL]).mean()) if len(actual_top) else np.nan
        )
        row["avg_actual_to_forecast_var_ratio"] = float(ratio.mean())
        row["median_actual_to_forecast_var_ratio"] = float(ratio.median())
        row["target_realized_vol_pct_mean"] = float(valid_g["target_realized_vol_pct"].mean())
        row["candidate_forecast_vol_pct_mean"] = float(valid_g["candidate_forecast_vol_pct"].mean())

        rows.append(row)

    return pd.DataFrame(rows)


def add_vs_existing(metrics: pd.DataFrame, join_cols: list[str]) -> pd.DataFrame:
    baseline = metrics[metrics["model_spec"] == EXISTING_SPEC].copy()

    baseline_keep = join_cols + [
        "rmse_log",
        "mae_log",
        "bias_log_pred_minus_actual",
        "underforecast_rate",
        "large_underforecast_1p5x_rate",
        "large_underforecast_2p0x_rate",
        "forecast_top_decile_capture_rate",
        "missed_top_decile_actual_rate",
        "underforecast_rate_in_actual_top_decile",
        "candidate_forecast_vol_pct_mean",
    ]

    rename_map = {
        c: f"existing_{c}"
        for c in baseline_keep
        if c not in join_cols
    }

    baseline = baseline[baseline_keep].rename(columns=rename_map)
    out = metrics.merge(baseline, on=join_cols, how="left")

    out["capture_improvement_vs_existing"] = (
        out["forecast_top_decile_capture_rate"] - out["existing_forecast_top_decile_capture_rate"]
    )
    out["missed_top_decile_reduction_vs_existing"] = (
        out["existing_missed_top_decile_actual_rate"] - out["missed_top_decile_actual_rate"]
    )
    out["rmse_change_vs_existing"] = out["rmse_log"] - out["existing_rmse_log"]
    out["mae_change_vs_existing"] = out["mae_log"] - out["existing_mae_log"]
    out["large_underforecast_2p0x_change_vs_existing"] = (
        out["large_underforecast_2p0x_rate"] - out["existing_large_underforecast_2p0x_rate"]
    )
    out["forecast_vol_mean_change_vs_existing"] = (
        out["candidate_forecast_vol_pct_mean"] - out["existing_candidate_forecast_vol_pct_mean"]
    )

    return out


def compute_prior_z_by_model_tenor(df: pd.DataFrame, value_col: str, window: int, min_periods: int) -> pd.Series:
    out = pd.Series(np.nan, index=df.index, dtype="float64")

    for _, g in df.sort_values(["model_spec", "tenor", "trade_date"]).groupby(["model_spec", "tenor"]):
        shifted = g[value_col].shift(1)
        roll_mean = shifted.rolling(window=window, min_periods=min_periods).mean()
        roll_std = shifted.rolling(window=window, min_periods=min_periods).std(ddof=1)
        z = (g[value_col] - roll_mean) / roll_std.replace(0.0, np.nan)
        out.loc[g.index] = z

    return out


def max_drawdown_from_pnl(pnl: pd.Series) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)
    cumulative = pnl.cumsum()
    running_max = cumulative.cummax()
    drawdown = cumulative - running_max
    return float(drawdown.min()) if len(drawdown) else np.nan


def worst_rolling_sum(pnl: pd.Series, window: int) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)
    if len(pnl) == 0:
        return np.nan
    if len(pnl) < window:
        return float(pnl.sum())
    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def summarize_trade_set(df: pd.DataFrame, universe_label: str) -> pd.DataFrame:
    rows = []

    if df.empty:
        return pd.DataFrame(columns=[
            "selection_universe",
            "model_spec",
            "trades",
            "date_min",
            "date_max",
            "win_rate",
            "total_pnl",
            "avg_pnl",
            "median_pnl",
            "worst_trade",
            "best_trade",
            "profit_factor",
            "selected_trade_drawdown",
            "worst_20_trade_sum",
            "trades_le_neg_100k",
            "pnl_2025",
            "trades_2025",
            "worst_trade_2025",
        ])

    d = df.copy()
    d["normalized_pnl_dollars"] = pd.to_numeric(d["normalized_pnl_dollars"], errors="coerce")
    d["win"] = pd.to_numeric(d["win"], errors="coerce")
    d = d[d["normalized_pnl_dollars"].notna()].copy()
    d = d.sort_values(["model_spec", "trade_date", "tenor"])

    for model_spec, g in d.groupby("model_spec", dropna=False):
        pnl = g["normalized_pnl_dollars"]
        wins = g["win"]

        gross_profit = float(pnl[pnl > 0].sum())
        gross_loss = float(pnl[pnl < 0].sum())
        profit_factor = gross_profit / abs(gross_loss) if gross_loss < 0 else np.inf

        y2025 = g[g["trade_date"].dt.year == 2025].copy()

        rows.append({
            "selection_universe": universe_label,
            "model_spec": model_spec,
            "trades": int(len(g)),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,
            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor,
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "trades_2025": int(len(y2025)),
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,
        })

    return pd.DataFrame(rows)


def compare_trade_summary_to_existing(summary: pd.DataFrame) -> pd.DataFrame:
    if summary.empty or EXISTING_SPEC not in set(summary["model_spec"]):
        return summary.copy()

    out_rows = []

    for universe, g in summary.groupby("selection_universe", dropna=False):
        base_rows = g[g["model_spec"] == EXISTING_SPEC]
        if base_rows.empty:
            continue

        base = base_rows.iloc[0]

        for _, row in g.iterrows():
            r = row.to_dict()

            r["trades_change_vs_existing"] = row["trades"] - base["trades"]
            r["win_rate_change_vs_existing"] = row["win_rate"] - base["win_rate"]
            r["total_pnl_change_vs_existing"] = row["total_pnl"] - base["total_pnl"]
            r["profit_factor_change_vs_existing"] = row["profit_factor"] - base["profit_factor"]
            r["worst_trade_change_vs_existing"] = row["worst_trade"] - base["worst_trade"]
            r["drawdown_change_vs_existing"] = row["selected_trade_drawdown"] - base["selected_trade_drawdown"]
            r["trades_le_neg_100k_change_vs_existing"] = row["trades_le_neg_100k"] - base["trades_le_neg_100k"]
            r["pnl_2025_change_vs_existing"] = row["pnl_2025"] - base["pnl_2025"]

            r["total_pnl_change_frac_vs_existing"] = (
                r["total_pnl_change_vs_existing"] / abs(base["total_pnl"])
                if abs(base["total_pnl"]) > 0 else np.nan
            )
            r["pnl_2025_change_frac_vs_existing"] = (
                r["pnl_2025_change_vs_existing"] / abs(base["pnl_2025"])
                if abs(base["pnl_2025"]) > 0 else np.nan
            )

            if row["model_spec"] == EXISTING_SPEC:
                r["preservation_screen"] = "BASELINE_REFERENCE"
            else:
                if universe == "thirty_day_anchor_only":
                    pass_screen = (
                        (r["total_pnl_change_frac_vs_existing"] >= -ACCEPTANCE_GUIDE["thirty_day_total_pnl_max_giveup_frac"])
                        and (row["profit_factor"] >= ACCEPTANCE_GUIDE["thirty_day_min_profit_factor"])
                        and (r["win_rate_change_vs_existing"] >= ACCEPTANCE_GUIDE["thirty_day_min_win_rate_delta"])
                        and (row["worst_trade"] >= ACCEPTANCE_GUIDE["thirty_day_min_worst_trade"])
                        and (row["selected_trade_drawdown"] >= ACCEPTANCE_GUIDE["thirty_day_min_drawdown"])
                        and (row["trades_le_neg_100k"] <= ACCEPTANCE_GUIDE["thirty_day_max_trades_le_neg_100k"])
                        and (row["pnl_2025"] >= base["pnl_2025"] * ACCEPTANCE_GUIDE["thirty_day_min_2025_pnl_frac"])
                    )
                elif universe == "one_back_trade_per_date_best_vrp":
                    pass_screen = (
                        (r["total_pnl_change_frac_vs_existing"] >= -ACCEPTANCE_GUIDE["bucket_selected_total_pnl_max_giveup_frac"])
                        and (row["profit_factor"] >= base["profit_factor"])
                        and (row["worst_trade"] >= base["worst_trade"])
                        and (row["selected_trade_drawdown"] >= base["selected_trade_drawdown"])
                        and (row["trades_le_neg_100k"] <= ACCEPTANCE_GUIDE["bucket_selected_max_trades_le_neg_100k"])
                    )
                else:
                    pass_screen = (
                        (row["profit_factor"] >= base["profit_factor"])
                        and (row["worst_trade"] >= base["worst_trade"])
                        and (row["selected_trade_drawdown"] >= base["selected_trade_drawdown"])
                        and (row["trades_le_neg_100k"] <= base["trades_le_neg_100k"])
                        and (r["total_pnl_change_frac_vs_existing"] >= -0.10)
                    )

                r["preservation_screen"] = "PASS" if pass_screen else "FAIL"

            out_rows.append(r)

    return pd.DataFrame(out_rows)


def classify_unfreeze_candidates(trade_vs_existing: pd.DataFrame, forecast_vs_existing: pd.DataFrame) -> pd.DataFrame:
    rows = []

    if trade_vs_existing.empty:
        return pd.DataFrame()

    forecast_key = (
        forecast_vs_existing[["model_spec", "rmse_change_vs_existing", "capture_improvement_vs_existing"]]
        .drop_duplicates("model_spec")
        .set_index("model_spec")
        .to_dict("index")
    )

    candidate_specs = [s for s in sorted(trade_vs_existing["model_spec"].unique()) if s != EXISTING_SPEC]

    for spec in candidate_specs:
        spec_rows = trade_vs_existing[trade_vs_existing["model_spec"] == spec].copy()

        thirty = spec_rows[spec_rows["selection_universe"] == "thirty_day_anchor_only"]
        bucket = spec_rows[spec_rows["selection_universe"] == "one_back_trade_per_date_best_vrp"]
        allq = spec_rows[spec_rows["selection_universe"] == "all_qualified_back_rows"]

        thirty_status = thirty["preservation_screen"].iloc[0] if len(thirty) else "MISSING"
        bucket_status = bucket["preservation_screen"].iloc[0] if len(bucket) else "MISSING"
        allq_status = allq["preservation_screen"].iloc[0] if len(allq) else "MISSING"

        if thirty_status == "PASS" and bucket_status == "PASS":
            recommendation = "ACCEPTABLE_UNFREEZE"
        elif thirty_status == "PASS" or (bucket_status == "PASS" and allq_status == "PASS"):
            recommendation = "BORDERLINE"
        else:
            recommendation = "REJECT"

        f = forecast_key.get(spec, {})

        rows.append({
            "model_spec": spec,
            "unfreeze_classification": recommendation,
            "thirty_day_screen": thirty_status,
            "bucket_selected_screen": bucket_status,
            "all_qualified_screen": allq_status,
            "back_rmse_change_vs_existing": f.get("rmse_change_vs_existing", np.nan),
            "back_capture_improvement_vs_existing": f.get("capture_improvement_vs_existing", np.nan),
            "thirty_day_total_pnl_change": (
                float(thirty["total_pnl_change_vs_existing"].iloc[0]) if len(thirty) else np.nan
            ),
            "thirty_day_total_pnl_change_frac": (
                float(thirty["total_pnl_change_frac_vs_existing"].iloc[0]) if len(thirty) else np.nan
            ),
            "thirty_day_profit_factor": (
                float(thirty["profit_factor"].iloc[0]) if len(thirty) else np.nan
            ),
            "thirty_day_win_rate": (
                float(thirty["win_rate"].iloc[0]) if len(thirty) else np.nan
            ),
            "thirty_day_worst_trade": (
                float(thirty["worst_trade"].iloc[0]) if len(thirty) else np.nan
            ),
            "thirty_day_drawdown": (
                float(thirty["selected_trade_drawdown"].iloc[0]) if len(thirty) else np.nan
            ),
            "thirty_day_trades_le_neg_100k": (
                int(thirty["trades_le_neg_100k"].iloc[0]) if len(thirty) else np.nan
            ),
            "thirty_day_pnl_2025": (
                float(thirty["pnl_2025"].iloc[0]) if len(thirty) else np.nan
            ),
            "bucket_selected_total_pnl_change": (
                float(bucket["total_pnl_change_vs_existing"].iloc[0]) if len(bucket) else np.nan
            ),
            "bucket_selected_profit_factor": (
                float(bucket["profit_factor"].iloc[0]) if len(bucket) else np.nan
            ),
            "bucket_selected_trades_le_neg_100k": (
                int(bucket["trades_le_neg_100k"].iloc[0]) if len(bucket) else np.nan
            ),
        })

    out = pd.DataFrame(rows)
    class_order = {"ACCEPTABLE_UNFREEZE": 0, "BORDERLINE": 1, "REJECT": 2}
    out["sort_key"] = out["unfreeze_classification"].map(class_order).fillna(99)
    out = out.sort_values(
        ["sort_key", "thirty_day_total_pnl_change", "bucket_selected_total_pnl_change"],
        ascending=[True, False, False],
    ).drop(columns=["sort_key"])
    return out


# ======================================================================================
# 2. Load latest Cell 4 feature panel
# ======================================================================================

cell4_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "04_front_middle_candidate_feature_panel_*.parquet",
    required=True,
)

panel = pd.read_parquet(cell4_panel_path)

duplicate_cols = panel.columns[panel.columns.duplicated()].tolist()
if duplicate_cols:
    print("Dropping duplicate column labels in Cell 6C source panel:")
    print(sorted(set(duplicate_cols)))
panel = panel.loc[:, ~panel.columns.duplicated(keep="last")].copy()

panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce").dt.normalize()
panel["last_forward_rv_date"] = pd.to_datetime(panel["last_forward_rv_date"], errors="coerce").dt.normalize()
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype("Int64")

required_cols = [
    "trade_date",
    "last_forward_rv_date",
    "tenor",
    "tenor_bucket",
    TARGET_LOG_COL,
    TARGET_VAR_COL,
    FORECAST_VAR_BASELINE_COL,
    "implied_variance",
    "RSI14",
    "win",
    "normalized_pnl_dollars",
]

missing_required = [c for c in required_cols if c not in panel.columns]
if missing_required:
    raise KeyError(f"Cell 6C missing required columns from Cell 4 panel: {missing_required}")

back_panel = panel[panel["tenor"].isin(BACK_TENORS)].copy()

for c in [TARGET_LOG_COL, TARGET_VAR_COL, FORECAST_VAR_BASELINE_COL, "implied_variance", "RSI14"]:
    back_panel[c] = pd.to_numeric(back_panel[c], errors="coerce")

back_panel = back_panel[
    back_panel["trade_date"].notna()
    & back_panel["last_forward_rv_date"].notna()
    & np.isfinite(back_panel[TARGET_LOG_COL])
    & np.isfinite(back_panel[TARGET_VAR_COL])
    & (back_panel[TARGET_VAR_COL] > 0)
    & np.isfinite(back_panel["implied_variance"])
    & (back_panel["implied_variance"] > 0)
].copy()

if back_panel.empty:
    raise RuntimeError("No valid Back rows found for Cell 6C.")

# Validate model specs.
feature_validation_rows = []
usable_model_specs = {}

for spec_name, spec in MODEL_SPECS.items():
    features = spec["features"]
    missing = [c for c in features if c not in back_panel.columns]
    forbidden_hits = forbidden_feature_hits(features, FORBIDDEN_MODEL_FEATURE_KEYWORDS)
    status = "PASS" if not missing and forbidden_hits.empty else "FAIL"

    feature_validation_rows.append({
        "model_spec": spec_name,
        "status": status,
        "feature_count": len(features),
        "estimator": spec["estimator"],
        "target_transform": spec["target_transform"],
        "alpha_grid": spec["alpha_grid"],
        "missing_features": missing,
        "forbidden_hits": forbidden_hits.to_dict("records") if not forbidden_hits.empty else [],
        "features": features,
    })

    if status == "PASS":
        usable_model_specs[spec_name] = spec

feature_validation = pd.DataFrame(feature_validation_rows)

if feature_validation["status"].eq("FAIL").any():
    display(feature_validation)
    raise RuntimeError("One or more Back model specs failed feature validation.")

# ======================================================================================
# 3. Existing Back baseline rows
# ======================================================================================

baseline = back_panel.copy()
baseline["model_spec"] = EXISTING_SPEC
baseline["model_source"] = "existing_locked_oos_forecast"
baseline["estimator"] = "existing"
baseline["target_transform"] = "existing"
baseline["selected_alpha_candidate"] = pd.to_numeric(baseline.get("selected_alpha", np.nan), errors="coerce")
baseline["train_rows_used_candidate"] = pd.to_numeric(baseline.get("train_rows_used", np.nan), errors="coerce")
baseline["test_year_candidate"] = baseline["trade_date"].dt.year.astype("Int64")
baseline["fit_status_candidate"] = "existing_forecast"
baseline["winsor_low_candidate"] = np.nan
baseline["winsor_high_candidate"] = np.nan

if PRED_LOG_BASELINE_COL in baseline.columns:
    baseline["predicted_log_variance_candidate"] = pd.to_numeric(baseline[PRED_LOG_BASELINE_COL], errors="coerce")
else:
    baseline["predicted_log_variance_candidate"] = np.log(baseline[FORECAST_VAR_BASELINE_COL].clip(lower=EPS))

baseline["forecast_variance_candidate"] = baseline[FORECAST_VAR_BASELINE_COL]
baseline["candidate_forecast_vol_pct"] = np.sqrt(baseline["forecast_variance_candidate"].clip(lower=0)) * 100.0
baseline["target_realized_vol_pct"] = np.sqrt(baseline[TARGET_VAR_COL].clip(lower=0)) * 100.0

# ======================================================================================
# 4. Fit refined pure Back model specs OOS
# ======================================================================================

forecast_rows = []
fit_log_rows = []
inner_cv_rows_all = []

for model_spec_name, spec in usable_model_specs.items():
    features = spec["features"]
    estimator_name = spec["estimator"]
    alpha_grid = spec["alpha_grid"]
    fallback_alpha = spec["fallback_alpha"]
    target_transform = spec["target_transform"]

    for tenor in BACK_TENORS:
        tenor_df = back_panel[back_panel["tenor"] == tenor].copy()
        tenor_df = tenor_df.sort_values("trade_date").reset_index(drop=True)

        test_years = sorted(tenor_df["trade_date"].dt.year.dropna().astype(int).unique().tolist())

        for test_year in test_years:
            test_start = pd.Timestamp(f"{test_year}-01-01")
            test_end = pd.Timestamp(f"{test_year + 1}-01-01")

            train_pool = tenor_df[
                (tenor_df["trade_date"] < test_start)
                & (tenor_df["last_forward_rv_date"] < test_start)
            ].copy()

            test_rows = tenor_df[
                (tenor_df["trade_date"] >= test_start)
                & (tenor_df["trade_date"] < test_end)
            ].copy()

            train_fit = train_pool.dropna(subset=features + [TARGET_LOG_COL]).copy()
            test_fit = test_rows.dropna(subset=features + [TARGET_LOG_COL]).copy()

            if len(test_fit) == 0:
                fit_log_rows.append({
                    "model_spec": model_spec_name,
                    "tenor": tenor,
                    "test_year": test_year,
                    "fit_status": "skipped_no_test_rows",
                    "selected_alpha": np.nan,
                    "train_rows_used": int(len(train_fit)),
                    "test_rows_scored": 0,
                    "feature_count": len(features),
                    "estimator": estimator_name,
                    "target_transform": target_transform,
                })
                continue

            if len(train_fit) < MIN_OUTER_TRAIN_ROWS:
                fit_log_rows.append({
                    "model_spec": model_spec_name,
                    "tenor": tenor,
                    "test_year": test_year,
                    "fit_status": "skipped_insufficient_train_rows",
                    "selected_alpha": np.nan,
                    "train_rows_used": int(len(train_fit)),
                    "test_rows_scored": 0,
                    "feature_count": len(features),
                    "estimator": estimator_name,
                    "target_transform": target_transform,
                })
                continue

            selected_alpha, inner_cv = choose_alpha_inner_yearly_cv(
                train_pool=train_fit,
                features=features,
                target_col=TARGET_LOG_COL,
                alpha_grid=alpha_grid,
                fallback_alpha=fallback_alpha,
                estimator_name=estimator_name,
                target_transform=target_transform,
            )

            if not inner_cv.empty:
                inner_cv["outer_model_spec"] = model_spec_name
                inner_cv["outer_tenor"] = tenor
                inner_cv["outer_test_year"] = test_year
                inner_cv_rows_all.append(inner_cv)

            try:
                pred, fit_info = fit_model_predict(
                    train_df=train_fit,
                    test_df=test_fit,
                    features=features,
                    target_col=TARGET_LOG_COL,
                    alpha=selected_alpha,
                    estimator_name=estimator_name,
                    target_transform=target_transform,
                )

                scored = test_fit.copy()
                scored["model_spec"] = model_spec_name
                scored["model_source"] = "pure_back_unfreeze_model_spec_oos_refit"
                scored["estimator"] = estimator_name
                scored["target_transform"] = target_transform
                scored["selected_alpha_candidate"] = selected_alpha
                scored["train_rows_used_candidate"] = int(len(train_fit))
                scored["test_year_candidate"] = int(test_year)
                scored["fit_status_candidate"] = "candidate_fit"
                scored["winsor_low_candidate"] = fit_info.get("winsor_low", np.nan)
                scored["winsor_high_candidate"] = fit_info.get("winsor_high", np.nan)
                scored["predicted_log_variance_candidate"] = pred
                scored["forecast_variance_candidate"] = np.exp(pred)
                scored["candidate_forecast_vol_pct"] = np.sqrt(scored["forecast_variance_candidate"].clip(lower=0)) * 100.0
                scored["target_realized_vol_pct"] = np.sqrt(scored[TARGET_VAR_COL].clip(lower=0)) * 100.0

                forecast_rows.append(scored)

                fit_log_rows.append({
                    "model_spec": model_spec_name,
                    "tenor": tenor,
                    "test_year": test_year,
                    "fit_status": "candidate_fit",
                    "selected_alpha": selected_alpha,
                    "train_rows_used": int(len(train_fit)),
                    "test_rows_scored": int(len(scored)),
                    "feature_count": len(features),
                    "estimator": estimator_name,
                    "target_transform": target_transform,
                    "winsor_low": fit_info.get("winsor_low", np.nan),
                    "winsor_high": fit_info.get("winsor_high", np.nan),
                })

            except Exception as e:
                fit_log_rows.append({
                    "model_spec": model_spec_name,
                    "tenor": tenor,
                    "test_year": test_year,
                    "fit_status": "fit_error",
                    "selected_alpha": selected_alpha,
                    "train_rows_used": int(len(train_fit)),
                    "test_rows_scored": 0,
                    "feature_count": len(features),
                    "estimator": estimator_name,
                    "target_transform": target_transform,
                    "error": f"{type(e).__name__}: {e}",
                })

candidate_forecasts = pd.concat(forecast_rows, ignore_index=True) if forecast_rows else pd.DataFrame()
fit_log = pd.DataFrame(fit_log_rows)
inner_cv_all = pd.concat(inner_cv_rows_all, ignore_index=True) if inner_cv_rows_all else pd.DataFrame()

if candidate_forecasts.empty:
    display(fit_log)
    raise RuntimeError("No refined Back candidate forecasts were produced. See fit_log.")

# Align columns.
for c in baseline.columns:
    if c not in candidate_forecasts.columns:
        candidate_forecasts[c] = np.nan

candidate_forecasts = candidate_forecasts[baseline.columns].copy()
all_forecasts = pd.concat([baseline, candidate_forecasts], ignore_index=True)
all_forecasts = all_forecasts.loc[:, ~all_forecasts.columns.duplicated(keep="last")].copy()

# Core forecast diagnostics.
for c in [
    "forecast_variance_candidate",
    "predicted_log_variance_candidate",
    TARGET_VAR_COL,
    TARGET_LOG_COL,
    "implied_variance",
    "RSI14",
]:
    all_forecasts[c] = pd.to_numeric(all_forecasts[c], errors="coerce")

all_forecasts = all_forecasts[
    np.isfinite(all_forecasts["forecast_variance_candidate"])
    & np.isfinite(all_forecasts["predicted_log_variance_candidate"])
    & np.isfinite(all_forecasts[TARGET_VAR_COL])
    & np.isfinite(all_forecasts[TARGET_LOG_COL])
    & np.isfinite(all_forecasts["implied_variance"])
    & (all_forecasts["forecast_variance_candidate"] > 0)
    & (all_forecasts[TARGET_VAR_COL] > 0)
    & (all_forecasts["implied_variance"] > 0)
].copy()

all_forecasts["target_realized_vol_pct"] = np.sqrt(all_forecasts[TARGET_VAR_COL].clip(lower=0)) * 100.0
all_forecasts["candidate_forecast_vol_pct"] = np.sqrt(all_forecasts["forecast_variance_candidate"].clip(lower=0)) * 100.0
all_forecasts["log_error_pred_minus_actual"] = all_forecasts["predicted_log_variance_candidate"] - all_forecasts[TARGET_LOG_COL]
all_forecasts["actual_to_forecast_var_ratio"] = all_forecasts[TARGET_VAR_COL] / all_forecasts["forecast_variance_candidate"]

# ======================================================================================
# 5. Common Back row forecast diagnostics
# ======================================================================================

review = all_forecasts[
    all_forecasts["model_spec"].isin(REVIEW_SPECS)
    & all_forecasts["tenor"].isin(BACK_TENORS)
].copy()

row_model_counts = (
    review.groupby(["trade_date", "tenor"], as_index=False)
    .agg(model_count=("model_spec", "nunique"))
)

common_keys = row_model_counts[row_model_counts["model_count"] == len(REVIEW_SPECS)][["trade_date", "tenor"]].copy()

if common_keys.empty:
    raise RuntimeError("No common Back date × tenor rows found across all review specs.")

common = review.merge(common_keys, on=["trade_date", "tenor"], how="inner", validate="many_to_one").copy()

actual_common_rank_source = (
    common[["trade_date", "tenor", TARGET_VAR_COL]]
    .drop_duplicates(["trade_date", "tenor"])
    .copy()
)

actual_common_rank_source["actual_var_pct_rank_common_by_tenor"] = (
    actual_common_rank_source.groupby("tenor")[TARGET_VAR_COL].rank(pct=True)
)
actual_common_rank_source["actual_top_decile_common_by_tenor"] = (
    actual_common_rank_source["actual_var_pct_rank_common_by_tenor"] >= 0.90
)

common = common.merge(
    actual_common_rank_source[
        ["trade_date", "tenor", "actual_var_pct_rank_common_by_tenor", "actual_top_decile_common_by_tenor"]
    ],
    on=["trade_date", "tenor"],
    how="left",
    validate="many_to_one",
)

common["forecast_var_pct_rank_common_by_model_tenor"] = (
    common.groupby(["model_spec", "tenor"])["forecast_variance_candidate"].rank(pct=True)
)
common["forecast_top_decile_common_by_model_tenor"] = (
    common["forecast_var_pct_rank_common_by_model_tenor"] >= 0.90
)

forecast_metrics_by_bucket = summarize_forecast_metrics(
    common,
    ["model_spec", "tenor_bucket"],
)

forecast_metrics_by_tenor = summarize_forecast_metrics(
    common,
    ["model_spec", "tenor", "tenor_bucket"],
)

forecast_metrics_by_year = summarize_forecast_metrics(
    common.assign(year=common["trade_date"].dt.year.astype("Int64")),
    ["model_spec", "year", "tenor_bucket"],
)

forecast_metrics_by_bucket_vs_existing = add_vs_existing(
    forecast_metrics_by_bucket,
    ["tenor_bucket"],
)

forecast_metrics_by_tenor_vs_existing = add_vs_existing(
    forecast_metrics_by_tenor,
    ["tenor", "tenor_bucket"],
)

# ======================================================================================
# 6. Rebuild Back VRP and fixed-threshold signal diagnostics
# ======================================================================================

signal_base = all_forecasts[
    all_forecasts["model_spec"].isin(REVIEW_SPECS)
    & all_forecasts["tenor"].isin(BACK_TENORS)
].copy()

signal_base = signal_base.sort_values(["model_spec", "tenor", "trade_date"]).reset_index(drop=True)

signal_base["model_vrp_log_candidate"] = np.log(
    signal_base["implied_variance"] / signal_base["forecast_variance_candidate"]
)

signal_base["model_vrp_z_3m_candidate"] = compute_prior_z_by_model_tenor(
    signal_base,
    value_col="model_vrp_log_candidate",
    window=63,
    min_periods=63,
)

signal_base["model_vrp_z_1y_candidate"] = compute_prior_z_by_model_tenor(
    signal_base,
    value_col="model_vrp_log_candidate",
    window=252,
    min_periods=252,
)

signal_base["locked_core_back_pass"] = (
    (signal_base["model_vrp_log_candidate"] > LOCKED_BACK_THRESHOLDS["model_vrp_log"])
    & (signal_base["model_vrp_z_3m_candidate"] > LOCKED_BACK_THRESHOLDS["model_vrp_z_3m"])
    & (signal_base["model_vrp_z_1y_candidate"] > LOCKED_BACK_THRESHOLDS["model_vrp_z_1y"])
    & (signal_base["RSI14"] < LOCKED_BACK_THRESHOLDS["RSI14_max"])
    & (signal_base["candidate_forecast_vol_pct"] > LOCKED_BACK_THRESHOLDS["forecast_vol_pct_min"])
)

common_signal = signal_base.merge(
    common_keys,
    on=["trade_date", "tenor"],
    how="inner",
    validate="many_to_one",
).copy()

common_signal["signal_inputs_available"] = (
    np.isfinite(common_signal["model_vrp_log_candidate"])
    & np.isfinite(common_signal["model_vrp_z_3m_candidate"])
    & np.isfinite(common_signal["model_vrp_z_1y_candidate"])
    & np.isfinite(common_signal["RSI14"])
    & np.isfinite(common_signal["candidate_forecast_vol_pct"])
    & common_signal["normalized_pnl_dollars"].notna()
)

qualified_all_rows = common_signal[
    common_signal["locked_core_back_pass"]
    & common_signal["signal_inputs_available"]
].copy()

qualified_30d_anchor = qualified_all_rows[qualified_all_rows["tenor"] == 30].copy()

qualified_bucket_selected = (
    qualified_all_rows
    .sort_values(
        ["model_spec", "trade_date", "model_vrp_z_1y_candidate", "model_vrp_z_3m_candidate", "model_vrp_log_candidate", "tenor"],
        ascending=[True, True, False, False, False, False],
    )
    .groupby(["model_spec", "trade_date"], as_index=False)
    .head(1)
    .copy()
)

trade_summary_all_qualified = summarize_trade_set(qualified_all_rows, "all_qualified_back_rows")
trade_summary_30d_anchor = summarize_trade_set(qualified_30d_anchor, "thirty_day_anchor_only")
trade_summary_bucket_selected = summarize_trade_set(qualified_bucket_selected, "one_back_trade_per_date_best_vrp")

trade_summary = pd.concat(
    [trade_summary_all_qualified, trade_summary_30d_anchor, trade_summary_bucket_selected],
    ignore_index=True,
)

trade_summary_vs_existing = compare_trade_summary_to_existing(trade_summary)

signal_pass_diagnostics = (
    common_signal
    .groupby(["model_spec", "tenor"], as_index=False)
    .agg(
        rows=("trade_date", "size"),
        signal_inputs_available_rows=("signal_inputs_available", "sum"),
        locked_core_back_pass_rows=("locked_core_back_pass", "sum"),
        pass_rate=("locked_core_back_pass", "mean"),
        avg_model_vrp_log=("model_vrp_log_candidate", "mean"),
        avg_z_3m=("model_vrp_z_3m_candidate", "mean"),
        avg_z_1y=("model_vrp_z_1y_candidate", "mean"),
        avg_forecast_vol_pct=("candidate_forecast_vol_pct", "mean"),
    )
)

unfreeze_recommendation = classify_unfreeze_candidates(
    trade_vs_existing=trade_summary_vs_existing,
    forecast_vs_existing=forecast_metrics_by_bucket_vs_existing,
)

# ======================================================================================
# 7. Validation
# ======================================================================================

validation_rows = []

def add_validation(check: str, status: str, detail: str):
    validation_rows.append({"check": check, "status": status, "detail": detail})

front_middle_candidate_rows = candidate_forecasts[
    candidate_forecasts["tenor"].isin(FRONT_TENORS + MIDDLE_TENORS)
]

bad_model_count_rows = int(
    (common.groupby(["trade_date", "tenor"])["model_spec"].nunique() != len(REVIEW_SPECS)).sum()
)

add_validation("cell4_panel_loaded", "PASS", f"path={cell4_panel_path}; rows={len(panel):,}")
add_validation("back_panel_non_empty", "PASS" if len(back_panel) > 0 else "FAIL", f"back rows={len(back_panel):,}")
add_validation("feature_validation", "PASS", "all pure Back model specs passed feature availability and forbidden-keyword audit")
add_validation("candidate_back_forecasts_produced", "PASS", f"candidate rows={len(candidate_forecasts):,}")
add_validation(
    "no_front_middle_candidate_refit_rows",
    "PASS" if front_middle_candidate_rows.empty else "FAIL",
    f"front/middle candidate rows={len(front_middle_candidate_rows):,}",
)
add_validation("common_back_keys_non_empty", "PASS", f"common date×tenor keys={len(common_keys):,}")
add_validation(
    "common_back_model_count",
    "PASS" if bad_model_count_rows == 0 else "FAIL",
    f"bad_model_count_rows={bad_model_count_rows}",
)
add_validation("fit_errors", "PASS" if not fit_log["fit_status"].eq("fit_error").any() else "WARN", f"fit_error_rows={int(fit_log['fit_status'].eq('fit_error').sum())}")
add_validation("no_blending", "PASS", "No old/new forecast blending performed")
add_validation("no_threshold_grid", "PASS", "Locked Back thresholds applied unchanged")
add_validation("locked_thresholds_unchanged", "PASS", str(LOCKED_BACK_THRESHOLDS))
add_validation("no_final_back_unfreeze_lock", "PASS", "Cell 6C is diagnostic only")

validation = pd.DataFrame(validation_rows)
failures = validation[validation["status"] == "FAIL"]

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 6C validation failed. See validation table above.")

# ======================================================================================
# 8. Save outputs
# ======================================================================================

date_min = common["trade_date"].min()
date_max = common["trade_date"].max()

forecast_panel_path = OUT_PROCESSED_DIR / (
    f"06C_back_model_spec_oos_forecast_panel_{date_min:%Y%m%d}_{date_max:%Y%m%d}_{TIMESTAMP}.parquet"
)
common_panel_path = OUT_PROCESSED_DIR / (
    f"06C_back_model_spec_common_row_panel_{date_min:%Y%m%d}_{date_max:%Y%m%d}_{TIMESTAMP}.parquet"
)
signal_panel_path = OUT_PROCESSED_DIR / (
    f"06C_back_model_spec_signal_threshold_panel_{date_min:%Y%m%d}_{date_max:%Y%m%d}_{TIMESTAMP}.parquet"
)

forecast_metrics_by_bucket_path = OUT_AUDIT_DIR / f"06C_back_forecast_metrics_by_bucket_{TIMESTAMP}.csv"
forecast_metrics_by_tenor_path = OUT_AUDIT_DIR / f"06C_back_forecast_metrics_by_tenor_{TIMESTAMP}.csv"
forecast_metrics_by_year_path = OUT_AUDIT_DIR / f"06C_back_forecast_metrics_by_year_{TIMESTAMP}.csv"
forecast_metrics_by_bucket_vs_existing_path = OUT_AUDIT_DIR / f"06C_back_forecast_metrics_by_bucket_vs_existing_{TIMESTAMP}.csv"
forecast_metrics_by_tenor_vs_existing_path = OUT_AUDIT_DIR / f"06C_back_forecast_metrics_by_tenor_vs_existing_{TIMESTAMP}.csv"
trade_summary_path = OUT_AUDIT_DIR / f"06C_back_locked_threshold_trade_summary_{TIMESTAMP}.csv"
trade_summary_vs_existing_path = OUT_AUDIT_DIR / f"06C_back_locked_threshold_trade_summary_vs_existing_{TIMESTAMP}.csv"
signal_pass_diagnostics_path = OUT_AUDIT_DIR / f"06C_back_signal_pass_diagnostics_{TIMESTAMP}.csv"
unfreeze_recommendation_path = OUT_AUDIT_DIR / f"06C_back_unfreeze_model_spec_recommendation_{TIMESTAMP}.csv"
fit_log_path = OUT_AUDIT_DIR / f"06C_back_model_spec_fit_log_{TIMESTAMP}.csv"
inner_cv_path = OUT_AUDIT_DIR / f"06C_back_model_spec_inner_cv_{TIMESTAMP}.csv"
feature_validation_path = OUT_AUDIT_DIR / f"06C_back_model_spec_feature_validation_{TIMESTAMP}.csv"
validation_path = OUT_AUDIT_DIR / f"06C_validation_{TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"06C_manifest_{TIMESTAMP}.json"

all_forecasts.to_parquet(forecast_panel_path, index=False)
common.to_parquet(common_panel_path, index=False)
common_signal.to_parquet(signal_panel_path, index=False)

forecast_metrics_by_bucket.to_csv(forecast_metrics_by_bucket_path, index=False)
forecast_metrics_by_tenor.to_csv(forecast_metrics_by_tenor_path, index=False)
forecast_metrics_by_year.to_csv(forecast_metrics_by_year_path, index=False)
forecast_metrics_by_bucket_vs_existing.to_csv(forecast_metrics_by_bucket_vs_existing_path, index=False)
forecast_metrics_by_tenor_vs_existing.to_csv(forecast_metrics_by_tenor_vs_existing_path, index=False)
trade_summary.to_csv(trade_summary_path, index=False)
trade_summary_vs_existing.to_csv(trade_summary_vs_existing_path, index=False)
signal_pass_diagnostics.to_csv(signal_pass_diagnostics_path, index=False)
unfreeze_recommendation.to_csv(unfreeze_recommendation_path, index=False)
fit_log.to_csv(fit_log_path, index=False)
feature_validation.to_csv(feature_validation_path, index=False)
validation.to_csv(validation_path, index=False)

if not inner_cv_all.empty:
    inner_cv_all.to_csv(inner_cv_path, index=False)
else:
    pd.DataFrame().to_csv(inner_cv_path, index=False)

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "cell": "Cell 6C — Pure Back unfreeze model-spec refinement",
            "timestamp": TIMESTAMP,
            "source_cell4_panel": str(cell4_panel_path),
            "existing_spec": EXISTING_SPEC,
            "review_specs": REVIEW_SPECS,
            "model_specs": MODEL_SPECS,
            "back_tenors_refit_for_diagnostic": BACK_TENORS,
            "front_middle_tenors_not_refit": FRONT_TENORS + MIDDLE_TENORS,
            "locked_back_thresholds": LOCKED_BACK_THRESHOLDS,
            "acceptance_guide": ACCEPTANCE_GUIDE,
            "notes": [
                "Diagnostic only; does not unfreeze Back in production/research decision.",
                "No blending performed.",
                "No threshold optimization or threshold grid performed.",
                "Back thresholds are unchanged from locked Core Back.",
                "No Front/Middle candidate refit performed.",
                "Signal summaries use locked Core Back thresholds.",
                "one_back_trade_per_date_best_vrp is a diagnostic one-per-date selector, not a locked production rule.",
            ],
        },
        f,
        indent=2,
    )

# ======================================================================================
# 9. Display concise review
# ======================================================================================

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 320)

print("=" * 100)
print("Cell 6C — Pure Back unfreeze model-spec refinement")
print("=" * 100)
print(f"Source Cell 4 panel:        {cell4_panel_path}")
print(f"Back tenors refit:          {BACK_TENORS}")
print(f"Front/Middle refit rows:    {len(front_middle_candidate_rows):,}")
print(f"Candidate Back rows:        {len(candidate_forecasts):,}")
print(f"Existing baseline rows:     {len(baseline):,}")
print(f"Common Back date×tenor:     {len(common_keys):,}")
print(f"Common Back forecast rows:  {len(common):,}")
print(f"Date range:                 {date_min.date()} to {date_max.date()}")
print(f"Locked thresholds:          {LOCKED_BACK_THRESHOLDS}")
print("No blending:                True")
print("No threshold grid:          True")

print()
print("Validation:")
display(validation)

print()
print("Fit status counts:")
display(fit_log["fit_status"].value_counts(dropna=False).to_frame("count"))

print()
print("Unfreeze recommendation — diagnostic only:")
display(unfreeze_recommendation)

print()
print("Back forecast metrics vs existing baseline:")
forecast_bucket_display_cols = [
    "model_spec",
    "tenor_bucket",
    "rows",
    "rmse_log",
    "mae_log",
    "bias_log_pred_minus_actual",
    "underforecast_rate",
    "large_underforecast_1p5x_rate",
    "large_underforecast_2p0x_rate",
    "forecast_top_decile_capture_rate",
    "missed_top_decile_actual_rate",
    "underforecast_rate_in_actual_top_decile",
    "candidate_forecast_vol_pct_mean",
    "existing_rmse_log",
    "existing_forecast_top_decile_capture_rate",
    "capture_improvement_vs_existing",
    "missed_top_decile_reduction_vs_existing",
    "rmse_change_vs_existing",
    "large_underforecast_2p0x_change_vs_existing",
    "forecast_vol_mean_change_vs_existing",
]
display(
    forecast_metrics_by_bucket_vs_existing
    .sort_values(["capture_improvement_vs_existing", "rmse_change_vs_existing"], ascending=[False, True])
    .loc[:, forecast_bucket_display_cols]
)

print()
print("Back forecast metrics by tenor vs existing baseline:")
forecast_tenor_display_cols = [
    "model_spec",
    "tenor",
    "tenor_bucket",
    "rows",
    "rmse_log",
    "mae_log",
    "bias_log_pred_minus_actual",
    "underforecast_rate",
    "large_underforecast_2p0x_rate",
    "forecast_top_decile_capture_rate",
    "missed_top_decile_actual_rate",
    "candidate_forecast_vol_pct_mean",
    "existing_forecast_top_decile_capture_rate",
    "capture_improvement_vs_existing",
    "rmse_change_vs_existing",
    "forecast_vol_mean_change_vs_existing",
]
display(
    forecast_metrics_by_tenor_vs_existing
    .sort_values(["tenor", "capture_improvement_vs_existing", "rmse_change_vs_existing"], ascending=[True, False, True])
    .loc[:, forecast_tenor_display_cols]
)

print()
print("Locked Core Back signal pass diagnostics:")
display(signal_pass_diagnostics)

print()
print("Locked Core Back threshold trade summary vs existing:")
trade_display_cols = [
    "selection_universe",
    "model_spec",
    "preservation_screen",
    "trades",
    "win_rate",
    "profit_factor",
    "total_pnl",
    "total_pnl_change_frac_vs_existing",
    "worst_trade",
    "selected_trade_drawdown",
    "worst_20_trade_sum",
    "trades_le_neg_100k",
    "pnl_2025",
    "pnl_2025_change_frac_vs_existing",
    "trades_2025",
    "worst_trade_2025",
    "trades_change_vs_existing",
    "win_rate_change_vs_existing",
    "total_pnl_change_vs_existing",
    "profit_factor_change_vs_existing",
    "worst_trade_change_vs_existing",
    "drawdown_change_vs_existing",
    "trades_le_neg_100k_change_vs_existing",
    "pnl_2025_change_vs_existing",
]
display(
    trade_summary_vs_existing
    .sort_values(["selection_universe", "preservation_screen", "total_pnl"], ascending=[True, True, False])
    .loc[:, [c for c in trade_display_cols if c in trade_summary_vs_existing.columns]]
)

print()
print("Saved outputs:")
saved_paths = [
    forecast_panel_path,
    common_panel_path,
    signal_panel_path,
    forecast_metrics_by_bucket_path,
    forecast_metrics_by_tenor_path,
    forecast_metrics_by_year_path,
    forecast_metrics_by_bucket_vs_existing_path,
    forecast_metrics_by_tenor_vs_existing_path,
    trade_summary_path,
    trade_summary_vs_existing_path,
    signal_pass_diagnostics_path,
    unfreeze_recommendation_path,
    fit_log_path,
    inner_cv_path,
    feature_validation_path,
    validation_path,
    manifest_path,
]

for p in saved_paths:
    print(f"  {p}")

print("\nCELL 6C PASSED")

In [ ]:
# ======================================================================================
# Cell 7A — Unified fds_no_min_return denominator OOS test
# ======================================================================================
#
# Objective:
#   Test one exact denominator model spec across all nine tenors:
#
#       fds_no_min_return
#
#   This is the first unified-denominator test after deciding that Back should be
#   unfrozen only if we can use the same exact model spec across Front/Middle/Back.
#
# Unified feature set:
#   - candidate_log_downside_rv_5d
#   - candidate_log_downside_rv_10d
#   - candidate_log_downside_rv_21d
#   - candidate_log_downside_rv_63d
#   - candidate_downside_share_5d
#   - candidate_downside_share_10d
#   - candidate_max_abs_return_3d
#   - candidate_max_abs_return_5d
#   - candidate_max_abs_return_10d
#
# Explicitly excluded:
#   - candidate_min_return_* features
#   - blending
#   - Back-specific model spec
#   - Front/Middle-specific model spec
#   - threshold optimization
#
# Scope:
#   - Load latest Cell 4 enriched candidate-feature panel.
#   - Fit unified fds_no_min_return for all exact tenors:
#       9, 12, 15, 18, 21, 24, 27, 30, 33
#   - Use expanding yearly OOS leakage guard:
#       train trade_date < test-year start
#       train last_forward_rv_date < test-year start
#   - Compare unified model vs existing har_downside_v1.
#   - Where available, compare Front/Middle against prior leading candidates:
#       fast_rv_plus_shock
#       fast_downside_plus_shock
#       full_fast_state
#   - Rebuild model_vrp_log and prior-only z-scores by model × tenor.
#   - Apply existing diagnostic Core thresholds by bucket.
#
# Not in scope:
#   - No threshold sweep.
#   - No blending.
#   - No production model lock.
#   - No sizing changes.
#   - No Secondary threshold research.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ======================================================================================
# 0. Configuration
# ======================================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
NEW_BRANCH = "vrp_front_middle_corsi_forecast_repair_v1"

OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

FRONT_TENORS = [9, 12, 15]
MIDDLE_TENORS = [18, 21, 24]
BACK_TENORS = [27, 30, 33]
ALL_TENORS = FRONT_TENORS + MIDDLE_TENORS + BACK_TENORS

TENOR_BUCKET_MAP = {
    9: "Front",
    12: "Front",
    15: "Front",
    18: "Middle",
    21: "Middle",
    24: "Middle",
    27: "Back",
    30: "Back",
    33: "Back",
}

FORECAST_REPAIR_SCOPE_MAP = {
    "Front": "RESEARCH_FRONT",
    "Middle": "RESEARCH_MIDDLE",
    "Back": "UNFROZEN_BACK_DIAGNOSTIC",
}

TARGET_LOG_COL = "target_log_variance"
TARGET_VAR_COL = "target_realized_variance"

FORECAST_VAR_BASELINE_COL = "forecast_variance_har_downside_v1"
PRED_LOG_BASELINE_COL = "predicted_log_variance_har_downside_v1"

EXISTING_SPEC = "existing_har_downside_v1"
UNIFIED_SPEC = "unified_fds_no_min_return"

ALPHA_GRID = [1.0, 10.0, 100.0, 300.0, 1000.0]
FALLBACK_ALPHA = 100.0

MIN_OUTER_TRAIN_ROWS = 250
MIN_INNER_TRAIN_ROWS = 250
MIN_INNER_VAL_ROWS = 30
EPS = 1e-12

# One exact model spec for all buckets.
UNIFIED_FEATURES = [
    "candidate_log_downside_rv_5d",
    "candidate_log_downside_rv_10d",
    "candidate_log_downside_rv_21d",
    "candidate_log_downside_rv_63d",
    "candidate_downside_share_5d",
    "candidate_downside_share_10d",
    "candidate_max_abs_return_3d",
    "candidate_max_abs_return_5d",
    "candidate_max_abs_return_10d",
]

EXPLICITLY_EXCLUDED_FEATURES = [
    "candidate_min_return_3d",
    "candidate_min_return_5d",
    "candidate_min_return_10d",
]

FORBIDDEN_MODEL_FEATURE_KEYWORDS = [
    "implied", "vix", "vrp", "z_", "rsi", "actual_dte", "win", "pnl",
    "credit", "strike", "expiry", "expiration", "option", "delta", "bucket", "tenor",
    "target", "forward", "last_forward", "trade_outcome",
]

# Diagnostic Core thresholds.
# Back is intentionally held at the locked Back thresholds from the prior work.
CORE_THRESHOLDS_BY_BUCKET = {
    "Front": {
        "model_vrp_log": 0.60,
        "model_vrp_z_3m": 0.55,
        "model_vrp_z_1y": 0.65,
        "RSI14_max": 70.0,
        "forecast_vol_pct_min": 8.5,
    },
    "Middle": {
        "model_vrp_log": 0.65,
        "model_vrp_z_3m": 0.75,
        "model_vrp_z_1y": 0.65,
        "RSI14_max": 68.0,
        "forecast_vol_pct_min": 8.5,
    },
    "Back": {
        "model_vrp_log": 0.70,
        "model_vrp_z_3m": 0.70,
        "model_vrp_z_1y": 0.70,
        "RSI14_max": 70.0,
        "forecast_vol_pct_min": 8.5,
    },
}

PRIOR_FRONT_MIDDLE_MODELS = [
    "fast_rv_plus_shock",
    "fast_downside_plus_shock",
    "full_fast_state",
]

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def forbidden_feature_hits(cols: list[str], forbidden_keywords: list[str]) -> pd.DataFrame:
    hits = []
    for col in cols:
        col_lower = col.lower()
        for kw in forbidden_keywords:
            if kw in col_lower:
                hits.append({"feature": col, "forbidden_keyword": kw})
    return pd.DataFrame(hits, columns=["feature", "forbidden_keyword"])


def make_ridge_model(alpha: float) -> Pipeline:
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=alpha)),
        ]
    )


def fit_ridge_predict(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    features: list[str],
    target_col: str,
    alpha: float,
) -> np.ndarray:
    model = make_ridge_model(alpha)
    model.fit(train_df[features].to_numpy(), train_df[target_col].to_numpy())
    return model.predict(test_df[features].to_numpy())


def choose_alpha_inner_yearly_cv(
    train_pool: pd.DataFrame,
    features: list[str],
    target_col: str,
    alpha_grid: list[float],
    fallback_alpha: float,
) -> tuple[float, pd.DataFrame]:
    cv_rows = []

    available_years = sorted(train_pool["trade_date"].dt.year.dropna().astype(int).unique().tolist())

    for val_year in available_years:
        val_start = pd.Timestamp(f"{val_year}-01-01")
        val_end = pd.Timestamp(f"{val_year + 1}-01-01")

        inner_train = train_pool[
            (train_pool["trade_date"] < val_start)
            & (train_pool["last_forward_rv_date"] < val_start)
        ].copy()

        inner_val = train_pool[
            (train_pool["trade_date"] >= val_start)
            & (train_pool["trade_date"] < val_end)
        ].copy()

        inner_train = inner_train.dropna(subset=features + [target_col])
        inner_val = inner_val.dropna(subset=features + [target_col])

        if len(inner_train) < MIN_INNER_TRAIN_ROWS or len(inner_val) < MIN_INNER_VAL_ROWS:
            continue

        for alpha in alpha_grid:
            try:
                pred = fit_ridge_predict(
                    train_df=inner_train,
                    test_df=inner_val,
                    features=features,
                    target_col=target_col,
                    alpha=alpha,
                )

                cv_rows.append({
                    "val_year": val_year,
                    "alpha": alpha,
                    "inner_train_rows": int(len(inner_train)),
                    "inner_val_rows": int(len(inner_val)),
                    "rmse_log": float(np.sqrt(mean_squared_error(inner_val[target_col], pred))),
                    "mae_log": float(mean_absolute_error(inner_val[target_col], pred)),
                    "bias_log_pred_minus_actual": float(np.mean(pred - inner_val[target_col].to_numpy())),
                })

            except Exception as e:
                cv_rows.append({
                    "val_year": val_year,
                    "alpha": alpha,
                    "inner_train_rows": int(len(inner_train)),
                    "inner_val_rows": int(len(inner_val)),
                    "rmse_log": np.nan,
                    "mae_log": np.nan,
                    "bias_log_pred_minus_actual": np.nan,
                    "error": f"{type(e).__name__}: {e}",
                })

    cv = pd.DataFrame(cv_rows)

    if cv.empty or cv["rmse_log"].notna().sum() == 0:
        return fallback_alpha, cv

    alpha_summary = (
        cv.dropna(subset=["rmse_log"])
        .groupby("alpha", as_index=False)
        .agg(
            cv_folds=("val_year", "nunique"),
            mean_rmse_log=("rmse_log", "mean"),
            mean_mae_log=("mae_log", "mean"),
            mean_bias_log_pred_minus_actual=("bias_log_pred_minus_actual", "mean"),
        )
        .sort_values(["mean_rmse_log", "mean_mae_log", "alpha"], ascending=[True, True, True])
    )

    if alpha_summary.empty:
        return fallback_alpha, cv

    return float(alpha_summary.iloc[0]["alpha"]), cv


def summarize_forecast_metrics(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    rows = []

    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_cols, keys))

        valid_g = g[
            np.isfinite(g["predicted_log_variance_candidate"])
            & np.isfinite(g[TARGET_LOG_COL])
            & np.isfinite(g["forecast_variance_candidate"])
            & np.isfinite(g[TARGET_VAR_COL])
            & (g["forecast_variance_candidate"] > 0)
            & (g[TARGET_VAR_COL] > 0)
        ].copy()

        row["rows"] = int(len(valid_g))

        if len(valid_g) == 0:
            rows.append(row)
            continue

        log_error = valid_g["predicted_log_variance_candidate"] - valid_g[TARGET_LOG_COL]
        ratio = valid_g[TARGET_VAR_COL] / valid_g["forecast_variance_candidate"]
        actual_top = valid_g[valid_g["actual_top_decile_common_by_tenor"]].copy()

        row["rmse_log"] = float(np.sqrt(mean_squared_error(valid_g[TARGET_LOG_COL], valid_g["predicted_log_variance_candidate"])))
        row["mae_log"] = float(mean_absolute_error(valid_g[TARGET_LOG_COL], valid_g["predicted_log_variance_candidate"]))
        row["bias_log_pred_minus_actual"] = float(log_error.mean())
        row["median_log_error"] = float(log_error.median())
        row["underforecast_rate"] = float((valid_g["forecast_variance_candidate"] < valid_g[TARGET_VAR_COL]).mean())
        row["large_underforecast_1p5x_rate"] = float((ratio > 1.5).mean())
        row["large_underforecast_2p0x_rate"] = float((ratio > 2.0).mean())
        row["actual_top_decile_rows"] = int(len(actual_top))
        row["forecast_top_decile_capture_rate"] = (
            float(actual_top["forecast_top_decile_common_by_model_tenor"].mean()) if len(actual_top) else np.nan
        )
        row["missed_top_decile_actual_rate"] = (
            float((~actual_top["forecast_top_decile_common_by_model_tenor"]).mean()) if len(actual_top) else np.nan
        )
        row["underforecast_rate_in_actual_top_decile"] = (
            float((actual_top["forecast_variance_candidate"] < actual_top[TARGET_VAR_COL]).mean()) if len(actual_top) else np.nan
        )
        row["avg_actual_to_forecast_var_ratio"] = float(ratio.mean())
        row["median_actual_to_forecast_var_ratio"] = float(ratio.median())
        row["target_realized_vol_pct_mean"] = float(valid_g["target_realized_vol_pct"].mean())
        row["candidate_forecast_vol_pct_mean"] = float(valid_g["candidate_forecast_vol_pct"].mean())
        row["forecast_vol_minus_target_vol_mean"] = (
            row["candidate_forecast_vol_pct_mean"] - row["target_realized_vol_pct_mean"]
        )

        rows.append(row)

    return pd.DataFrame(rows)


def add_vs_existing(metrics: pd.DataFrame, join_cols: list[str]) -> pd.DataFrame:
    baseline = metrics[metrics["model_spec"] == EXISTING_SPEC].copy()

    baseline_keep = join_cols + [
        "rmse_log",
        "mae_log",
        "bias_log_pred_minus_actual",
        "underforecast_rate",
        "large_underforecast_1p5x_rate",
        "large_underforecast_2p0x_rate",
        "forecast_top_decile_capture_rate",
        "missed_top_decile_actual_rate",
        "underforecast_rate_in_actual_top_decile",
        "candidate_forecast_vol_pct_mean",
    ]

    rename_map = {
        c: f"existing_{c}"
        for c in baseline_keep
        if c not in join_cols
    }

    baseline = baseline[baseline_keep].rename(columns=rename_map)
    out = metrics.merge(baseline, on=join_cols, how="left")

    out["capture_improvement_vs_existing"] = (
        out["forecast_top_decile_capture_rate"] - out["existing_forecast_top_decile_capture_rate"]
    )
    out["missed_top_decile_reduction_vs_existing"] = (
        out["existing_missed_top_decile_actual_rate"] - out["missed_top_decile_actual_rate"]
    )
    out["rmse_change_vs_existing"] = out["rmse_log"] - out["existing_rmse_log"]
    out["mae_change_vs_existing"] = out["mae_log"] - out["existing_mae_log"]
    out["large_underforecast_2p0x_change_vs_existing"] = (
        out["large_underforecast_2p0x_rate"] - out["existing_large_underforecast_2p0x_rate"]
    )
    out["forecast_vol_mean_change_vs_existing"] = (
        out["candidate_forecast_vol_pct_mean"] - out["existing_candidate_forecast_vol_pct_mean"]
    )

    return out


def add_common_top_decile_flags(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    actual_rank_source = (
        d[["trade_date", "tenor", TARGET_VAR_COL]]
        .drop_duplicates(["trade_date", "tenor"])
        .copy()
    )

    actual_rank_source["actual_var_pct_rank_common_by_tenor"] = (
        actual_rank_source.groupby("tenor")[TARGET_VAR_COL].rank(pct=True)
    )
    actual_rank_source["actual_top_decile_common_by_tenor"] = (
        actual_rank_source["actual_var_pct_rank_common_by_tenor"] >= 0.90
    )

    d = d.merge(
        actual_rank_source[
            ["trade_date", "tenor", "actual_var_pct_rank_common_by_tenor", "actual_top_decile_common_by_tenor"]
        ],
        on=["trade_date", "tenor"],
        how="left",
        validate="many_to_one",
    )

    d["forecast_var_pct_rank_common_by_model_tenor"] = (
        d.groupby(["model_spec", "tenor"])["forecast_variance_candidate"].rank(pct=True)
    )
    d["forecast_top_decile_common_by_model_tenor"] = (
        d["forecast_var_pct_rank_common_by_model_tenor"] >= 0.90
    )

    return d


def compute_prior_z_by_model_tenor(df: pd.DataFrame, value_col: str, window: int, min_periods: int) -> pd.Series:
    out = pd.Series(np.nan, index=df.index, dtype="float64")

    for _, g in df.sort_values(["model_spec", "tenor", "trade_date"]).groupby(["model_spec", "tenor"]):
        shifted = g[value_col].shift(1)
        roll_mean = shifted.rolling(window=window, min_periods=min_periods).mean()
        roll_std = shifted.rolling(window=window, min_periods=min_periods).std(ddof=1)
        z = (g[value_col] - roll_mean) / roll_std.replace(0.0, np.nan)
        out.loc[g.index] = z

    return out


def max_drawdown_from_pnl(pnl: pd.Series) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)
    cumulative = pnl.cumsum()
    running_max = cumulative.cummax()
    drawdown = cumulative - running_max
    return float(drawdown.min()) if len(drawdown) else np.nan


def worst_rolling_sum(pnl: pd.Series, window: int) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)
    if len(pnl) == 0:
        return np.nan
    if len(pnl) < window:
        return float(pnl.sum())
    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def summarize_trade_set(df: pd.DataFrame, universe_label: str, group_cols: list[str]) -> pd.DataFrame:
    rows = []

    if df.empty:
        return pd.DataFrame()

    d = df.copy()
    d["normalized_pnl_dollars"] = pd.to_numeric(d["normalized_pnl_dollars"], errors="coerce")
    d["win"] = pd.to_numeric(d["win"], errors="coerce")
    d = d[d["normalized_pnl_dollars"].notna()].copy()
    d = d.sort_values(["model_spec", "trade_date", "tenor"])

    for keys, g in d.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_cols, keys))
        pnl = g["normalized_pnl_dollars"]
        wins = g["win"]

        gross_profit = float(pnl[pnl > 0].sum())
        gross_loss = float(pnl[pnl < 0].sum())
        profit_factor = gross_profit / abs(gross_loss) if gross_loss < 0 else np.inf

        y2025 = g[g["trade_date"].dt.year == 2025].copy()

        row.update({
            "selection_universe": universe_label,
            "trades": int(len(g)),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,
            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor,
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "trades_2025": int(len(y2025)),
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,
        })

        rows.append(row)

    return pd.DataFrame(rows)


def compare_trade_summary_to_existing(summary: pd.DataFrame, join_cols: list[str]) -> pd.DataFrame:
    if summary.empty or EXISTING_SPEC not in set(summary["model_spec"]):
        return summary.copy()

    out_rows = []

    base_group_cols = ["selection_universe"] + join_cols

    for keys, g in summary.groupby(base_group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        base_rows = g[g["model_spec"] == EXISTING_SPEC]
        if base_rows.empty:
            continue

        base = base_rows.iloc[0]

        for _, row in g.iterrows():
            r = row.to_dict()

            r["trades_change_vs_existing"] = row["trades"] - base["trades"]
            r["win_rate_change_vs_existing"] = row["win_rate"] - base["win_rate"]
            r["total_pnl_change_vs_existing"] = row["total_pnl"] - base["total_pnl"]
            r["profit_factor_change_vs_existing"] = row["profit_factor"] - base["profit_factor"]
            r["worst_trade_change_vs_existing"] = row["worst_trade"] - base["worst_trade"]
            r["drawdown_change_vs_existing"] = row["selected_trade_drawdown"] - base["selected_trade_drawdown"]
            r["trades_le_neg_100k_change_vs_existing"] = row["trades_le_neg_100k"] - base["trades_le_neg_100k"]
            r["pnl_2025_change_vs_existing"] = row["pnl_2025"] - base["pnl_2025"]

            r["total_pnl_change_frac_vs_existing"] = (
                r["total_pnl_change_vs_existing"] / abs(base["total_pnl"])
                if abs(base["total_pnl"]) > 0 else np.nan
            )
            r["pnl_2025_change_frac_vs_existing"] = (
                r["pnl_2025_change_vs_existing"] / abs(base["pnl_2025"])
                if abs(base["pnl_2025"]) > 0 else np.nan
            )

            if row["model_spec"] == EXISTING_SPEC:
                r["comparison_label"] = "BASELINE_REFERENCE"
            else:
                r["comparison_label"] = "UNIFIED_VS_EXISTING"

            out_rows.append(r)

    return pd.DataFrame(out_rows)


# ======================================================================================
# 2. Load latest Cell 4 feature panel
# ======================================================================================

cell4_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "04_front_middle_candidate_feature_panel_*.parquet",
    required=True,
)

panel = pd.read_parquet(cell4_panel_path)

duplicate_cols = panel.columns[panel.columns.duplicated()].tolist()
if duplicate_cols:
    print("Dropping duplicate column labels in Cell 7A source panel:")
    print(sorted(set(duplicate_cols)))
panel = panel.loc[:, ~panel.columns.duplicated(keep="last")].copy()

panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce").dt.normalize()
panel["last_forward_rv_date"] = pd.to_datetime(panel["last_forward_rv_date"], errors="coerce").dt.normalize()
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype("Int64")

required_cols = [
    "trade_date",
    "last_forward_rv_date",
    "tenor",
    TARGET_LOG_COL,
    TARGET_VAR_COL,
    FORECAST_VAR_BASELINE_COL,
    "implied_variance",
    "RSI14",
    "win",
    "normalized_pnl_dollars",
] + UNIFIED_FEATURES

missing_required = [c for c in required_cols if c not in panel.columns]
if missing_required:
    raise KeyError(f"Cell 7A missing required columns from Cell 4 panel: {missing_required}")

excluded_present = [c for c in EXPLICITLY_EXCLUDED_FEATURES if c in UNIFIED_FEATURES]
if excluded_present:
    raise RuntimeError(f"Explicitly excluded features are present in UNIFIED_FEATURES: {excluded_present}")

feature_forbidden_hits = forbidden_feature_hits(UNIFIED_FEATURES, FORBIDDEN_MODEL_FEATURE_KEYWORDS)
if not feature_forbidden_hits.empty:
    display(feature_forbidden_hits)
    raise RuntimeError("Unified model features failed forbidden-keyword audit.")

universe = panel[panel["tenor"].isin(ALL_TENORS)].copy()

universe["tenor_bucket"] = universe["tenor"].astype(int).map(TENOR_BUCKET_MAP)
universe["forecast_repair_scope"] = universe["tenor_bucket"].map(FORECAST_REPAIR_SCOPE_MAP)

for c in [TARGET_LOG_COL, TARGET_VAR_COL, FORECAST_VAR_BASELINE_COL, "implied_variance", "RSI14"] + UNIFIED_FEATURES:
    universe[c] = pd.to_numeric(universe[c], errors="coerce")

universe = universe[
    universe["trade_date"].notna()
    & universe["last_forward_rv_date"].notna()
    & universe["tenor"].notna()
    & universe["tenor_bucket"].notna()
    & np.isfinite(universe[TARGET_LOG_COL])
    & np.isfinite(universe[TARGET_VAR_COL])
    & (universe[TARGET_VAR_COL] > 0)
    & np.isfinite(universe["implied_variance"])
    & (universe["implied_variance"] > 0)
].copy()

if universe.empty:
    raise RuntimeError("No valid all-tenor rows found for Cell 7A.")

# ======================================================================================
# 3. Existing baseline rows
# ======================================================================================

baseline = universe.copy()
baseline["model_spec"] = EXISTING_SPEC
baseline["model_source"] = "existing_har_downside_v1_forecast"
baseline["selected_alpha_candidate"] = pd.to_numeric(baseline.get("selected_alpha", np.nan), errors="coerce")
baseline["train_rows_used_candidate"] = pd.to_numeric(baseline.get("train_rows_used", np.nan), errors="coerce")
baseline["test_year_candidate"] = baseline["trade_date"].dt.year.astype("Int64")
baseline["fit_status_candidate"] = "existing_forecast"

if PRED_LOG_BASELINE_COL in baseline.columns:
    baseline["predicted_log_variance_candidate"] = pd.to_numeric(baseline[PRED_LOG_BASELINE_COL], errors="coerce")
else:
    baseline["predicted_log_variance_candidate"] = np.log(baseline[FORECAST_VAR_BASELINE_COL].clip(lower=EPS))

baseline["forecast_variance_candidate"] = baseline[FORECAST_VAR_BASELINE_COL]
baseline["candidate_forecast_vol_pct"] = np.sqrt(baseline["forecast_variance_candidate"].clip(lower=0)) * 100.0
baseline["target_realized_vol_pct"] = np.sqrt(baseline[TARGET_VAR_COL].clip(lower=0)) * 100.0

# ======================================================================================
# 4. Fit unified fds_no_min_return model OOS for all tenors
# ======================================================================================

forecast_rows = []
fit_log_rows = []
inner_cv_rows_all = []

for tenor in ALL_TENORS:
    tenor_df = universe[universe["tenor"] == tenor].copy()
    tenor_df = tenor_df.sort_values("trade_date").reset_index(drop=True)

    test_years = sorted(tenor_df["trade_date"].dt.year.dropna().astype(int).unique().tolist())

    for test_year in test_years:
        test_start = pd.Timestamp(f"{test_year}-01-01")
        test_end = pd.Timestamp(f"{test_year + 1}-01-01")

        train_pool = tenor_df[
            (tenor_df["trade_date"] < test_start)
            & (tenor_df["last_forward_rv_date"] < test_start)
        ].copy()

        test_rows = tenor_df[
            (tenor_df["trade_date"] >= test_start)
            & (tenor_df["trade_date"] < test_end)
        ].copy()

        train_fit = train_pool.dropna(subset=UNIFIED_FEATURES + [TARGET_LOG_COL]).copy()
        test_fit = test_rows.dropna(subset=UNIFIED_FEATURES + [TARGET_LOG_COL]).copy()

        if len(test_fit) == 0:
            fit_log_rows.append({
                "model_spec": UNIFIED_SPEC,
                "tenor": tenor,
                "test_year": test_year,
                "fit_status": "skipped_no_test_rows",
                "selected_alpha": np.nan,
                "train_rows_used": int(len(train_fit)),
                "test_rows_scored": 0,
                "feature_count": len(UNIFIED_FEATURES),
            })
            continue

        if len(train_fit) < MIN_OUTER_TRAIN_ROWS:
            fit_log_rows.append({
                "model_spec": UNIFIED_SPEC,
                "tenor": tenor,
                "test_year": test_year,
                "fit_status": "skipped_insufficient_train_rows",
                "selected_alpha": np.nan,
                "train_rows_used": int(len(train_fit)),
                "test_rows_scored": 0,
                "feature_count": len(UNIFIED_FEATURES),
            })
            continue

        selected_alpha, inner_cv = choose_alpha_inner_yearly_cv(
            train_pool=train_fit,
            features=UNIFIED_FEATURES,
            target_col=TARGET_LOG_COL,
            alpha_grid=ALPHA_GRID,
            fallback_alpha=FALLBACK_ALPHA,
        )

        if not inner_cv.empty:
            inner_cv["outer_model_spec"] = UNIFIED_SPEC
            inner_cv["outer_tenor"] = tenor
            inner_cv["outer_test_year"] = test_year
            inner_cv_rows_all.append(inner_cv)

        try:
            pred = fit_ridge_predict(
                train_df=train_fit,
                test_df=test_fit,
                features=UNIFIED_FEATURES,
                target_col=TARGET_LOG_COL,
                alpha=selected_alpha,
            )

            scored = test_fit.copy()
            scored["model_spec"] = UNIFIED_SPEC
            scored["model_source"] = "unified_fds_no_min_return_oos_refit"
            scored["selected_alpha_candidate"] = selected_alpha
            scored["train_rows_used_candidate"] = int(len(train_fit))
            scored["test_year_candidate"] = int(test_year)
            scored["fit_status_candidate"] = "candidate_fit"
            scored["predicted_log_variance_candidate"] = pred
            scored["forecast_variance_candidate"] = np.exp(pred)
            scored["candidate_forecast_vol_pct"] = np.sqrt(scored["forecast_variance_candidate"].clip(lower=0)) * 100.0
            scored["target_realized_vol_pct"] = np.sqrt(scored[TARGET_VAR_COL].clip(lower=0)) * 100.0

            forecast_rows.append(scored)

            fit_log_rows.append({
                "model_spec": UNIFIED_SPEC,
                "tenor": tenor,
                "test_year": test_year,
                "fit_status": "candidate_fit",
                "selected_alpha": selected_alpha,
                "train_rows_used": int(len(train_fit)),
                "test_rows_scored": int(len(scored)),
                "feature_count": len(UNIFIED_FEATURES),
            })

        except Exception as e:
            fit_log_rows.append({
                "model_spec": UNIFIED_SPEC,
                "tenor": tenor,
                "test_year": test_year,
                "fit_status": "fit_error",
                "selected_alpha": selected_alpha,
                "train_rows_used": int(len(train_fit)),
                "test_rows_scored": 0,
                "feature_count": len(UNIFIED_FEATURES),
                "error": f"{type(e).__name__}: {e}",
            })

candidate_forecasts = pd.concat(forecast_rows, ignore_index=True) if forecast_rows else pd.DataFrame()
fit_log = pd.DataFrame(fit_log_rows)
inner_cv_all = pd.concat(inner_cv_rows_all, ignore_index=True) if inner_cv_rows_all else pd.DataFrame()

if candidate_forecasts.empty:
    display(fit_log)
    raise RuntimeError("No unified candidate forecasts were produced. See fit_log.")

# Align columns.
for c in baseline.columns:
    if c not in candidate_forecasts.columns:
        candidate_forecasts[c] = np.nan

candidate_forecasts = candidate_forecasts[baseline.columns].copy()

all_forecasts = pd.concat([baseline, candidate_forecasts], ignore_index=True)
all_forecasts = all_forecasts.loc[:, ~all_forecasts.columns.duplicated(keep="last")].copy()

for c in [
    "forecast_variance_candidate",
    "predicted_log_variance_candidate",
    TARGET_VAR_COL,
    TARGET_LOG_COL,
    "implied_variance",
    "RSI14",
]:
    all_forecasts[c] = pd.to_numeric(all_forecasts[c], errors="coerce")

all_forecasts = all_forecasts[
    np.isfinite(all_forecasts["forecast_variance_candidate"])
    & np.isfinite(all_forecasts["predicted_log_variance_candidate"])
    & np.isfinite(all_forecasts[TARGET_VAR_COL])
    & np.isfinite(all_forecasts[TARGET_LOG_COL])
    & np.isfinite(all_forecasts["implied_variance"])
    & (all_forecasts["forecast_variance_candidate"] > 0)
    & (all_forecasts[TARGET_VAR_COL] > 0)
    & (all_forecasts["implied_variance"] > 0)
].copy()

all_forecasts["target_realized_vol_pct"] = np.sqrt(all_forecasts[TARGET_VAR_COL].clip(lower=0)) * 100.0
all_forecasts["candidate_forecast_vol_pct"] = np.sqrt(all_forecasts["forecast_variance_candidate"].clip(lower=0)) * 100.0
all_forecasts["log_error_pred_minus_actual"] = all_forecasts["predicted_log_variance_candidate"] - all_forecasts[TARGET_LOG_COL]
all_forecasts["actual_to_forecast_var_ratio"] = all_forecasts[TARGET_VAR_COL] / all_forecasts["forecast_variance_candidate"]

# ======================================================================================
# 5. Unified vs existing common-row forecast diagnostics
# ======================================================================================

review_specs_main = [EXISTING_SPEC, UNIFIED_SPEC]

review = all_forecasts[
    all_forecasts["model_spec"].isin(review_specs_main)
    & all_forecasts["tenor"].isin(ALL_TENORS)
].copy()

row_model_counts = (
    review.groupby(["trade_date", "tenor"], as_index=False)
    .agg(model_count=("model_spec", "nunique"))
)

common_keys = row_model_counts[row_model_counts["model_count"] == len(review_specs_main)][["trade_date", "tenor"]].copy()

if common_keys.empty:
    raise RuntimeError("No common all-tenor date × tenor rows found for unified vs existing comparison.")

common = review.merge(common_keys, on=["trade_date", "tenor"], how="inner", validate="many_to_one").copy()
common = add_common_top_decile_flags(common)

forecast_metrics_by_bucket = summarize_forecast_metrics(
    common,
    ["model_spec", "tenor_bucket", "forecast_repair_scope"],
)

forecast_metrics_by_tenor = summarize_forecast_metrics(
    common,
    ["model_spec", "tenor", "tenor_bucket", "forecast_repair_scope"],
)

forecast_metrics_by_year = summarize_forecast_metrics(
    common.assign(year=common["trade_date"].dt.year.astype("Int64")),
    ["model_spec", "year", "tenor_bucket", "forecast_repair_scope"],
)

forecast_metrics_by_bucket_vs_existing = add_vs_existing(
    forecast_metrics_by_bucket,
    ["tenor_bucket", "forecast_repair_scope"],
)

forecast_metrics_by_tenor_vs_existing = add_vs_existing(
    forecast_metrics_by_tenor,
    ["tenor", "tenor_bucket", "forecast_repair_scope"],
)

forecast_metrics_by_year_vs_existing = add_vs_existing(
    forecast_metrics_by_year,
    ["year", "tenor_bucket", "forecast_repair_scope"],
)

# ======================================================================================
# 6. Optional Front/Middle prior-candidate comparison
# ======================================================================================

prior_front_middle_comparison = pd.DataFrame()
prior_front_middle_metrics = pd.DataFrame()
prior_front_middle_metrics_vs_existing = pd.DataFrame()
prior_source_path = None

try:
    prior_source_path = latest_file(
        OUT_PROCESSED_DIR,
        "05_front_middle_oos_forecast_candidate_panel_*.parquet",
        required=False,
    )

    if prior_source_path is not None:
        prior = pd.read_parquet(prior_source_path)
        prior = prior.loc[:, ~prior.columns.duplicated(keep="last")].copy()

        prior["trade_date"] = pd.to_datetime(prior["trade_date"], errors="coerce").dt.normalize()
        prior["tenor"] = pd.to_numeric(prior["tenor"], errors="coerce").astype("Int64")

        if "model_family" in prior.columns:
            prior = prior[
                prior["model_family"].isin(PRIOR_FRONT_MIDDLE_MODELS)
                & prior["tenor"].isin(FRONT_TENORS + MIDDLE_TENORS)
            ].copy()

            prior["model_spec"] = prior["model_family"]
            prior["tenor_bucket"] = prior["tenor"].astype(int).map(TENOR_BUCKET_MAP)
            prior["forecast_repair_scope"] = prior["tenor_bucket"].map(FORECAST_REPAIR_SCOPE_MAP)

            needed_prior_cols = [
                "trade_date",
                "tenor",
                "tenor_bucket",
                "forecast_repair_scope",
                "model_spec",
                TARGET_LOG_COL,
                TARGET_VAR_COL,
                "predicted_log_variance_candidate",
                "forecast_variance_candidate",
                "candidate_forecast_vol_pct",
                "target_realized_vol_pct",
            ]

            prior_missing = [c for c in needed_prior_cols if c not in prior.columns]

            if not prior_missing and not prior.empty:
                prior = prior[needed_prior_cols].copy()

                main_fm = all_forecasts[
                    all_forecasts["model_spec"].isin([EXISTING_SPEC, UNIFIED_SPEC])
                    & all_forecasts["tenor"].isin(FRONT_TENORS + MIDDLE_TENORS)
                ][needed_prior_cols].copy()

                prior_compare_raw = pd.concat([main_fm, prior], ignore_index=True)

                prior_review_specs = [EXISTING_SPEC, UNIFIED_SPEC] + PRIOR_FRONT_MIDDLE_MODELS

                prior_counts = (
                    prior_compare_raw
                    .groupby(["trade_date", "tenor"], as_index=False)
                    .agg(model_count=("model_spec", "nunique"))
                )

                prior_common_keys = prior_counts[
                    prior_counts["model_count"] == len(prior_review_specs)
                ][["trade_date", "tenor"]].copy()

                if not prior_common_keys.empty:
                    prior_front_middle_comparison = prior_compare_raw.merge(
                        prior_common_keys,
                        on=["trade_date", "tenor"],
                        how="inner",
                        validate="many_to_one",
                    ).copy()

                    prior_front_middle_comparison = add_common_top_decile_flags(prior_front_middle_comparison)

                    prior_front_middle_metrics = summarize_forecast_metrics(
                        prior_front_middle_comparison,
                        ["model_spec", "tenor_bucket", "forecast_repair_scope"],
                    )

                    prior_front_middle_metrics_vs_existing = add_vs_existing(
                        prior_front_middle_metrics,
                        ["tenor_bucket", "forecast_repair_scope"],
                    )

except Exception as e:
    print(f"Prior Front/Middle comparison skipped due to error: {type(e).__name__}: {e}")

# ======================================================================================
# 7. Rebuild VRP and diagnostic threshold pass panel
# ======================================================================================

signal_base = all_forecasts[
    all_forecasts["model_spec"].isin(review_specs_main)
    & all_forecasts["tenor"].isin(ALL_TENORS)
].copy()

signal_base = signal_base.sort_values(["model_spec", "tenor", "trade_date"]).reset_index(drop=True)

signal_base["model_vrp_log_candidate"] = np.log(
    signal_base["implied_variance"] / signal_base["forecast_variance_candidate"]
)

signal_base["model_vrp_z_3m_candidate"] = compute_prior_z_by_model_tenor(
    signal_base,
    value_col="model_vrp_log_candidate",
    window=63,
    min_periods=63,
)

signal_base["model_vrp_z_1y_candidate"] = compute_prior_z_by_model_tenor(
    signal_base,
    value_col="model_vrp_log_candidate",
    window=252,
    min_periods=252,
)

threshold_rows = []
for bucket, params in CORE_THRESHOLDS_BY_BUCKET.items():
    threshold_rows.append({
        "tenor_bucket": bucket,
        "threshold_model_vrp_log": params["model_vrp_log"],
        "threshold_model_vrp_z_3m": params["model_vrp_z_3m"],
        "threshold_model_vrp_z_1y": params["model_vrp_z_1y"],
        "threshold_RSI14_max": params["RSI14_max"],
        "threshold_forecast_vol_pct_min": params["forecast_vol_pct_min"],
    })

threshold_table = pd.DataFrame(threshold_rows)

signal_base = signal_base.merge(
    threshold_table,
    on="tenor_bucket",
    how="left",
    validate="many_to_one",
)

signal_base["diagnostic_core_pass"] = (
    (signal_base["model_vrp_log_candidate"] > signal_base["threshold_model_vrp_log"])
    & (signal_base["model_vrp_z_3m_candidate"] > signal_base["threshold_model_vrp_z_3m"])
    & (signal_base["model_vrp_z_1y_candidate"] > signal_base["threshold_model_vrp_z_1y"])
    & (signal_base["RSI14"] < signal_base["threshold_RSI14_max"])
    & (signal_base["candidate_forecast_vol_pct"] > signal_base["threshold_forecast_vol_pct_min"])
)

signal_base["signal_inputs_available"] = (
    np.isfinite(signal_base["model_vrp_log_candidate"])
    & np.isfinite(signal_base["model_vrp_z_3m_candidate"])
    & np.isfinite(signal_base["model_vrp_z_1y_candidate"])
    & np.isfinite(signal_base["RSI14"])
    & np.isfinite(signal_base["candidate_forecast_vol_pct"])
    & signal_base["normalized_pnl_dollars"].notna()
)

# Restrict signal diagnostics to unified-vs-existing common date×tenor rows.
common_signal = signal_base.merge(
    common_keys,
    on=["trade_date", "tenor"],
    how="inner",
    validate="many_to_one",
).copy()

qualified_all_rows = common_signal[
    common_signal["diagnostic_core_pass"]
    & common_signal["signal_inputs_available"]
].copy()

qualified_one_per_bucket_date = (
    qualified_all_rows
    .sort_values(
        ["model_spec", "tenor_bucket", "trade_date", "model_vrp_z_1y_candidate", "model_vrp_z_3m_candidate", "model_vrp_log_candidate", "tenor"],
        ascending=[True, True, True, False, False, False, False],
    )
    .groupby(["model_spec", "tenor_bucket", "trade_date"], as_index=False)
    .head(1)
    .copy()
)

qualified_30d_anchor = qualified_all_rows[qualified_all_rows["tenor"] == 30].copy()

trade_summary_all = summarize_trade_set(
    qualified_all_rows,
    universe_label="all_qualified_rows",
    group_cols=["model_spec", "tenor_bucket"],
)

trade_summary_one_per_bucket = summarize_trade_set(
    qualified_one_per_bucket_date,
    universe_label="one_trade_per_bucket_date",
    group_cols=["model_spec", "tenor_bucket"],
)

trade_summary_30d = summarize_trade_set(
    qualified_30d_anchor,
    universe_label="thirty_day_anchor_only",
    group_cols=["model_spec", "tenor_bucket"],
)

trade_summary = pd.concat(
    [trade_summary_all, trade_summary_one_per_bucket, trade_summary_30d],
    ignore_index=True,
)

trade_summary_vs_existing = compare_trade_summary_to_existing(
    trade_summary,
    join_cols=["tenor_bucket"],
)

signal_pass_diagnostics = (
    common_signal
    .groupby(["model_spec", "tenor", "tenor_bucket"], as_index=False)
    .agg(
        rows=("trade_date", "size"),
        signal_inputs_available_rows=("signal_inputs_available", "sum"),
        diagnostic_core_pass_rows=("diagnostic_core_pass", "sum"),
        pass_rate=("diagnostic_core_pass", "mean"),
        avg_model_vrp_log=("model_vrp_log_candidate", "mean"),
        avg_z_3m=("model_vrp_z_3m_candidate", "mean"),
        avg_z_1y=("model_vrp_z_1y_candidate", "mean"),
        avg_forecast_vol_pct=("candidate_forecast_vol_pct", "mean"),
    )
)

# ======================================================================================
# 8. Unified recommendation diagnostics
# ======================================================================================

bucket_recommendation_rows = []

for bucket in ["Front", "Middle", "Back"]:
    fm = forecast_metrics_by_bucket_vs_existing[
        (forecast_metrics_by_bucket_vs_existing["model_spec"] == UNIFIED_SPEC)
        & (forecast_metrics_by_bucket_vs_existing["tenor_bucket"] == bucket)
    ]

    tm = trade_summary_vs_existing[
        (trade_summary_vs_existing["model_spec"] == UNIFIED_SPEC)
        & (trade_summary_vs_existing["tenor_bucket"] == bucket)
        & (trade_summary_vs_existing["selection_universe"] == "all_qualified_rows")
    ]

    one = trade_summary_vs_existing[
        (trade_summary_vs_existing["model_spec"] == UNIFIED_SPEC)
        & (trade_summary_vs_existing["tenor_bucket"] == bucket)
        & (trade_summary_vs_existing["selection_universe"] == "one_trade_per_bucket_date")
    ]

    row = {
        "tenor_bucket": bucket,
        "model_spec": UNIFIED_SPEC,
    }

    if len(fm):
        r = fm.iloc[0]
        row.update({
            "rmse_change_vs_existing": r.get("rmse_change_vs_existing", np.nan),
            "capture_improvement_vs_existing": r.get("capture_improvement_vs_existing", np.nan),
            "forecast_vol_mean_change_vs_existing": r.get("forecast_vol_mean_change_vs_existing", np.nan),
            "large_underforecast_2p0x_change_vs_existing": r.get("large_underforecast_2p0x_change_vs_existing", np.nan),
        })

    if len(tm):
        r = tm.iloc[0]
        row.update({
            "all_qualified_trades_change": r.get("trades_change_vs_existing", np.nan),
            "all_qualified_total_pnl_change": r.get("total_pnl_change_vs_existing", np.nan),
            "all_qualified_win_rate_change": r.get("win_rate_change_vs_existing", np.nan),
            "all_qualified_profit_factor_change": r.get("profit_factor_change_vs_existing", np.nan),
            "all_qualified_worst_trade_change": r.get("worst_trade_change_vs_existing", np.nan),
            "all_qualified_drawdown_change": r.get("drawdown_change_vs_existing", np.nan),
            "all_qualified_neg_100k_change": r.get("trades_le_neg_100k_change_vs_existing", np.nan),
            "all_qualified_2025_pnl_change": r.get("pnl_2025_change_vs_existing", np.nan),
        })

    if len(one):
        r = one.iloc[0]
        row.update({
            "one_per_bucket_total_pnl_change": r.get("total_pnl_change_vs_existing", np.nan),
            "one_per_bucket_win_rate_change": r.get("win_rate_change_vs_existing", np.nan),
            "one_per_bucket_profit_factor_change": r.get("profit_factor_change_vs_existing", np.nan),
            "one_per_bucket_worst_trade_change": r.get("worst_trade_change_vs_existing", np.nan),
            "one_per_bucket_drawdown_change": r.get("drawdown_change_vs_existing", np.nan),
            "one_per_bucket_neg_100k_change": r.get("trades_le_neg_100k_change_vs_existing", np.nan),
        })

    # Simple diagnostic classification.
    # This is not a final lock rule.
    rmse_ok = row.get("rmse_change_vs_existing", np.nan) <= 0.05
    capture_ok = row.get("capture_improvement_vs_existing", np.nan) >= 0.00
    pnl_ok = row.get("all_qualified_total_pnl_change", 0.0) >= -250_000
    risk_ok = row.get("one_per_bucket_neg_100k_change", 0.0) <= 1

    if rmse_ok and capture_ok and pnl_ok and risk_ok:
        row["bucket_unified_status"] = "PASS_DIAGNOSTIC"
    elif rmse_ok and capture_ok:
        row["bucket_unified_status"] = "BORDERLINE_REVIEW"
    else:
        row["bucket_unified_status"] = "FAIL_REVIEW"

    bucket_recommendation_rows.append(row)

unified_recommendation = pd.DataFrame(bucket_recommendation_rows)

if (
    len(unified_recommendation)
    and unified_recommendation["bucket_unified_status"].isin(["PASS_DIAGNOSTIC", "BORDERLINE_REVIEW"]).all()
):
    overall_unified_status = "ADVANCE_TO_SIGNAL_THRESHOLD_RESEARCH"
elif (
    len(unified_recommendation)
    and unified_recommendation["bucket_unified_status"].eq("PASS_DIAGNOSTIC").sum() >= 2
):
    overall_unified_status = "MIXED_ADVANCE_WITH_REVIEW"
else:
    overall_unified_status = "DO_NOT_ADVANCE_YET"

overall_recommendation = pd.DataFrame([{
    "model_spec": UNIFIED_SPEC,
    "overall_unified_status": overall_unified_status,
    "notes": (
        "Diagnostic only. Advance means build signal-threshold research panels, "
        "not production lock."
    ),
}])

# ======================================================================================
# 9. Validation
# ======================================================================================

validation_rows = []

def add_validation(check: str, status: str, detail: str):
    validation_rows.append({"check": check, "status": status, "detail": detail})

expected_tenors_found = sorted(universe["tenor"].dropna().astype(int).unique().tolist())
missing_tenors = sorted(set(ALL_TENORS) - set(expected_tenors_found))

front_middle_candidate_refit_rows = candidate_forecasts[
    candidate_forecasts["tenor"].isin(FRONT_TENORS + MIDDLE_TENORS)
]
back_candidate_refit_rows = candidate_forecasts[
    candidate_forecasts["tenor"].isin(BACK_TENORS)
]

bad_model_count_rows = int(
    (common.groupby(["trade_date", "tenor"])["model_spec"].nunique() != len(review_specs_main)).sum()
)

add_validation("cell4_panel_loaded", "PASS", f"path={cell4_panel_path}; rows={len(panel):,}")
add_validation("all_expected_tenors_present", "PASS" if not missing_tenors else "FAIL", f"missing_tenors={missing_tenors}")
add_validation("unified_feature_validation", "PASS", f"features={UNIFIED_FEATURES}")
add_validation("excluded_min_return_features", "PASS", f"excluded={EXPLICITLY_EXCLUDED_FEATURES}")
add_validation("candidate_forecasts_produced", "PASS", f"candidate rows={len(candidate_forecasts):,}")
add_validation("front_middle_candidate_refit_rows", "PASS", f"front/middle rows={len(front_middle_candidate_refit_rows):,}")
add_validation("back_candidate_refit_rows", "PASS", f"back rows={len(back_candidate_refit_rows):,}")
add_validation("common_keys_non_empty", "PASS" if len(common_keys) > 0 else "FAIL", f"common date×tenor keys={len(common_keys):,}")
add_validation("common_model_count", "PASS" if bad_model_count_rows == 0 else "FAIL", f"bad_model_count_rows={bad_model_count_rows}")
add_validation("fit_errors", "PASS" if not fit_log["fit_status"].eq("fit_error").any() else "WARN", f"fit_error_rows={int(fit_log['fit_status'].eq('fit_error').sum())}")
add_validation("no_blending", "PASS", "No old/new forecast blending performed")
add_validation("no_threshold_grid", "PASS", "No threshold sweep or threshold optimization performed")
add_validation("back_thresholds_unchanged", "PASS", str(CORE_THRESHOLDS_BY_BUCKET["Back"]))
add_validation("no_final_lock", "PASS", "Cell 7A is diagnostic / research only")

validation = pd.DataFrame(validation_rows)
failures = validation[validation["status"] == "FAIL"]

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 7A validation failed. See validation table above.")

# ======================================================================================
# 10. Save outputs
# ======================================================================================

date_min = common["trade_date"].min()
date_max = common["trade_date"].max()

forecast_panel_path = OUT_PROCESSED_DIR / (
    f"07A_unified_fds_no_min_return_oos_forecast_panel_{date_min:%Y%m%d}_{date_max:%Y%m%d}_{TIMESTAMP}.parquet"
)
common_panel_path = OUT_PROCESSED_DIR / (
    f"07A_unified_common_row_panel_{date_min:%Y%m%d}_{date_max:%Y%m%d}_{TIMESTAMP}.parquet"
)
signal_panel_path = OUT_PROCESSED_DIR / (
    f"07A_unified_signal_threshold_diagnostic_panel_{date_min:%Y%m%d}_{date_max:%Y%m%d}_{TIMESTAMP}.parquet"
)
qualified_panel_path = OUT_PROCESSED_DIR / (
    f"07A_unified_qualified_trade_diagnostic_panel_{date_min:%Y%m%d}_{date_max:%Y%m%d}_{TIMESTAMP}.parquet"
)

forecast_metrics_by_bucket_path = OUT_AUDIT_DIR / f"07A_unified_common_row_metrics_by_bucket_{TIMESTAMP}.csv"
forecast_metrics_by_tenor_path = OUT_AUDIT_DIR / f"07A_unified_common_row_metrics_by_tenor_{TIMESTAMP}.csv"
forecast_metrics_by_year_path = OUT_AUDIT_DIR / f"07A_unified_common_row_metrics_by_year_{TIMESTAMP}.csv"
forecast_metrics_by_bucket_vs_existing_path = OUT_AUDIT_DIR / f"07A_unified_common_row_metrics_by_bucket_vs_existing_{TIMESTAMP}.csv"
forecast_metrics_by_tenor_vs_existing_path = OUT_AUDIT_DIR / f"07A_unified_common_row_metrics_by_tenor_vs_existing_{TIMESTAMP}.csv"
forecast_metrics_by_year_vs_existing_path = OUT_AUDIT_DIR / f"07A_unified_common_row_metrics_by_year_vs_existing_{TIMESTAMP}.csv"
prior_front_middle_metrics_path = OUT_AUDIT_DIR / f"07A_prior_front_middle_candidate_comparison_{TIMESTAMP}.csv"
trade_summary_path = OUT_AUDIT_DIR / f"07A_unified_trade_summary_by_bucket_{TIMESTAMP}.csv"
trade_summary_vs_existing_path = OUT_AUDIT_DIR / f"07A_unified_trade_summary_by_bucket_vs_existing_{TIMESTAMP}.csv"
signal_pass_diagnostics_path = OUT_AUDIT_DIR / f"07A_unified_signal_pass_diagnostics_{TIMESTAMP}.csv"
threshold_table_path = OUT_AUDIT_DIR / f"07A_diagnostic_core_thresholds_used_{TIMESTAMP}.csv"
unified_recommendation_path = OUT_AUDIT_DIR / f"07A_unified_model_recommendation_{TIMESTAMP}.csv"
overall_recommendation_path = OUT_AUDIT_DIR / f"07A_overall_recommendation_{TIMESTAMP}.csv"
fit_log_path = OUT_AUDIT_DIR / f"07A_unified_fit_log_{TIMESTAMP}.csv"
inner_cv_path = OUT_AUDIT_DIR / f"07A_unified_inner_cv_{TIMESTAMP}.csv"
validation_path = OUT_AUDIT_DIR / f"07A_validation_{TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"07A_manifest_{TIMESTAMP}.json"

all_forecasts.to_parquet(forecast_panel_path, index=False)
common.to_parquet(common_panel_path, index=False)
common_signal.to_parquet(signal_panel_path, index=False)
qualified_all_rows.to_parquet(qualified_panel_path, index=False)

forecast_metrics_by_bucket.to_csv(forecast_metrics_by_bucket_path, index=False)
forecast_metrics_by_tenor.to_csv(forecast_metrics_by_tenor_path, index=False)
forecast_metrics_by_year.to_csv(forecast_metrics_by_year_path, index=False)
forecast_metrics_by_bucket_vs_existing.to_csv(forecast_metrics_by_bucket_vs_existing_path, index=False)
forecast_metrics_by_tenor_vs_existing.to_csv(forecast_metrics_by_tenor_vs_existing_path, index=False)
forecast_metrics_by_year_vs_existing.to_csv(forecast_metrics_by_year_vs_existing_path, index=False)

if not prior_front_middle_metrics_vs_existing.empty:
    prior_front_middle_metrics_vs_existing.to_csv(prior_front_middle_metrics_path, index=False)
else:
    pd.DataFrame().to_csv(prior_front_middle_metrics_path, index=False)

trade_summary.to_csv(trade_summary_path, index=False)
trade_summary_vs_existing.to_csv(trade_summary_vs_existing_path, index=False)
signal_pass_diagnostics.to_csv(signal_pass_diagnostics_path, index=False)
threshold_table.to_csv(threshold_table_path, index=False)
unified_recommendation.to_csv(unified_recommendation_path, index=False)
overall_recommendation.to_csv(overall_recommendation_path, index=False)
fit_log.to_csv(fit_log_path, index=False)
validation.to_csv(validation_path, index=False)

if not inner_cv_all.empty:
    inner_cv_all.to_csv(inner_cv_path, index=False)
else:
    pd.DataFrame().to_csv(inner_cv_path, index=False)

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "cell": "Cell 7A — Unified fds_no_min_return denominator OOS test",
            "timestamp": TIMESTAMP,
            "source_cell4_panel": str(cell4_panel_path),
            "prior_front_middle_source": str(prior_source_path) if prior_source_path is not None else None,
            "existing_spec": EXISTING_SPEC,
            "unified_spec": UNIFIED_SPEC,
            "all_tenors": ALL_TENORS,
            "unified_features": UNIFIED_FEATURES,
            "explicitly_excluded_features": EXPLICITLY_EXCLUDED_FEATURES,
            "alpha_grid": ALPHA_GRID,
            "fallback_alpha": FALLBACK_ALPHA,
            "diagnostic_core_thresholds_by_bucket": CORE_THRESHOLDS_BY_BUCKET,
            "overall_unified_status": overall_unified_status,
            "notes": [
                "One exact model spec was used across all nine tenors.",
                "No min-return features were used.",
                "No blending was performed.",
                "No threshold grid or threshold optimization was performed.",
                "Back thresholds were held unchanged.",
                "Diagnostic Core threshold pass is not a final production signal lock.",
            ],
        },
        f,
        indent=2,
    )

# ======================================================================================
# 11. Display concise review
# ======================================================================================

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 320)

print("=" * 100)
print("Cell 7A — Unified fds_no_min_return denominator OOS test")
print("=" * 100)
print(f"Source Cell 4 panel:             {cell4_panel_path}")
print(f"Unified model spec:              {UNIFIED_SPEC}")
print(f"All tenors:                      {ALL_TENORS}")
print(f"Unified features:                {UNIFIED_FEATURES}")
print(f"Explicitly excluded features:    {EXPLICITLY_EXCLUDED_FEATURES}")
print(f"Candidate forecast rows:         {len(candidate_forecasts):,}")
print(f"Existing baseline rows:          {len(baseline):,}")
print(f"Common date×tenor keys:          {len(common_keys):,}")
print(f"Common forecast rows:            {len(common):,}")
print(f"Date range:                      {date_min.date()} to {date_max.date()}")
print(f"No blending:                     True")
print(f"No threshold grid:               True")
print(f"Back thresholds unchanged:       {CORE_THRESHOLDS_BY_BUCKET['Back']}")
print(f"Overall diagnostic status:       {overall_unified_status}")

print()
print("Validation:")
display(validation)

print()
print("Fit status counts:")
display(fit_log["fit_status"].value_counts(dropna=False).to_frame("count"))

print()
print("Overall recommendation:")
display(overall_recommendation)

print()
print("Bucket recommendation diagnostics:")
display(unified_recommendation)

print()
print("Unified vs existing forecast metrics by bucket:")
bucket_display_cols = [
    "model_spec",
    "tenor_bucket",
    "forecast_repair_scope",
    "rows",
    "rmse_log",
    "mae_log",
    "bias_log_pred_minus_actual",
    "underforecast_rate",
    "large_underforecast_1p5x_rate",
    "large_underforecast_2p0x_rate",
    "forecast_top_decile_capture_rate",
    "missed_top_decile_actual_rate",
    "underforecast_rate_in_actual_top_decile",
    "candidate_forecast_vol_pct_mean",
    "existing_rmse_log",
    "existing_forecast_top_decile_capture_rate",
    "capture_improvement_vs_existing",
    "missed_top_decile_reduction_vs_existing",
    "rmse_change_vs_existing",
    "large_underforecast_2p0x_change_vs_existing",
    "forecast_vol_mean_change_vs_existing",
]
display(
    forecast_metrics_by_bucket_vs_existing
    .sort_values(["tenor_bucket", "model_spec"])
    .loc[:, bucket_display_cols]
)

print()
print("Unified vs existing forecast metrics by tenor:")
tenor_display_cols = [
    "model_spec",
    "tenor",
    "tenor_bucket",
    "forecast_repair_scope",
    "rows",
    "rmse_log",
    "mae_log",
    "bias_log_pred_minus_actual",
    "underforecast_rate",
    "large_underforecast_2p0x_rate",
    "forecast_top_decile_capture_rate",
    "missed_top_decile_actual_rate",
    "candidate_forecast_vol_pct_mean",
    "existing_forecast_top_decile_capture_rate",
    "capture_improvement_vs_existing",
    "rmse_change_vs_existing",
    "forecast_vol_mean_change_vs_existing",
]
display(
    forecast_metrics_by_tenor_vs_existing
    .sort_values(["tenor", "model_spec"])
    .loc[:, tenor_display_cols]
)

if not prior_front_middle_metrics_vs_existing.empty:
    print()
    print("Front/Middle prior candidate comparison, common rows where available:")
    prior_display_cols = [
        "model_spec",
        "tenor_bucket",
        "forecast_repair_scope",
        "rows",
        "rmse_log",
        "mae_log",
        "bias_log_pred_minus_actual",
        "large_underforecast_2p0x_rate",
        "forecast_top_decile_capture_rate",
        "missed_top_decile_actual_rate",
        "candidate_forecast_vol_pct_mean",
        "capture_improvement_vs_existing",
        "rmse_change_vs_existing",
        "forecast_vol_mean_change_vs_existing",
    ]
    display(
        prior_front_middle_metrics_vs_existing
        .sort_values(["tenor_bucket", "capture_improvement_vs_existing", "rmse_change_vs_existing"], ascending=[True, False, True])
        .loc[:, prior_display_cols]
    )

print()
print("Diagnostic Core threshold pass rates:")
display(signal_pass_diagnostics)

print()
print("Trade summary by bucket vs existing:")
trade_display_cols = [
    "selection_universe",
    "tenor_bucket",
    "model_spec",
    "comparison_label",
    "trades",
    "win_rate",
    "profit_factor",
    "total_pnl",
    "total_pnl_change_frac_vs_existing",
    "worst_trade",
    "selected_trade_drawdown",
    "worst_20_trade_sum",
    "trades_le_neg_100k",
    "pnl_2025",
    "pnl_2025_change_frac_vs_existing",
    "trades_2025",
    "worst_trade_2025",
    "trades_change_vs_existing",
    "win_rate_change_vs_existing",
    "total_pnl_change_vs_existing",
    "profit_factor_change_vs_existing",
    "worst_trade_change_vs_existing",
    "drawdown_change_vs_existing",
    "trades_le_neg_100k_change_vs_existing",
    "pnl_2025_change_vs_existing",
]
display(
    trade_summary_vs_existing
    .sort_values(["selection_universe", "tenor_bucket", "model_spec"])
    .loc[:, [c for c in trade_display_cols if c in trade_summary_vs_existing.columns]]
)

print()
print("Saved outputs:")
saved_paths = [
    forecast_panel_path,
    common_panel_path,
    signal_panel_path,
    qualified_panel_path,
    forecast_metrics_by_bucket_path,
    forecast_metrics_by_tenor_path,
    forecast_metrics_by_year_path,
    forecast_metrics_by_bucket_vs_existing_path,
    forecast_metrics_by_tenor_vs_existing_path,
    forecast_metrics_by_year_vs_existing_path,
    prior_front_middle_metrics_path,
    trade_summary_path,
    trade_summary_vs_existing_path,
    signal_pass_diagnostics_path,
    threshold_table_path,
    unified_recommendation_path,
    overall_recommendation_path,
    fit_log_path,
    inner_cv_path,
    validation_path,
    manifest_path,
]

for p in saved_paths:
    print(f"  {p}")

print("\nCELL 7A PASSED")

In [ ]:
# ======================================================================================
# Cell 7B — Unified denominator visual sanity checks with completed-forward guard
# ======================================================================================
#
# Objective:
#   Regenerate the unified denominator visual sanity-check plots, but only plot
#   forward realized vol when the forward realized window is complete.
#
# Fix versus first Cell 7B:
#   The latest date may have target_realized_variance populated mechanically even when
#   the forward window has not completed. This cell uses:
#
#       forward_realized_complete = last_forward_rv_date <= max_available_trade_date
#
#   and suppresses the realized line for incomplete tenors.
#
# Scope:
#   - Load latest Cell 7A unified forecast panel.
#   - Select current snapshot and relevant completed-forward diagnostic dates.
#   - Save PNG plots and audit CSVs.
#
# Explicitly not in scope:
#   - No model fitting.
#   - No threshold sweep.
#   - No signal changes.
#   - No parameter changes.
#   - No production lock.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ======================================================================================
# 0. Configuration
# ======================================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
BRANCH = "vrp_front_middle_corsi_forecast_repair_v1"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / BRANCH
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / BRANCH
CHART_DIR = AUDIT_DIR / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

UNIFIED_SPEC = "unified_fds_no_min_return"
EXISTING_SPEC = "existing_har_downside_v1"

ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

TENOR_BUCKET_MAP = {
    9: "Front",
    12: "Front",
    15: "Front",
    18: "Middle",
    21: "Middle",
    24: "Middle",
    27: "Back",
    30: "Back",
    33: "Back",
}

TARGET_VAR_COL = "target_realized_variance"
TARGET_LOG_COL = "target_log_variance"

HARD_CODED_DATES_TO_INCLUDE_IF_PRESENT = {
    "covid_crash_down_day": "2020-03-16",
    "covid_high_vol_aftermath": "2020-03-24",
    "known_selected_trade_test": "2026-03-30",
}

MAX_PLOTS = 12

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def safe_label(s: str) -> str:
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:90]


def add_validation(rows: list[dict], check: str, status: str, detail: str):
    rows.append({"check": check, "status": status, "detail": detail})


def annualized_vol_pct_from_variance(x: pd.Series) -> pd.Series:
    return np.sqrt(pd.to_numeric(x, errors="coerce").clip(lower=0)) * 100.0


def build_date_point_panel(df: pd.DataFrame, max_available_trade_date: pd.Timestamp) -> pd.DataFrame:
    """
    Build one row per trade_date × tenor for plotting, using:
      - unified forecast
      - existing forecast reference
      - implied variance
      - forward realized variance target

    Forward realized values are kept in raw columns, but plotting columns are set
    to NaN unless last_forward_rv_date <= max_available_trade_date.
    """
    unified = df[df["model_spec"] == UNIFIED_SPEC].copy()
    existing = df[df["model_spec"] == EXISTING_SPEC].copy()

    unified_cols = [
        "trade_date",
        "tenor",
        "implied_variance",
        TARGET_VAR_COL,
        "forecast_variance_candidate",
        "candidate_forecast_vol_pct",
    ]

    if "last_forward_rv_date" not in unified.columns:
        raise KeyError(
            "Cell 7B requires last_forward_rv_date to determine whether "
            "forward realized vol is complete."
        )

    unified_cols.append("last_forward_rv_date")

    optional_cols = ["normalized_pnl_dollars", "win"]
    for c in optional_cols:
        if c in unified.columns:
            unified_cols.append(c)

    unified_plot = unified[unified_cols].copy()
    unified_plot = unified_plot.rename(
        columns={
            "forecast_variance_candidate": "unified_forecast_variance",
            "candidate_forecast_vol_pct": "unified_forecast_vol_pct",
        }
    )

    existing_plot = existing[
        [
            "trade_date",
            "tenor",
            "forecast_variance_candidate",
            "candidate_forecast_vol_pct",
        ]
    ].copy()

    existing_plot = existing_plot.rename(
        columns={
            "forecast_variance_candidate": "existing_forecast_variance",
            "candidate_forecast_vol_pct": "existing_forecast_vol_pct",
        }
    )

    out = unified_plot.merge(
        existing_plot,
        on=["trade_date", "tenor"],
        how="left",
        validate="one_to_one",
    )

    out["tenor"] = pd.to_numeric(out["tenor"], errors="coerce").astype("Int64")
    out["tenor_bucket"] = out["tenor"].astype(int).map(TENOR_BUCKET_MAP)

    out["implied_vol_pct"] = annualized_vol_pct_from_variance(out["implied_variance"])
    out["forward_realized_vol_pct_raw"] = annualized_vol_pct_from_variance(out[TARGET_VAR_COL])

    out["forward_realized_complete"] = (
        out["last_forward_rv_date"].notna()
        & (out["last_forward_rv_date"] <= max_available_trade_date)
    )

    out["forward_realized_vol_pct_plot"] = np.where(
        out["forward_realized_complete"],
        out["forward_realized_vol_pct_raw"],
        np.nan,
    )

    out["target_realized_variance_plot"] = np.where(
        out["forward_realized_complete"],
        out[TARGET_VAR_COL],
        np.nan,
    )

    out["implied_minus_forecast_vol_pts"] = out["implied_vol_pct"] - out["unified_forecast_vol_pct"]
    out["realized_minus_forecast_vol_pts_plot"] = out["forward_realized_vol_pct_plot"] - out["unified_forecast_vol_pct"]
    out["existing_minus_unified_forecast_vol_pts"] = out["existing_forecast_vol_pct"] - out["unified_forecast_vol_pct"]

    out["implied_to_forecast_var_ratio"] = out["implied_variance"] / out["unified_forecast_variance"]
    out["realized_to_forecast_var_ratio_plot"] = out["target_realized_variance_plot"] / out["unified_forecast_variance"]
    out["existing_to_unified_forecast_var_ratio"] = out["existing_forecast_variance"] / out["unified_forecast_variance"]

    out["abs_realized_minus_forecast_vol_pts_plot"] = out["realized_minus_forecast_vol_pts_plot"].abs()
    out["abs_implied_minus_forecast_vol_pts"] = out["implied_minus_forecast_vol_pts"].abs()

    return out


def dates_with_all_tenors_and_required_cols(
    point_panel: pd.DataFrame,
    required_cols: list[str],
    require_completed_realized: bool = False,
) -> pd.Series:
    rows = []

    for trade_date, g in point_panel.groupby("trade_date"):
        tenors = set(g["tenor"].dropna().astype(int).tolist())
        has_all_tenors = tenors == set(ALL_TENORS)

        required_ok = True
        for col in required_cols:
            required_ok = required_ok and g[col].notna().all()

        realized_complete_ok = True
        if require_completed_realized:
            realized_complete_ok = g["forward_realized_complete"].all()

        rows.append({
            "trade_date": trade_date,
            "has_all_tenors": has_all_tenors,
            "required_ok": required_ok,
            "realized_complete_ok": realized_complete_ok,
            "include": has_all_tenors and required_ok and realized_complete_ok,
        })

    status = pd.DataFrame(rows)
    return status[status["include"]]["trade_date"]


def build_visual_date_metrics(point_panel: pd.DataFrame) -> pd.DataFrame:
    metrics = (
        point_panel.groupby("trade_date", as_index=False)
        .agg(
            tenor_count=("tenor", lambda x: len(set(pd.to_numeric(x, errors="coerce").dropna().astype(int)))),
            completed_forward_tenor_count=("forward_realized_complete", "sum"),
            avg_implied_vol_pct=("implied_vol_pct", "mean"),
            avg_unified_forecast_vol_pct=("unified_forecast_vol_pct", "mean"),
            avg_existing_forecast_vol_pct=("existing_forecast_vol_pct", "mean"),
            avg_forward_realized_vol_pct_completed=("forward_realized_vol_pct_plot", "mean"),
            avg_forward_realized_vol_pct_raw=("forward_realized_vol_pct_raw", "mean"),
            avg_implied_minus_forecast_vol_pts=("implied_minus_forecast_vol_pts", "mean"),
            avg_realized_minus_forecast_vol_pts_completed=("realized_minus_forecast_vol_pts_plot", "mean"),
            avg_abs_implied_minus_forecast_vol_pts=("abs_implied_minus_forecast_vol_pts", "mean"),
            avg_abs_realized_minus_forecast_vol_pts_completed=("abs_realized_minus_forecast_vol_pts_plot", "mean"),
            avg_implied_to_forecast_var_ratio=("implied_to_forecast_var_ratio", "mean"),
            avg_realized_to_forecast_var_ratio_completed=("realized_to_forecast_var_ratio_plot", "mean"),
            latest_last_forward_rv_date=("last_forward_rv_date", "max"),
        )
    )

    metrics["avg_forecast_minus_implied_vol_pts"] = -metrics["avg_implied_minus_forecast_vol_pts"]
    metrics["all_forward_realized_complete"] = metrics["completed_forward_tenor_count"] == len(ALL_TENORS)

    return metrics


def select_visual_dates(point_panel: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Select:
      - latest current snapshot using implied/forecast only
      - latest completed forward-realized date
      - high implied vol
      - high completed realized vol
      - largest implied - forecast gap
      - largest forecast - implied gap
      - largest completed realized - forecast miss
      - calm date
      - hard-coded dates if present
    """
    required_for_current = [
        "implied_vol_pct",
        "unified_forecast_vol_pct",
        "unified_forecast_variance",
        "implied_variance",
    ]

    required_for_realized = required_for_current + [
        "forward_realized_vol_pct_plot",
        "target_realized_variance_plot",
    ]

    current_complete_dates = dates_with_all_tenors_and_required_cols(
        point_panel,
        required_cols=required_for_current,
        require_completed_realized=False,
    )

    realized_complete_dates = dates_with_all_tenors_and_required_cols(
        point_panel,
        required_cols=required_for_realized,
        require_completed_realized=True,
    )

    if current_complete_dates.empty:
        raise RuntimeError("No complete all-tenor dates found with implied and unified forecast values.")

    date_metrics = build_visual_date_metrics(point_panel)
    current_metrics = date_metrics[date_metrics["trade_date"].isin(current_complete_dates)].copy()
    realized_metrics = date_metrics[date_metrics["trade_date"].isin(realized_complete_dates)].copy()

    selected_rows = []

    def add_date(label: str, trade_date, reason: str):
        if pd.isna(trade_date):
            return
        selected_rows.append({
            "label": label,
            "trade_date": pd.Timestamp(trade_date).normalize(),
            "reason": reason,
        })

    add_date(
        "latest_current_snapshot",
        current_complete_dates.max(),
        "Latest date with all nine tenors and implied/unified forecast available. Forward realized may be incomplete and is suppressed if incomplete.",
    )

    if not realized_complete_dates.empty:
        add_date(
            "latest_completed_forward_realized",
            realized_complete_dates.max(),
            "Latest date with all nine tenors and completed forward realized target.",
        )

    if not current_metrics.empty:
        add_date(
            "highest_average_implied_vol",
            current_metrics.sort_values("avg_implied_vol_pct", ascending=False).iloc[0]["trade_date"],
            "Highest average implied-vol term structure.",
        )

        add_date(
            "largest_implied_minus_forecast_gap",
            current_metrics.sort_values("avg_implied_minus_forecast_vol_pts", ascending=False).iloc[0]["trade_date"],
            "Largest average implied vol minus unified forecast vol.",
        )

        add_date(
            "largest_forecast_minus_implied_gap",
            current_metrics.sort_values("avg_forecast_minus_implied_vol_pts", ascending=False).iloc[0]["trade_date"],
            "Largest average unified forecast vol minus implied vol.",
        )

        add_date(
            "lowest_average_implied_vol",
            current_metrics.sort_values("avg_implied_vol_pct", ascending=True).iloc[0]["trade_date"],
            "Lowest average implied-vol calm-regime date.",
        )

    if not realized_metrics.empty:
        add_date(
            "highest_average_completed_forward_realized_vol",
            realized_metrics.sort_values("avg_forward_realized_vol_pct_completed", ascending=False).iloc[0]["trade_date"],
            "Highest average completed forward realized-vol target date.",
        )

        add_date(
            "largest_completed_realized_minus_forecast_miss",
            realized_metrics.sort_values("avg_realized_minus_forecast_vol_pts_completed", ascending=False).iloc[0]["trade_date"],
            "Largest average completed forward realized vol minus unified forecast vol.",
        )

        add_date(
            "largest_abs_completed_realized_forecast_miss",
            realized_metrics.sort_values("avg_abs_realized_minus_forecast_vol_pts_completed", ascending=False).iloc[0]["trade_date"],
            "Largest average absolute completed realized-vs-forecast miss.",
        )

    available_dates = set(point_panel["trade_date"].dropna().dt.normalize().unique())

    for label, date_str in HARD_CODED_DATES_TO_INCLUDE_IF_PRESENT.items():
        dt = pd.Timestamp(date_str).normalize()
        if dt in available_dates:
            add_date(
                label,
                dt,
                f"Hard-coded diagnostic date included if present: {date_str}.",
            )

    selected = pd.DataFrame(selected_rows)

    if selected.empty:
        raise RuntimeError("No visual dates selected.")

    selected = (
        selected.groupby("trade_date", as_index=False)
        .agg(
            label=("label", lambda x: "__".join(sorted(set(x)))),
            reason=("reason", lambda x: " | ".join(sorted(set(x)))),
        )
        .sort_values("trade_date", ascending=False)
    )

    selected = selected.head(MAX_PLOTS).copy()
    selected = selected.merge(date_metrics, on="trade_date", how="left")
    selected["date_str"] = selected["trade_date"].dt.strftime("%Y-%m-%d")

    return selected, date_metrics


def plot_date_term_structure(
    point_panel: pd.DataFrame,
    trade_date: pd.Timestamp,
    label: str,
    output_dir: Path,
    max_available_trade_date: pd.Timestamp,
) -> Path:
    d = point_panel[point_panel["trade_date"] == trade_date].copy()
    d = d[d["tenor"].isin(ALL_TENORS)].copy()
    d = d.sort_values("tenor")

    if d.empty:
        raise RuntimeError(f"No plot data found for {trade_date}")

    date_str = pd.Timestamp(trade_date).strftime("%Y-%m-%d")
    label_safe = safe_label(label)
    out_path = output_dir / f"07B_unified_term_structure_{label_safe}_{date_str}_{TIMESTAMP}.png"

    x = d["tenor"].astype(int).to_numpy()

    completed_count = int(d["forward_realized_complete"].sum())
    all_completed = completed_count == len(ALL_TENORS)
    any_completed = completed_count > 0

    if all_completed:
        realized_status = "forward realized complete"
    elif any_completed:
        realized_status = f"forward realized partially complete ({completed_count}/9 tenors)"
    else:
        realized_status = "forward realized incomplete / suppressed"

    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(13.5, 9.5),
        sharex=True,
        gridspec_kw={"height_ratios": [2.0, 1.15]},
    )

    ax = axes[0]

    ax.plot(
        x,
        d["implied_vol_pct"],
        marker="o",
        linewidth=2.5,
        label="Implied vol",
    )
    ax.plot(
        x,
        d["unified_forecast_vol_pct"],
        marker="o",
        linewidth=2.5,
        label="Unified forecast vol",
    )

    if any_completed:
        ax.plot(
            x,
            d["forward_realized_vol_pct_plot"],
            marker="o",
            linewidth=2.5,
            label="Forward realized vol, completed tenors only",
        )

    if d["existing_forecast_vol_pct"].notna().any():
        ax.plot(
            x,
            d["existing_forecast_vol_pct"],
            marker="o",
            linewidth=1.8,
            linestyle="--",
            label="Existing HAR/downside forecast vol",
        )

    ax.axvline(16.5, linewidth=1.0, linestyle=":", alpha=0.55)
    ax.axvline(25.5, linewidth=1.0, linestyle=":", alpha=0.55)

    ax.set_title(
        f"Unified denominator sanity check — {date_str}\n"
        f"{label.replace('__', ' / ')} — {realized_status}",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_ylabel("Annualized vol %")
    ax.grid(True, alpha=0.30)
    ax.legend(loc="best")

    y_top = ax.get_ylim()[1]
    ax.text(12, y_top * 0.98, "Front", ha="center", va="top", fontsize=9, alpha=0.75)
    ax.text(21, y_top * 0.98, "Middle", ha="center", va="top", fontsize=9, alpha=0.75)
    ax.text(30, y_top * 0.98, "Back", ha="center", va="top", fontsize=9, alpha=0.75)

    ax2 = axes[1]

    ax2.plot(
        x,
        d["implied_to_forecast_var_ratio"],
        marker="o",
        linewidth=2.2,
        label="Implied variance / forecast variance",
    )

    if any_completed:
        ax2.plot(
            x,
            d["realized_to_forecast_var_ratio_plot"],
            marker="o",
            linewidth=2.2,
            label="Realized variance / forecast variance, completed tenors only",
        )

    if d["existing_to_unified_forecast_var_ratio"].notna().any():
        ax2.plot(
            x,
            d["existing_to_unified_forecast_var_ratio"],
            marker="o",
            linewidth=1.6,
            linestyle="--",
            label="Existing forecast variance / unified forecast variance",
        )

    ax2.axhline(1.0, linewidth=1.0, linestyle=":", alpha=0.70)
    ax2.axvline(16.5, linewidth=1.0, linestyle=":", alpha=0.55)
    ax2.axvline(25.5, linewidth=1.0, linestyle=":", alpha=0.55)

    ax2.set_xlabel("Tenor")
    ax2.set_ylabel("Variance ratio")
    ax2.set_xticks(ALL_TENORS)
    ax2.grid(True, alpha=0.30)
    ax2.legend(loc="best")

    avg_imp = d["implied_vol_pct"].mean()
    avg_fcst = d["unified_forecast_vol_pct"].mean()
    avg_real = d["forward_realized_vol_pct_plot"].mean()

    if np.isfinite(avg_real):
        realized_footer = f"completed forward realized: {avg_real:.2f}%"
    else:
        realized_footer = "completed forward realized: unavailable"

    footer = (
        f"Average vol — implied: {avg_imp:.2f}%, "
        f"forecast: {avg_fcst:.2f}%, "
        f"{realized_footer}. "
        f"Max available trade date: {max_available_trade_date:%Y-%m-%d}"
    )

    fig.text(0.01, 0.01, footer, fontsize=9, alpha=0.75)

    plt.tight_layout(rect=(0, 0.025, 1, 1))
    fig.savefig(out_path, dpi=160)
    plt.close(fig)

    return out_path


# ======================================================================================
# 2. Load latest Cell 7A unified panel
# ======================================================================================

source_panel_path = latest_file(
    PROCESSED_DIR,
    "07A_unified_fds_no_min_return_oos_forecast_panel_*.parquet",
    required=True,
)

panel = pd.read_parquet(source_panel_path)

duplicate_cols = panel.columns[panel.columns.duplicated()].tolist()
if duplicate_cols:
    print("Dropping duplicate column labels in Cell 7B source panel:")
    print(sorted(set(duplicate_cols)))

panel = panel.loc[:, ~panel.columns.duplicated(keep="last")].copy()

panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce").dt.normalize()
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype("Int64")

if "last_forward_rv_date" not in panel.columns:
    raise KeyError("Cell 7B requires last_forward_rv_date in the Cell 7A panel.")

panel["last_forward_rv_date"] = pd.to_datetime(panel["last_forward_rv_date"], errors="coerce").dt.normalize()

required_cols = [
    "trade_date",
    "tenor",
    "model_spec",
    "last_forward_rv_date",
    "implied_variance",
    TARGET_VAR_COL,
    "forecast_variance_candidate",
    "candidate_forecast_vol_pct",
]

missing_cols = [c for c in required_cols if c not in panel.columns]
if missing_cols:
    raise KeyError(f"Cell 7B missing required columns from Cell 7A panel: {missing_cols}")

for c in [
    "implied_variance",
    TARGET_VAR_COL,
    "forecast_variance_candidate",
    "candidate_forecast_vol_pct",
]:
    panel[c] = pd.to_numeric(panel[c], errors="coerce")

panel = panel[
    panel["model_spec"].isin([UNIFIED_SPEC, EXISTING_SPEC])
    & panel["tenor"].isin(ALL_TENORS)
    & panel["trade_date"].notna()
].copy()

if panel.empty:
    raise RuntimeError("Cell 7B loaded no usable rows from the Cell 7A panel.")

max_available_trade_date = panel["trade_date"].max()

# ======================================================================================
# 3. Build point panel and select dates
# ======================================================================================

point_panel = build_date_point_panel(
    df=panel,
    max_available_trade_date=max_available_trade_date,
)

selected_dates, visual_date_metrics = select_visual_dates(point_panel)

selected_point_panel = point_panel.merge(
    selected_dates[["trade_date", "label"]],
    on="trade_date",
    how="inner",
    validate="many_to_one",
).copy()

# ======================================================================================
# 4. Plot selected dates
# ======================================================================================

chart_rows = []

for _, row in selected_dates.sort_values("trade_date").iterrows():
    trade_date = pd.Timestamp(row["trade_date"]).normalize()
    label = row["label"]

    try:
        chart_path = plot_date_term_structure(
            point_panel=point_panel,
            trade_date=trade_date,
            label=label,
            output_dir=CHART_DIR,
            max_available_trade_date=max_available_trade_date,
        )

        chart_rows.append({
            "trade_date": trade_date,
            "date_str": trade_date.strftime("%Y-%m-%d"),
            "label": label,
            "completed_forward_tenor_count": int(
                point_panel.loc[point_panel["trade_date"] == trade_date, "forward_realized_complete"].sum()
            ),
            "chart_path": str(chart_path),
            "status": "SAVED",
        })

    except Exception as e:
        chart_rows.append({
            "trade_date": trade_date,
            "date_str": trade_date.strftime("%Y-%m-%d"),
            "label": label,
            "completed_forward_tenor_count": np.nan,
            "chart_path": None,
            "status": "ERROR",
            "error": f"{type(e).__name__}: {e}",
        })

chart_manifest = pd.DataFrame(chart_rows)

# ======================================================================================
# 5. Save audit outputs
# ======================================================================================

selected_dates_path = AUDIT_DIR / f"07B_selected_visual_dates_{TIMESTAMP}.csv"
visual_date_metrics_path = AUDIT_DIR / f"07B_visual_date_metrics_{TIMESTAMP}.csv"
selected_point_panel_path = AUDIT_DIR / f"07B_selected_visual_point_panel_{TIMESTAMP}.csv"
chart_manifest_path = AUDIT_DIR / f"07B_chart_manifest_{TIMESTAMP}.csv"
validation_path = AUDIT_DIR / f"07B_validation_{TIMESTAMP}.csv"
manifest_path = AUDIT_DIR / f"07B_manifest_{TIMESTAMP}.json"

selected_dates.to_csv(selected_dates_path, index=False)
visual_date_metrics.to_csv(visual_date_metrics_path, index=False)
selected_point_panel.to_csv(selected_point_panel_path, index=False)
chart_manifest.to_csv(chart_manifest_path, index=False)

# ======================================================================================
# 6. Validation
# ======================================================================================

validation_rows = []

chart_saved_count = int(chart_manifest["status"].eq("SAVED").sum())
chart_error_count = int(chart_manifest["status"].eq("ERROR").sum())

unique_tenors_in_points = sorted(point_panel["tenor"].dropna().astype(int).unique().tolist())
missing_tenors = sorted(set(ALL_TENORS) - set(unique_tenors_in_points))

latest_date = max_available_trade_date
latest_rows = point_panel[point_panel["trade_date"] == latest_date].copy()
latest_completed_count = int(latest_rows["forward_realized_complete"].sum()) if not latest_rows.empty else 0

current_snapshot_count = selected_dates["label"].str.contains("latest_current_snapshot", na=False).sum()
completed_snapshot_count = selected_dates["label"].str.contains("latest_completed_forward_realized", na=False).sum()

incomplete_realized_suppressed_rows = int(
    point_panel["forward_realized_vol_pct_raw"].notna().sum()
    - point_panel["forward_realized_vol_pct_plot"].notna().sum()
)

add_validation(
    validation_rows,
    "source_panel_loaded",
    "PASS",
    f"path={source_panel_path}; rows={len(panel):,}",
)
add_validation(
    validation_rows,
    "max_available_trade_date",
    "PASS",
    f"{max_available_trade_date:%Y-%m-%d}",
)
add_validation(
    validation_rows,
    "unified_and_existing_present",
    "PASS" if set([UNIFIED_SPEC, EXISTING_SPEC]).issubset(set(panel["model_spec"].unique())) else "FAIL",
    f"models={sorted(panel['model_spec'].dropna().unique().tolist())}",
)
add_validation(
    validation_rows,
    "all_tenors_present",
    "PASS" if not missing_tenors else "FAIL",
    f"missing_tenors={missing_tenors}",
)
add_validation(
    validation_rows,
    "selected_dates_nonempty",
    "PASS" if len(selected_dates) > 0 else "FAIL",
    f"selected_dates={len(selected_dates):,}",
)
add_validation(
    validation_rows,
    "latest_current_snapshot_selected",
    "PASS" if current_snapshot_count >= 1 else "FAIL",
    f"latest_current_snapshot_count={current_snapshot_count}",
)
add_validation(
    validation_rows,
    "latest_completed_forward_realized_selected",
    "PASS" if completed_snapshot_count >= 1 else "WARN",
    f"latest_completed_forward_realized_count={completed_snapshot_count}",
)
add_validation(
    validation_rows,
    "latest_forward_completion_count",
    "PASS",
    f"latest_date={latest_date:%Y-%m-%d}; completed_forward_tenors={latest_completed_count}/9",
)
add_validation(
    validation_rows,
    "incomplete_forward_realized_suppressed",
    "PASS",
    f"suppressed_rows={incomplete_realized_suppressed_rows:,}",
)
add_validation(
    validation_rows,
    "charts_saved",
    "PASS" if chart_saved_count > 0 else "FAIL",
    f"saved={chart_saved_count}; errors={chart_error_count}",
)
add_validation(
    validation_rows,
    "no_model_fitting",
    "PASS",
    "Cell 7B only reads Cell 7A output and creates plots.",
)
add_validation(
    validation_rows,
    "no_threshold_or_signal_changes",
    "PASS",
    "No threshold, signal, or parameter changes performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"] == "FAIL"]

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "cell": "Cell 7B — Unified denominator visual sanity checks with completed-forward guard",
            "timestamp": TIMESTAMP,
            "source_panel": str(source_panel_path),
            "unified_spec": UNIFIED_SPEC,
            "existing_spec": EXISTING_SPEC,
            "all_tenors": ALL_TENORS,
            "max_available_trade_date": f"{max_available_trade_date:%Y-%m-%d}",
            "hard_coded_dates_to_include_if_present": HARD_CODED_DATES_TO_INCLUDE_IF_PRESENT,
            "selected_dates_path": str(selected_dates_path),
            "visual_date_metrics_path": str(visual_date_metrics_path),
            "selected_point_panel_path": str(selected_point_panel_path),
            "chart_manifest_path": str(chart_manifest_path),
            "validation_path": str(validation_path),
            "notes": [
                "Plotting-only cell.",
                "No fitting performed.",
                "No threshold or signal changes performed.",
                "Forward realized vol is plotted only when last_forward_rv_date <= max_available_trade_date.",
                "Latest current snapshot does not require completed forward realized values.",
            ],
        },
        f,
        indent=2,
    )

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 7B validation failed. See validation table above.")

# ======================================================================================
# 7. Display concise review
# ======================================================================================

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 320)

print("=" * 100)
print("Cell 7B — Unified denominator visual sanity checks with completed-forward guard")
print("=" * 100)
print(f"Source Cell 7A panel:             {source_panel_path}")
print(f"Unified model spec:               {UNIFIED_SPEC}")
print(f"Existing reference spec:          {EXISTING_SPEC}")
print(f"Max available trade date:         {max_available_trade_date:%Y-%m-%d}")
print(f"Chart output directory:           {CHART_DIR}")
print(f"Selected dates:                   {len(selected_dates):,}")
print(f"Charts saved:                     {chart_saved_count:,}")
print(f"Chart errors:                     {chart_error_count:,}")
print(f"Latest completed forward tenors:  {latest_completed_count}/9")
print(f"Suppressed incomplete RV rows:    {incomplete_realized_suppressed_rows:,}")
print("No model fitting:                 True")
print("No threshold changes:             True")
print("No signal changes:                True")

print()
print("Validation:")
display(validation)

print()
print("Selected visual dates:")
display(
    selected_dates[
        [
            "date_str",
            "label",
            "reason",
            "completed_forward_tenor_count",
            "all_forward_realized_complete",
            "avg_implied_vol_pct",
            "avg_unified_forecast_vol_pct",
            "avg_forward_realized_vol_pct_completed",
            "avg_forward_realized_vol_pct_raw",
            "avg_implied_minus_forecast_vol_pts",
            "avg_realized_minus_forecast_vol_pts_completed",
            "avg_implied_to_forecast_var_ratio",
            "avg_realized_to_forecast_var_ratio_completed",
            "latest_last_forward_rv_date",
        ]
    ].sort_values("date_str", ascending=False)
)

print()
print("Chart manifest:")
display(chart_manifest.sort_values("date_str", ascending=False))

print()
print("Saved audit outputs:")
saved_paths = [
    selected_dates_path,
    visual_date_metrics_path,
    selected_point_panel_path,
    chart_manifest_path,
    validation_path,
    manifest_path,
]
for p in saved_paths:
    print(f"  {p}")

print("\nCELL 7B v2 PASSED")

In [ ]:
# ======================================================================================
# Cell 7C — Unified denominator term-structure dashboard charts
# ======================================================================================
#
# Objective:
#   Create cleaner term-structure dashboard charts for the unified denominator:
#
#       unified_fds_no_min_return
#
# Charts created:
#   1. Current vol term structure:
#        - implied vol current / prior / 5 trading days ago
#        - unified forecast vol current / prior / 5 trading days ago
#
#   2. Current VRP log term structure:
#        - model_vrp_log current / prior / 5 trading days ago
#
#   3. Current z-score term structure:
#        - 3m VRP z-score current
#        - 1y VRP z-score current
#        - bucket threshold references
#
#   4. Current Core threshold overlay:
#        - model_vrp_log vs threshold
#        - 3m z vs threshold
#        - 1y z vs threshold
#
#   5. Selected historical term-structure examples:
#        - implied vol
#        - unified forecast vol
#        - forward realized vol only when completed
#
# Scope:
#   - Load latest Cell 7A unified forecast panel.
#   - Use completed-forward guard from Cell 7B v2.
#   - Save dashboard PNGs and audit CSVs.
#
# Explicitly not in scope:
#   - No model fitting.
#   - No threshold sweep.
#   - No signal changes.
#   - No parameter changes.
#   - No production lock.
#   - No Secondary.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ======================================================================================
# 0. Configuration
# ======================================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
BRANCH = "vrp_front_middle_corsi_forecast_repair_v1"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / BRANCH
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / BRANCH
CHART_DIR = AUDIT_DIR / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

UNIFIED_SPEC = "unified_fds_no_min_return"
EXISTING_SPEC = "existing_har_downside_v1"

ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

TENOR_BUCKET_MAP = {
    9: "Front",
    12: "Front",
    15: "Front",
    18: "Middle",
    21: "Middle",
    24: "Middle",
    27: "Back",
    30: "Back",
    33: "Back",
}

CORE_THRESHOLDS_BY_BUCKET = {
    "Front": {
        "model_vrp_log": 0.60,
        "model_vrp_z_3m": 0.55,
        "model_vrp_z_1y": 0.65,
        "RSI14_max": 70.0,
        "forecast_vol_pct_min": 8.5,
    },
    "Middle": {
        "model_vrp_log": 0.65,
        "model_vrp_z_3m": 0.75,
        "model_vrp_z_1y": 0.65,
        "RSI14_max": 68.0,
        "forecast_vol_pct_min": 8.5,
    },
    "Back": {
        "model_vrp_log": 0.70,
        "model_vrp_z_3m": 0.70,
        "model_vrp_z_1y": 0.70,
        "RSI14_max": 70.0,
        "forecast_vol_pct_min": 8.5,
    },
}

TARGET_VAR_COL = "target_realized_variance"
TARGET_LOG_COL = "target_log_variance"

HARD_CODED_DATES_TO_INCLUDE_IF_PRESENT = {
    "covid_crash_down_day": "2020-03-16",
    "covid_high_vol_aftermath": "2020-03-24",
    "known_selected_trade_test": "2026-03-30",
}

MAX_HISTORICAL_EXAMPLE_PLOTS = 12

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory, pattern, required=True):
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def safe_label(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:90]


def add_validation(rows, check, status, detail):
    rows.append({"check": check, "status": status, "detail": detail})


def annualized_vol_pct_from_variance(x):
    return np.sqrt(pd.to_numeric(x, errors="coerce").clip(lower=0)) * 100.0


def compute_prior_z_by_tenor(df, value_col, window, min_periods):
    out = pd.Series(np.nan, index=df.index, dtype="float64")

    for _, g in df.sort_values(["tenor", "trade_date"]).groupby("tenor"):
        shifted = g[value_col].shift(1)
        roll_mean = shifted.rolling(window=window, min_periods=min_periods).mean()
        roll_std = shifted.rolling(window=window, min_periods=min_periods).std(ddof=1)
        z = (g[value_col] - roll_mean) / roll_std.replace(0.0, np.nan)
        out.loc[g.index] = z

    return out


def build_threshold_by_tenor(metric_name):
    rows = []
    for tenor in ALL_TENORS:
        bucket = TENOR_BUCKET_MAP[tenor]
        rows.append({
            "tenor": tenor,
            "tenor_bucket": bucket,
            "threshold": CORE_THRESHOLDS_BY_BUCKET[bucket][metric_name],
        })
    return pd.DataFrame(rows)


def build_point_panel(df, max_available_trade_date):
    """
    Build one row per trade_date × tenor for the unified model, plus existing forecast reference.
    Forward realized plotting columns are populated only when last_forward_rv_date is complete.
    """
    unified = df[df["model_spec"] == UNIFIED_SPEC].copy()
    existing = df[df["model_spec"] == EXISTING_SPEC].copy()

    if "last_forward_rv_date" not in unified.columns:
        raise KeyError("Cell 7C requires last_forward_rv_date in the Cell 7A panel.")

    unified_cols = [
        "trade_date",
        "tenor",
        "last_forward_rv_date",
        "implied_variance",
        TARGET_VAR_COL,
        "forecast_variance_candidate",
        "candidate_forecast_vol_pct",
    ]

    optional_cols = ["RSI14", "normalized_pnl_dollars", "win"]
    for c in optional_cols:
        if c in unified.columns:
            unified_cols.append(c)

    unified_plot = unified[unified_cols].copy()
    unified_plot = unified_plot.rename(
        columns={
            "forecast_variance_candidate": "unified_forecast_variance",
            "candidate_forecast_vol_pct": "unified_forecast_vol_pct",
        }
    )

    existing_plot = existing[
        [
            "trade_date",
            "tenor",
            "forecast_variance_candidate",
            "candidate_forecast_vol_pct",
        ]
    ].copy()

    existing_plot = existing_plot.rename(
        columns={
            "forecast_variance_candidate": "existing_forecast_variance",
            "candidate_forecast_vol_pct": "existing_forecast_vol_pct",
        }
    )

    out = unified_plot.merge(
        existing_plot,
        on=["trade_date", "tenor"],
        how="left",
        validate="one_to_one",
    )

    out["tenor"] = pd.to_numeric(out["tenor"], errors="coerce").astype("Int64")
    out["tenor_bucket"] = out["tenor"].astype(int).map(TENOR_BUCKET_MAP)

    out["implied_vol_pct"] = annualized_vol_pct_from_variance(out["implied_variance"])
    out["forward_realized_vol_pct_raw"] = annualized_vol_pct_from_variance(out[TARGET_VAR_COL])

    out["forward_realized_complete"] = (
        out["last_forward_rv_date"].notna()
        & (out["last_forward_rv_date"] <= max_available_trade_date)
    )

    out["forward_realized_vol_pct_plot"] = np.where(
        out["forward_realized_complete"],
        out["forward_realized_vol_pct_raw"],
        np.nan,
    )

    out["target_realized_variance_plot"] = np.where(
        out["forward_realized_complete"],
        out[TARGET_VAR_COL],
        np.nan,
    )

    out["model_vrp_log"] = np.log(out["implied_variance"] / out["unified_forecast_variance"])
    out["model_vrp_z_3m"] = compute_prior_z_by_tenor(out, "model_vrp_log", window=63, min_periods=63)
    out["model_vrp_z_1y"] = compute_prior_z_by_tenor(out, "model_vrp_log", window=252, min_periods=252)

    out["implied_minus_forecast_vol_pts"] = out["implied_vol_pct"] - out["unified_forecast_vol_pct"]
    out["realized_minus_forecast_vol_pts_plot"] = out["forward_realized_vol_pct_plot"] - out["unified_forecast_vol_pct"]

    out["implied_to_forecast_var_ratio"] = out["implied_variance"] / out["unified_forecast_variance"]
    out["realized_to_forecast_var_ratio_plot"] = out["target_realized_variance_plot"] / out["unified_forecast_variance"]
    out["existing_to_unified_forecast_var_ratio"] = out["existing_forecast_variance"] / out["unified_forecast_variance"]

    # Add diagnostic threshold values.
    out["threshold_model_vrp_log"] = out["tenor_bucket"].map(
        {k: v["model_vrp_log"] for k, v in CORE_THRESHOLDS_BY_BUCKET.items()}
    )
    out["threshold_model_vrp_z_3m"] = out["tenor_bucket"].map(
        {k: v["model_vrp_z_3m"] for k, v in CORE_THRESHOLDS_BY_BUCKET.items()}
    )
    out["threshold_model_vrp_z_1y"] = out["tenor_bucket"].map(
        {k: v["model_vrp_z_1y"] for k, v in CORE_THRESHOLDS_BY_BUCKET.items()}
    )
    out["threshold_RSI14_max"] = out["tenor_bucket"].map(
        {k: v["RSI14_max"] for k, v in CORE_THRESHOLDS_BY_BUCKET.items()}
    )
    out["threshold_forecast_vol_pct_min"] = out["tenor_bucket"].map(
        {k: v["forecast_vol_pct_min"] for k, v in CORE_THRESHOLDS_BY_BUCKET.items()}
    )

    out["diagnostic_core_pass"] = (
        (out["model_vrp_log"] > out["threshold_model_vrp_log"])
        & (out["model_vrp_z_3m"] > out["threshold_model_vrp_z_3m"])
        & (out["model_vrp_z_1y"] > out["threshold_model_vrp_z_1y"])
        & (out["unified_forecast_vol_pct"] > out["threshold_forecast_vol_pct_min"])
    )

    if "RSI14" in out.columns:
        out["diagnostic_core_pass"] = out["diagnostic_core_pass"] & (
            pd.to_numeric(out["RSI14"], errors="coerce") < out["threshold_RSI14_max"]
        )

    return out


def complete_dates(point_panel, required_cols, require_completed_forward=False):
    rows = []

    for trade_date, g in point_panel.groupby("trade_date"):
        tenors = set(g["tenor"].dropna().astype(int).tolist())
        has_all_tenors = tenors == set(ALL_TENORS)

        required_ok = True
        for col in required_cols:
            required_ok = required_ok and g[col].notna().all()

        completed_ok = True
        if require_completed_forward:
            completed_ok = g["forward_realized_complete"].all()

        rows.append({
            "trade_date": trade_date,
            "has_all_tenors": has_all_tenors,
            "required_ok": required_ok,
            "completed_ok": completed_ok,
            "include": has_all_tenors and required_ok and completed_ok,
        })

    status = pd.DataFrame(rows)
    return status[status["include"]]["trade_date"].sort_values().reset_index(drop=True)


def build_visual_date_metrics(point_panel):
    metrics = (
        point_panel.groupby("trade_date", as_index=False)
        .agg(
            tenor_count=("tenor", lambda x: len(set(pd.to_numeric(x, errors="coerce").dropna().astype(int)))),
            completed_forward_tenor_count=("forward_realized_complete", "sum"),
            avg_implied_vol_pct=("implied_vol_pct", "mean"),
            avg_unified_forecast_vol_pct=("unified_forecast_vol_pct", "mean"),
            avg_existing_forecast_vol_pct=("existing_forecast_vol_pct", "mean"),
            avg_forward_realized_vol_pct_completed=("forward_realized_vol_pct_plot", "mean"),
            avg_forward_realized_vol_pct_raw=("forward_realized_vol_pct_raw", "mean"),
            avg_model_vrp_log=("model_vrp_log", "mean"),
            avg_model_vrp_z_3m=("model_vrp_z_3m", "mean"),
            avg_model_vrp_z_1y=("model_vrp_z_1y", "mean"),
            avg_implied_minus_forecast_vol_pts=("implied_minus_forecast_vol_pts", "mean"),
            avg_realized_minus_forecast_vol_pts_completed=("realized_minus_forecast_vol_pts_plot", "mean"),
            avg_implied_to_forecast_var_ratio=("implied_to_forecast_var_ratio", "mean"),
            avg_realized_to_forecast_var_ratio_completed=("realized_to_forecast_var_ratio_plot", "mean"),
            latest_last_forward_rv_date=("last_forward_rv_date", "max"),
        )
    )

    metrics["avg_forecast_minus_implied_vol_pts"] = -metrics["avg_implied_minus_forecast_vol_pts"]
    metrics["avg_abs_realized_minus_forecast_vol_pts_completed"] = (
        point_panel.assign(abs_realized_miss=point_panel["realized_minus_forecast_vol_pts_plot"].abs())
        .groupby("trade_date")["abs_realized_miss"]
        .mean()
        .reindex(metrics["trade_date"])
        .to_numpy()
    )
    metrics["all_forward_realized_complete"] = metrics["completed_forward_tenor_count"] == len(ALL_TENORS)

    return metrics


def select_current_term_structure_dates(point_panel):
    required = [
        "implied_vol_pct",
        "unified_forecast_vol_pct",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
    ]

    dates = complete_dates(
        point_panel=point_panel,
        required_cols=required,
        require_completed_forward=False,
    )

    if dates.empty:
        raise RuntimeError("No complete all-tenor current term-structure dates found.")

    current_date = dates.iloc[-1]
    prior_date = dates.iloc[-2] if len(dates) >= 2 else pd.NaT
    five_day_date = dates.iloc[-6] if len(dates) >= 6 else pd.NaT

    rows = [
        {
            "date_role": "current",
            "trade_date": current_date,
            "date_str": current_date.strftime("%Y-%m-%d"),
            "available": True,
        },
        {
            "date_role": "prior_trading_day",
            "trade_date": prior_date,
            "date_str": prior_date.strftime("%Y-%m-%d") if pd.notna(prior_date) else None,
            "available": pd.notna(prior_date),
        },
        {
            "date_role": "five_trading_days_ago",
            "trade_date": five_day_date,
            "date_str": five_day_date.strftime("%Y-%m-%d") if pd.notna(five_day_date) else None,
            "available": pd.notna(five_day_date),
        },
    ]

    return pd.DataFrame(rows)


def select_historical_example_dates(point_panel, max_plots=12):
    date_metrics = build_visual_date_metrics(point_panel)

    current_dates = complete_dates(
        point_panel,
        required_cols=["implied_vol_pct", "unified_forecast_vol_pct", "model_vrp_log"],
        require_completed_forward=False,
    )

    realized_dates = complete_dates(
        point_panel,
        required_cols=[
            "implied_vol_pct",
            "unified_forecast_vol_pct",
            "forward_realized_vol_pct_plot",
            "target_realized_variance_plot",
        ],
        require_completed_forward=True,
    )

    selected_rows = []

    def add_date(label, trade_date, reason):
        if pd.isna(trade_date):
            return
        selected_rows.append({
            "label": label,
            "trade_date": pd.Timestamp(trade_date).normalize(),
            "reason": reason,
        })

    if not current_dates.empty:
        add_date(
            "latest_current_snapshot",
            current_dates.max(),
            "Latest current snapshot with implied and forecast term structures.",
        )

    if not realized_dates.empty:
        add_date(
            "latest_completed_forward_realized",
            realized_dates.max(),
            "Latest date with completed forward realized term structure.",
        )

    current_metrics = date_metrics[date_metrics["trade_date"].isin(current_dates)].copy()
    realized_metrics = date_metrics[date_metrics["trade_date"].isin(realized_dates)].copy()

    if not current_metrics.empty:
        add_date(
            "highest_average_implied_vol",
            current_metrics.sort_values("avg_implied_vol_pct", ascending=False).iloc[0]["trade_date"],
            "Highest average implied-vol term structure.",
        )
        add_date(
            "largest_implied_minus_forecast_gap",
            current_metrics.sort_values("avg_implied_minus_forecast_vol_pts", ascending=False).iloc[0]["trade_date"],
            "Largest average implied vol minus unified forecast vol.",
        )
        add_date(
            "largest_forecast_minus_implied_gap",
            current_metrics.sort_values("avg_forecast_minus_implied_vol_pts", ascending=False).iloc[0]["trade_date"],
            "Largest average unified forecast vol minus implied vol.",
        )
        add_date(
            "lowest_average_implied_vol",
            current_metrics.sort_values("avg_implied_vol_pct", ascending=True).iloc[0]["trade_date"],
            "Lowest average implied-vol calm date.",
        )
        add_date(
            "highest_average_vrp_log",
            current_metrics.sort_values("avg_model_vrp_log", ascending=False).iloc[0]["trade_date"],
            "Highest average model_vrp_log term structure.",
        )

    if not realized_metrics.empty:
        add_date(
            "highest_average_completed_forward_realized_vol",
            realized_metrics.sort_values("avg_forward_realized_vol_pct_completed", ascending=False).iloc[0]["trade_date"],
            "Highest average completed forward realized vol.",
        )
        add_date(
            "largest_completed_realized_minus_forecast_miss",
            realized_metrics.sort_values("avg_realized_minus_forecast_vol_pts_completed", ascending=False).iloc[0]["trade_date"],
            "Largest completed forward realized minus forecast miss.",
        )
        add_date(
            "largest_abs_completed_realized_forecast_miss",
            realized_metrics.sort_values("avg_abs_realized_minus_forecast_vol_pts_completed", ascending=False).iloc[0]["trade_date"],
            "Largest absolute completed realized-vs-forecast miss.",
        )

    available_dates = set(point_panel["trade_date"].dropna().dt.normalize().unique())

    for label, date_str in HARD_CODED_DATES_TO_INCLUDE_IF_PRESENT.items():
        dt = pd.Timestamp(date_str).normalize()
        if dt in available_dates:
            add_date(
                label,
                dt,
                f"Hard-coded diagnostic date included if present: {date_str}.",
            )

    selected = pd.DataFrame(selected_rows)
    if selected.empty:
        raise RuntimeError("No historical example dates selected.")

    selected = (
        selected.groupby("trade_date", as_index=False)
        .agg(
            label=("label", lambda x: "__".join(sorted(set(x)))),
            reason=("reason", lambda x: " | ".join(sorted(set(x)))),
        )
        .sort_values("trade_date", ascending=False)
        .head(max_plots)
        .copy()
    )

    selected = selected.merge(date_metrics, on="trade_date", how="left")
    selected["date_str"] = selected["trade_date"].dt.strftime("%Y-%m-%d")

    return selected, date_metrics


def add_bucket_guides(ax):
    ax.axvline(16.5, linewidth=1.0, linestyle=":", alpha=0.55)
    ax.axvline(25.5, linewidth=1.0, linestyle=":", alpha=0.55)

    y_min, y_max = ax.get_ylim()
    y_top = y_max - 0.02 * (y_max - y_min)

    ax.text(12, y_top, "Front", ha="center", va="top", fontsize=9, alpha=0.75)
    ax.text(21, y_top, "Middle", ha="center", va="top", fontsize=9, alpha=0.75)
    ax.text(30, y_top, "Back", ha="center", va="top", fontsize=9, alpha=0.75)


def plot_current_vol_term_structure(point_panel, current_dates, out_dir):
    selected = current_dates[current_dates["available"]].copy()
    role_order = ["current", "prior_trading_day", "five_trading_days_ago"]
    selected["role_order"] = selected["date_role"].map({r: i for i, r in enumerate(role_order)})
    selected = selected.sort_values("role_order")

    current_date = selected[selected["date_role"] == "current"]["trade_date"].iloc[0]
    current_str = current_date.strftime("%Y-%m-%d")

    out_path = out_dir / f"07C_current_vol_term_structure_latest_prior_5d_{current_str}_{TIMESTAMP}.png"

    fig, ax = plt.subplots(figsize=(14, 7.5))

    for _, row in selected.iterrows():
        role = row["date_role"]
        dt = row["trade_date"]
        d = point_panel[point_panel["trade_date"] == dt].sort_values("tenor").copy()
        x = d["tenor"].astype(int).to_numpy()

        label_date = dt.strftime("%Y-%m-%d")
        role_label = role.replace("_", " ")

        implied_label = f"Implied vol — {role_label} ({label_date})"
        forecast_label = f"Forecast vol — {role_label} ({label_date})"

        implied_style = "-" if role == "current" else "--" if role == "prior_trading_day" else ":"
        forecast_style = "-" if role == "current" else "--" if role == "prior_trading_day" else ":"

        ax.plot(
            x,
            d["implied_vol_pct"],
            marker="o",
            linewidth=2.4 if role == "current" else 1.7,
            linestyle=implied_style,
            label=implied_label,
        )

        ax.plot(
            x,
            d["unified_forecast_vol_pct"],
            marker="s",
            linewidth=2.4 if role == "current" else 1.7,
            linestyle=forecast_style,
            label=forecast_label,
        )

    ax.set_title(
        f"Current vol term structure — latest vs prior and 5 trading days ago\nLatest: {current_str}",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Tenor")
    ax.set_ylabel("Annualized vol %")
    ax.set_xticks(ALL_TENORS)
    ax.grid(True, alpha=0.30)
    add_bucket_guides(ax)
    ax.legend(loc="best", fontsize=8)

    plt.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.close(fig)

    return out_path


def plot_current_vrp_log_term_structure(point_panel, current_dates, out_dir):
    selected = current_dates[current_dates["available"]].copy()
    role_order = ["current", "prior_trading_day", "five_trading_days_ago"]
    selected["role_order"] = selected["date_role"].map({r: i for i, r in enumerate(role_order)})
    selected = selected.sort_values("role_order")

    current_date = selected[selected["date_role"] == "current"]["trade_date"].iloc[0]
    current_str = current_date.strftime("%Y-%m-%d")

    out_path = out_dir / f"07C_current_vrp_log_term_structure_latest_prior_5d_{current_str}_{TIMESTAMP}.png"

    fig, ax = plt.subplots(figsize=(14, 7.0))

    for _, row in selected.iterrows():
        role = row["date_role"]
        dt = row["trade_date"]
        d = point_panel[point_panel["trade_date"] == dt].sort_values("tenor").copy()
        x = d["tenor"].astype(int).to_numpy()

        label_date = dt.strftime("%Y-%m-%d")
        role_label = role.replace("_", " ")
        style = "-" if role == "current" else "--" if role == "prior_trading_day" else ":"

        ax.plot(
            x,
            d["model_vrp_log"],
            marker="o",
            linewidth=2.4 if role == "current" else 1.8,
            linestyle=style,
            label=f"model_vrp_log — {role_label} ({label_date})",
        )

    threshold = build_threshold_by_tenor("model_vrp_log")
    ax.plot(
        threshold["tenor"],
        threshold["threshold"],
        linewidth=2.0,
        linestyle="--",
        label="Core threshold by bucket",
    )

    ax.axhline(0.0, linewidth=1.0, linestyle=":", alpha=0.75)

    ax.set_title(
        f"Current VRP log term structure — latest vs prior and 5 trading days ago\nLatest: {current_str}",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Tenor")
    ax.set_ylabel("log(implied variance / forecast variance)")
    ax.set_xticks(ALL_TENORS)
    ax.grid(True, alpha=0.30)
    add_bucket_guides(ax)
    ax.legend(loc="best", fontsize=8)

    plt.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.close(fig)

    return out_path


def plot_current_zscore_term_structure(point_panel, current_dates, out_dir):
    current_date = current_dates.loc[current_dates["date_role"] == "current", "trade_date"].iloc[0]
    current_str = current_date.strftime("%Y-%m-%d")

    d = point_panel[point_panel["trade_date"] == current_date].sort_values("tenor").copy()
    x = d["tenor"].astype(int).to_numpy()

    threshold_3m = build_threshold_by_tenor("model_vrp_z_3m")
    threshold_1y = build_threshold_by_tenor("model_vrp_z_1y")

    out_path = out_dir / f"07C_current_vrp_zscore_term_structure_{current_str}_{TIMESTAMP}.png"

    fig, ax = plt.subplots(figsize=(14, 7.0))

    ax.plot(
        x,
        d["model_vrp_z_3m"],
        marker="o",
        linewidth=2.4,
        label="3m VRP z-score",
    )

    ax.plot(
        x,
        d["model_vrp_z_1y"],
        marker="o",
        linewidth=2.4,
        label="1y VRP z-score",
    )

    ax.plot(
        threshold_3m["tenor"],
        threshold_3m["threshold"],
        linewidth=1.8,
        linestyle="--",
        label="3m z threshold by bucket",
    )

    ax.plot(
        threshold_1y["tenor"],
        threshold_1y["threshold"],
        linewidth=1.8,
        linestyle=":",
        label="1y z threshold by bucket",
    )

    ax.axhline(0.0, linewidth=1.0, linestyle=":", alpha=0.70)

    ax.set_title(
        f"Current VRP z-score term structure — {current_str}",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Tenor")
    ax.set_ylabel("Z-score")
    ax.set_xticks(ALL_TENORS)
    ax.grid(True, alpha=0.30)
    add_bucket_guides(ax)
    ax.legend(loc="best", fontsize=8)

    plt.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.close(fig)

    return out_path


def plot_current_threshold_overlay(point_panel, current_dates, out_dir):
    current_date = current_dates.loc[current_dates["date_role"] == "current", "trade_date"].iloc[0]
    current_str = current_date.strftime("%Y-%m-%d")

    d = point_panel[point_panel["trade_date"] == current_date].sort_values("tenor").copy()
    x = d["tenor"].astype(int).to_numpy()

    out_path = out_dir / f"07C_current_core_threshold_overlay_{current_str}_{TIMESTAMP}.png"

    fig, axes = plt.subplots(
        nrows=3,
        ncols=1,
        figsize=(14, 10.5),
        sharex=True,
    )

    specs = [
        {
            "ax": axes[0],
            "metric": "model_vrp_log",
            "threshold_metric": "model_vrp_log",
            "title": "VRP log",
            "ylabel": "log IV/FV",
        },
        {
            "ax": axes[1],
            "metric": "model_vrp_z_3m",
            "threshold_metric": "model_vrp_z_3m",
            "title": "3m VRP z-score",
            "ylabel": "3m z",
        },
        {
            "ax": axes[2],
            "metric": "model_vrp_z_1y",
            "threshold_metric": "model_vrp_z_1y",
            "title": "1y VRP z-score",
            "ylabel": "1y z",
        },
    ]

    for spec in specs:
        ax = spec["ax"]
        threshold = build_threshold_by_tenor(spec["threshold_metric"])

        ax.plot(
            x,
            d[spec["metric"]],
            marker="o",
            linewidth=2.4,
            label=spec["title"],
        )
        ax.plot(
            threshold["tenor"],
            threshold["threshold"],
            linewidth=2.0,
            linestyle="--",
            label="Core threshold by bucket",
        )
        ax.axhline(0.0, linewidth=1.0, linestyle=":", alpha=0.65)
        ax.set_ylabel(spec["ylabel"])
        ax.grid(True, alpha=0.30)
        add_bucket_guides(ax)
        ax.legend(loc="best", fontsize=8)

    axes[0].set_title(
        f"Current Core threshold overlay — unified denominator — {current_str}",
        fontsize=14,
        fontweight="bold",
    )
    axes[-1].set_xlabel("Tenor")
    axes[-1].set_xticks(ALL_TENORS)

    # Add a compact pass/fail row in the footer.
    pass_tenors = d.loc[d["diagnostic_core_pass"], "tenor"].dropna().astype(int).tolist()
    footer = (
        "Diagnostic Core pass tenors on current date: "
        + (", ".join(map(str, pass_tenors)) if pass_tenors else "none")
    )
    fig.text(0.01, 0.01, footer, fontsize=9, alpha=0.80)

    plt.tight_layout(rect=(0, 0.025, 1, 1))
    fig.savefig(out_path, dpi=160)
    plt.close(fig)

    return out_path


def plot_historical_term_structure(point_panel, trade_date, label, out_dir, max_available_trade_date):
    d = point_panel[point_panel["trade_date"] == trade_date].copy()
    d = d[d["tenor"].isin(ALL_TENORS)].sort_values("tenor").copy()

    if d.empty:
        raise RuntimeError(f"No plot data found for {trade_date}")

    date_str = pd.Timestamp(trade_date).strftime("%Y-%m-%d")
    label_safe = safe_label(label)
    out_path = out_dir / f"07C_selected_date_term_structure_{label_safe}_{date_str}_{TIMESTAMP}.png"

    x = d["tenor"].astype(int).to_numpy()

    completed_count = int(d["forward_realized_complete"].sum())
    all_completed = completed_count == len(ALL_TENORS)
    any_completed = completed_count > 0

    if all_completed:
        realized_status = "forward realized complete"
    elif any_completed:
        realized_status = f"forward realized partially complete ({completed_count}/9 tenors)"
    else:
        realized_status = "forward realized incomplete / suppressed"

    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(13.5, 9.5),
        sharex=True,
        gridspec_kw={"height_ratios": [2.0, 1.15]},
    )

    ax = axes[0]

    ax.plot(
        x,
        d["implied_vol_pct"],
        marker="o",
        linewidth=2.5,
        label="Implied vol",
    )

    ax.plot(
        x,
        d["unified_forecast_vol_pct"],
        marker="o",
        linewidth=2.5,
        label="Unified forecast vol",
    )

    if any_completed:
        ax.plot(
            x,
            d["forward_realized_vol_pct_plot"],
            marker="o",
            linewidth=2.5,
            label="Forward realized vol, completed tenors only",
        )

    if d["existing_forecast_vol_pct"].notna().any():
        ax.plot(
            x,
            d["existing_forecast_vol_pct"],
            marker="o",
            linewidth=1.8,
            linestyle="--",
            label="Existing HAR/downside forecast vol",
        )

    ax.set_title(
        f"Selected date term structure — {date_str}\n"
        f"{label.replace('__', ' / ')} — {realized_status}",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_ylabel("Annualized vol %")
    ax.grid(True, alpha=0.30)
    add_bucket_guides(ax)
    ax.legend(loc="best", fontsize=8)

    ax2 = axes[1]

    ax2.plot(
        x,
        d["model_vrp_log"],
        marker="o",
        linewidth=2.2,
        label="model_vrp_log",
    )

    threshold = build_threshold_by_tenor("model_vrp_log")
    ax2.plot(
        threshold["tenor"],
        threshold["threshold"],
        linewidth=1.8,
        linestyle="--",
        label="Core VRP threshold by bucket",
    )

    ax2.axhline(0.0, linewidth=1.0, linestyle=":", alpha=0.70)

    ax2.set_xlabel("Tenor")
    ax2.set_ylabel("VRP log")
    ax2.set_xticks(ALL_TENORS)
    ax2.grid(True, alpha=0.30)
    add_bucket_guides(ax2)
    ax2.legend(loc="best", fontsize=8)

    avg_imp = d["implied_vol_pct"].mean()
    avg_fcst = d["unified_forecast_vol_pct"].mean()
    avg_real = d["forward_realized_vol_pct_plot"].mean()
    avg_vrp = d["model_vrp_log"].mean()

    if np.isfinite(avg_real):
        realized_footer = f"completed forward realized: {avg_real:.2f}%"
    else:
        realized_footer = "completed forward realized: unavailable"

    footer = (
        f"Average vol — implied: {avg_imp:.2f}%, "
        f"forecast: {avg_fcst:.2f}%, "
        f"{realized_footer}; avg VRP log: {avg_vrp:.3f}. "
        f"Max available trade date: {max_available_trade_date:%Y-%m-%d}"
    )

    fig.text(0.01, 0.01, footer, fontsize=9, alpha=0.75)

    plt.tight_layout(rect=(0, 0.025, 1, 1))
    fig.savefig(out_path, dpi=160)
    plt.close(fig)

    return out_path


# ======================================================================================
# 2. Load latest Cell 7A unified panel
# ======================================================================================

source_panel_path = latest_file(
    PROCESSED_DIR,
    "07A_unified_fds_no_min_return_oos_forecast_panel_*.parquet",
    required=True,
)

panel = pd.read_parquet(source_panel_path)

duplicate_cols = panel.columns[panel.columns.duplicated()].tolist()
if duplicate_cols:
    print("Dropping duplicate column labels in Cell 7C source panel:")
    print(sorted(set(duplicate_cols)))

panel = panel.loc[:, ~panel.columns.duplicated(keep="last")].copy()

panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce").dt.normalize()
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype("Int64")

if "last_forward_rv_date" not in panel.columns:
    raise KeyError("Cell 7C requires last_forward_rv_date in the Cell 7A panel.")

panel["last_forward_rv_date"] = pd.to_datetime(panel["last_forward_rv_date"], errors="coerce").dt.normalize()

required_cols = [
    "trade_date",
    "tenor",
    "model_spec",
    "last_forward_rv_date",
    "implied_variance",
    TARGET_VAR_COL,
    "forecast_variance_candidate",
    "candidate_forecast_vol_pct",
]

missing_cols = [c for c in required_cols if c not in panel.columns]
if missing_cols:
    raise KeyError(f"Cell 7C missing required columns from Cell 7A panel: {missing_cols}")

for c in [
    "implied_variance",
    TARGET_VAR_COL,
    "forecast_variance_candidate",
    "candidate_forecast_vol_pct",
]:
    panel[c] = pd.to_numeric(panel[c], errors="coerce")

panel = panel[
    panel["model_spec"].isin([UNIFIED_SPEC, EXISTING_SPEC])
    & panel["tenor"].isin(ALL_TENORS)
    & panel["trade_date"].notna()
].copy()

if panel.empty:
    raise RuntimeError("Cell 7C loaded no usable rows from the Cell 7A panel.")

max_available_trade_date = panel["trade_date"].max()

# ======================================================================================
# 3. Build term-structure point panel
# ======================================================================================

point_panel = build_point_panel(
    df=panel,
    max_available_trade_date=max_available_trade_date,
)

current_dates = select_current_term_structure_dates(point_panel)
historical_dates, visual_date_metrics = select_historical_example_dates(
    point_panel=point_panel,
    max_plots=MAX_HISTORICAL_EXAMPLE_PLOTS,
)

# ======================================================================================
# 4. Plot current dashboard charts
# ======================================================================================

chart_rows = []

def record_chart(chart_group, label, trade_date, chart_path, status="SAVED", error=None):
    chart_rows.append({
        "chart_group": chart_group,
        "label": label,
        "trade_date": trade_date,
        "date_str": pd.Timestamp(trade_date).strftime("%Y-%m-%d") if pd.notna(trade_date) else None,
        "chart_path": str(chart_path) if chart_path is not None else None,
        "status": status,
        "error": error,
    })

current_date = current_dates.loc[current_dates["date_role"] == "current", "trade_date"].iloc[0]

try:
    p = plot_current_vol_term_structure(point_panel, current_dates, CHART_DIR)
    record_chart("current_dashboard", "current_vol_term_structure_latest_prior_5d", current_date, p)
except Exception as e:
    record_chart("current_dashboard", "current_vol_term_structure_latest_prior_5d", current_date, None, "ERROR", f"{type(e).__name__}: {e}")

try:
    p = plot_current_vrp_log_term_structure(point_panel, current_dates, CHART_DIR)
    record_chart("current_dashboard", "current_vrp_log_term_structure_latest_prior_5d", current_date, p)
except Exception as e:
    record_chart("current_dashboard", "current_vrp_log_term_structure_latest_prior_5d", current_date, None, "ERROR", f"{type(e).__name__}: {e}")

try:
    p = plot_current_zscore_term_structure(point_panel, current_dates, CHART_DIR)
    record_chart("current_dashboard", "current_vrp_zscore_term_structure", current_date, p)
except Exception as e:
    record_chart("current_dashboard", "current_vrp_zscore_term_structure", current_date, None, "ERROR", f"{type(e).__name__}: {e}")

try:
    p = plot_current_threshold_overlay(point_panel, current_dates, CHART_DIR)
    record_chart("current_dashboard", "current_core_threshold_overlay", current_date, p)
except Exception as e:
    record_chart("current_dashboard", "current_core_threshold_overlay", current_date, None, "ERROR", f"{type(e).__name__}: {e}")

# ======================================================================================
# 5. Plot selected historical examples
# ======================================================================================

for _, row in historical_dates.sort_values("trade_date").iterrows():
    trade_date = pd.Timestamp(row["trade_date"]).normalize()
    label = row["label"]

    try:
        p = plot_historical_term_structure(
            point_panel=point_panel,
            trade_date=trade_date,
            label=label,
            out_dir=CHART_DIR,
            max_available_trade_date=max_available_trade_date,
        )
        record_chart("historical_examples", label, trade_date, p)
    except Exception as e:
        record_chart("historical_examples", label, trade_date, None, "ERROR", f"{type(e).__name__}: {e}")

chart_manifest = pd.DataFrame(chart_rows)

# ======================================================================================
# 6. Save audit outputs
# ======================================================================================

current_dates_path = AUDIT_DIR / f"07C_current_term_structure_dates_{TIMESTAMP}.csv"
historical_dates_path = AUDIT_DIR / f"07C_selected_historical_term_structure_dates_{TIMESTAMP}.csv"
visual_date_metrics_path = AUDIT_DIR / f"07C_visual_date_metrics_{TIMESTAMP}.csv"
point_panel_path = AUDIT_DIR / f"07C_term_structure_point_panel_{TIMESTAMP}.csv"
chart_manifest_path = AUDIT_DIR / f"07C_chart_manifest_{TIMESTAMP}.csv"
validation_path = AUDIT_DIR / f"07C_validation_{TIMESTAMP}.csv"
manifest_path = AUDIT_DIR / f"07C_manifest_{TIMESTAMP}.json"

current_dates.to_csv(current_dates_path, index=False)
historical_dates.to_csv(historical_dates_path, index=False)
visual_date_metrics.to_csv(visual_date_metrics_path, index=False)
point_panel.to_csv(point_panel_path, index=False)
chart_manifest.to_csv(chart_manifest_path, index=False)

# ======================================================================================
# 7. Validation
# ======================================================================================

validation_rows = []

chart_saved_count = int(chart_manifest["status"].eq("SAVED").sum())
chart_error_count = int(chart_manifest["status"].eq("ERROR").sum())
current_chart_saved_count = int(
    chart_manifest[
        (chart_manifest["chart_group"] == "current_dashboard")
        & (chart_manifest["status"] == "SAVED")
    ].shape[0]
)
historical_chart_saved_count = int(
    chart_manifest[
        (chart_manifest["chart_group"] == "historical_examples")
        & (chart_manifest["status"] == "SAVED")
    ].shape[0]
)

unique_tenors = sorted(point_panel["tenor"].dropna().astype(int).unique().tolist())
missing_tenors = sorted(set(ALL_TENORS) - set(unique_tenors))

latest_rows = point_panel[point_panel["trade_date"] == max_available_trade_date].copy()
latest_completed_count = int(latest_rows["forward_realized_complete"].sum()) if not latest_rows.empty else 0

add_validation(
    validation_rows,
    "source_panel_loaded",
    "PASS",
    f"path={source_panel_path}; rows={len(panel):,}",
)
add_validation(
    validation_rows,
    "max_available_trade_date",
    "PASS",
    f"{max_available_trade_date:%Y-%m-%d}",
)
add_validation(
    validation_rows,
    "unified_and_existing_present",
    "PASS" if set([UNIFIED_SPEC, EXISTING_SPEC]).issubset(set(panel["model_spec"].unique())) else "FAIL",
    f"models={sorted(panel['model_spec'].dropna().unique().tolist())}",
)
add_validation(
    validation_rows,
    "all_tenors_present",
    "PASS" if not missing_tenors else "FAIL",
    f"missing_tenors={missing_tenors}",
)
add_validation(
    validation_rows,
    "current_dates_selected",
    "PASS" if current_dates["available"].sum() >= 1 else "FAIL",
    f"current_dates={current_dates.to_dict('records')}",
)
add_validation(
    validation_rows,
    "historical_dates_selected",
    "PASS" if len(historical_dates) > 0 else "FAIL",
    f"historical_dates={len(historical_dates):,}",
)
add_validation(
    validation_rows,
    "current_dashboard_charts_saved",
    "PASS" if current_chart_saved_count >= 4 else "FAIL",
    f"saved={current_chart_saved_count}/4",
)
add_validation(
    validation_rows,
    "historical_example_charts_saved",
    "PASS" if historical_chart_saved_count > 0 else "FAIL",
    f"saved={historical_chart_saved_count}",
)
add_validation(
    validation_rows,
    "charts_saved",
    "PASS" if chart_saved_count > 0 and chart_error_count == 0 else "WARN",
    f"saved={chart_saved_count}; errors={chart_error_count}",
)
add_validation(
    validation_rows,
    "latest_forward_completion_count",
    "PASS",
    f"latest_date={max_available_trade_date:%Y-%m-%d}; completed_forward_tenors={latest_completed_count}/9",
)
add_validation(
    validation_rows,
    "completed_forward_guard",
    "PASS",
    "Forward realized vol is plotted only when forward_realized_complete is true.",
)
add_validation(
    validation_rows,
    "no_model_fitting",
    "PASS",
    "Cell 7C only reads Cell 7A output and creates plots.",
)
add_validation(
    validation_rows,
    "no_threshold_or_signal_changes",
    "PASS",
    "No threshold, signal, or parameter changes performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"] == "FAIL"]

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "cell": "Cell 7C — Unified denominator term-structure dashboard charts",
            "timestamp": TIMESTAMP,
            "source_panel": str(source_panel_path),
            "unified_spec": UNIFIED_SPEC,
            "existing_spec": EXISTING_SPEC,
            "all_tenors": ALL_TENORS,
            "max_available_trade_date": f"{max_available_trade_date:%Y-%m-%d}",
            "current_dates_path": str(current_dates_path),
            "historical_dates_path": str(historical_dates_path),
            "visual_date_metrics_path": str(visual_date_metrics_path),
            "point_panel_path": str(point_panel_path),
            "chart_manifest_path": str(chart_manifest_path),
            "validation_path": str(validation_path),
            "chart_directory": str(CHART_DIR),
            "notes": [
                "Charting-only cell.",
                "No fitting performed.",
                "No threshold or signal changes performed.",
                "Forward realized vol is plotted only when last_forward_rv_date <= max_available_trade_date.",
                "Current dashboard includes latest, prior trading day, and five-trading-day-ago term structures.",
            ],
        },
        f,
        indent=2,
    )

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 7C validation failed. See validation table above.")

# ======================================================================================
# 8. Display concise review
# ======================================================================================

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 320)

print("=" * 100)
print("Cell 7C — Unified denominator term-structure dashboard charts")
print("=" * 100)
print(f"Source Cell 7A panel:             {source_panel_path}")
print(f"Unified model spec:               {UNIFIED_SPEC}")
print(f"Existing reference spec:          {EXISTING_SPEC}")
print(f"Max available trade date:         {max_available_trade_date:%Y-%m-%d}")
print(f"Chart output directory:           {CHART_DIR}")
print(f"Current dashboard charts saved:   {current_chart_saved_count}")
print(f"Historical charts saved:          {historical_chart_saved_count}")
print(f"Total charts saved:               {chart_saved_count}")
print(f"Chart errors:                     {chart_error_count}")
print(f"Latest completed forward tenors:  {latest_completed_count}/9")
print("No model fitting:                 True")
print("No threshold changes:             True")
print("No signal changes:                True")

print()
print("Validation:")
display(validation)

print()
print("Current term-structure comparison dates:")
display(current_dates)

print()
print("Latest current term-structure values:")
latest_display_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "implied_vol_pct",
    "unified_forecast_vol_pct",
    "existing_forecast_vol_pct",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "threshold_model_vrp_log",
    "threshold_model_vrp_z_3m",
    "threshold_model_vrp_z_1y",
    "diagnostic_core_pass",
    "forward_realized_complete",
    "forward_realized_vol_pct_plot",
    "last_forward_rv_date",
]
display(
    point_panel[point_panel["trade_date"] == max_available_trade_date]
    .sort_values("tenor")
    .loc[:, [c for c in latest_display_cols if c in point_panel.columns]]
)

print()
print("Selected historical term-structure dates:")
display(
    historical_dates[
        [
            "date_str",
            "label",
            "reason",
            "completed_forward_tenor_count",
            "all_forward_realized_complete",
            "avg_implied_vol_pct",
            "avg_unified_forecast_vol_pct",
            "avg_forward_realized_vol_pct_completed",
            "avg_model_vrp_log",
            "avg_model_vrp_z_3m",
            "avg_model_vrp_z_1y",
            "latest_last_forward_rv_date",
        ]
    ].sort_values("date_str", ascending=False)
)

print()
print("Chart manifest:")
display(chart_manifest.sort_values(["chart_group", "date_str", "label"], ascending=[True, False, True]))

print()
print("Saved audit outputs:")
saved_paths = [
    current_dates_path,
    historical_dates_path,
    visual_date_metrics_path,
    point_panel_path,
    chart_manifest_path,
    validation_path,
    manifest_path,
]
for p in saved_paths:
    print(f"  {p}")

print("\nCELL 7C PASSED")

# ======================================================================================
# 9. Display saved term-structure charts inline
# ======================================================================================
# Display-only repair integrated into the clean notebook.
# This does not change model logic, thresholds, signals, or saved outputs.

from IPython.display import display as ipy_display, Image, Markdown

print("\nDisplaying saved Cell 7C charts inline:")

_chart_display = chart_manifest.copy()
_chart_display = _chart_display[_chart_display["status"].eq("SAVED")].copy()

if not _chart_display.empty:
    _chart_display["chart_exists"] = _chart_display["chart_path"].astype(str).map(lambda p: Path(p).exists())
    _chart_display = _chart_display[_chart_display["chart_exists"]].copy()

if _chart_display.empty:
    print("No existing chart PNGs found to display. Check chart_manifest for paths/errors.")
else:
    _chart_display["chart_group_sort"] = _chart_display["chart_group"].map({
        "current_dashboard": 0,
        "historical_examples": 1,
    }).fillna(9)
    _chart_display = _chart_display.sort_values(
        ["chart_group_sort", "date_str", "label"],
        ascending=[True, False, True],
    )

    for _, _row in _chart_display.iterrows():
        _title_parts = [
            str(_row.get("chart_group", "")).replace("_", " ").title(),
            str(_row.get("date_str", "")),
            str(_row.get("label", "")).replace("_", " "),
        ]
        _title = " — ".join([x for x in _title_parts if x and x != "nan"])
        ipy_display(Markdown(f"### {_title}"))
        ipy_display(Image(filename=str(_row["chart_path"])))

print("\nCELL 7C INLINE DISPLAY COMPLETE")
